# Notebook A - semantic collection plus participation instruments

The semantic collectors and optimized Authorized Access block are unchanged. New section 13d only inventories optional administrative inputs and maintains a local first-observed ledger. Run-all cache policy and original table/checkpoint structures are preserved.


## 0. Configuration and shared runtime
Only these settings normally need editing. No API keys are printed or saved in the run report.


In [ ]:
# @title 0. Settings and environment -- safe defaults for Run all
# Change this folder only when your existing artifacts are elsewhere.
DRIVE_ROOT = "/content/drive/MyDrive/dbgap_analysis"
RUN_MODE = "reuse"  # "reuse" | "check_stale" | "clean"
STALE_MONTHS = 3    # Calendar months, not a 90-day approximation.
# Empty means every block inherits RUN_MODE. Example: {"icite": "check_stale"}.
BLOCK_MODES = {}
# Sources: fhir, dar, counts, metadata, sstr, clinical_trials, pubmed,
#          authorized_access, icite, reporter_publications, reporter_projects, ftp.
# Local products: root_table, joins, timing, icite_validation.
# clean archives existing registered files; it does not permanently delete them.

FTP_RUN_MODE = "full"  # "audit" uses a separate folder and never replaces full FTP outputs.
FTP_AUDIT_RANDOM_N = 50
SOURCE_FLUSH_EVERY = 100
PUBMED_EMAIL = ""  # Supply your real contact email for recurring queries.
ENABLE_PUBLIC_ATTRIBUTION = True
ENABLE_PUBMED_GRANTS = False  # Optional: direct study acknowledgement is the slide-1 default.
ENABLE_REPORTER = True
ENABLE_FTP_TIMING = True
MAKE_TIMING_FIGURES = False  # B owns the preliminary deck plots.
DERIVED_RECIPE_VERSION = "harmonized-A-v1"  # Existing joins are unchanged.
ENABLE_SEMANTIC_COLLECTION = True
MESH_YEAR = 2026  # Fixed vocabulary year; never silently switch to a newer MeSH release.
SEMANTIC_MAX_REQUESTS = 5000  # Bound new remote requests; already saved exact-version pages are reused.
SEMANTIC_MAX_FAILURES_PER_HOST = 10
# Semantic collection follows RUN_MODE / BLOCK_MODES['semantics'] and STALE_MONTHS.
# It creates semantic_sources/ sidecars; it never overwrites original MeSH or SSTR columns.

import sys, subprocess, importlib.util
_missing = [p for p in ("requests", "pandas", "numpy", "pyarrow", "matplotlib", "bs4")
            if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *["beautifulsoup4" if p == "bs4" else p for p in _missing]], check=True)
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None and str(DRIVE_ROOT).startswith("/content/drive/"):
    drive.mount("/content/drive")

import os, io, json, re, time, random, hashlib, shutil, warnings
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin
from itertools import combinations
from ftplib import FTP, all_errors
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display
os.makedirs(DRIVE_ROOT, exist_ok=True)
PATHS = {
    'studies_raw': os.path.join(DRIVE_ROOT, 'dbgap_studies_raw.parquet'),
    'fhir_checkpoint': os.path.join(DRIVE_ROOT, 'fhir_fetch_checkpoint.json'),
    'root_accessions': os.path.join(DRIVE_ROOT, 'dbgap_root_accessions.parquet'),
    'dar_records': os.path.join(DRIVE_ROOT, 'dar_records.parquet'),
    'dar_checkpoint': os.path.join(DRIVE_ROOT, 'dar_fetch_checkpoint.json'),
    'study_counts': os.path.join(DRIVE_ROOT, 'study_counts.parquet'),
    'study_counts_checkpoint': os.path.join(DRIVE_ROOT, 'study_counts_checkpoint.json'),
    'study_page_meta': os.path.join(DRIVE_ROOT, 'study_page_metadata.parquet'),
    'study_page_checkpoint': os.path.join(DRIVE_ROOT, 'study_page_checkpoint.json'),
    'sstr_summary': os.path.join(DRIVE_ROOT, 'sstr_study_summary.parquet'),
    'sstr_checkpoint': os.path.join(DRIVE_ROOT, 'sstr_checkpoint.json'),
    'mesh_data': os.path.join(DRIVE_ROOT, 'mesh_conditions.parquet'),
    'mesh_checkpoint': os.path.join(DRIVE_ROOT, 'mesh_checkpoint.json'),
    'clinical_trials': os.path.join(DRIVE_ROOT, 'clinical_trials_candidates.parquet'),
    'clinical_trials_checkpoint': os.path.join(DRIVE_ROOT, 'clinical_trials_checkpoint.json'),
    'pub_verification': os.path.join(DRIVE_ROOT, 'publication_verification.parquet'),
    'pub_verification_checkpoint': os.path.join(DRIVE_ROOT, 'publication_verification_checkpoint.json'),
    'clinical_trials_direct': os.path.join(DRIVE_ROOT, 'dbgap_ctgov_direct_links.parquet'),
    'clinical_trials_direct_checkpoint': os.path.join(DRIVE_ROOT, 'dbgap_ctgov_direct_checkpoint.json'),
    'pubmed_smoketest': os.path.join(DRIVE_ROOT, 'pubmed_metadata_smoketest.parquet'),
    'pubmed_smoketest_checkpoint': os.path.join(DRIVE_ROOT, 'pubmed_metadata_smoketest_checkpoint.json'),
    'joined': os.path.join(DRIVE_ROOT, 'joined_accessions_dar.parquet'),
    'pubmed_metadata': os.path.join(DRIVE_ROOT, 'pubmed_metadata_full.parquet'),
    'pubmed_checkpoint': os.path.join(DRIVE_ROOT, 'pubmed_metadata_full_checkpoint.json'),
    'pubmed_links': os.path.join(DRIVE_ROOT, 'dbgap_study_pubmed_links_full.parquet'),
    'icite_metrics': os.path.join(DRIVE_ROOT, 'icite_rcr_metrics.parquet'),
    'icite_checkpoint': os.path.join(DRIVE_ROOT, 'icite_rcr_checkpoint.json'),
    'icite_study_rollup': os.path.join(DRIVE_ROOT, 'icite_study_rcr_rollup.parquet'),
    'icite_verification': os.path.join(DRIVE_ROOT, 'icite_verification_report.parquet'),
    'authorized_access_metadata': os.path.join(DRIVE_ROOT, 'dbgap_authorized_access_api_metadata.parquet'),
    'authorized_access_checkpoint': os.path.join(DRIVE_ROOT, 'dbgap_authorized_access_api_checkpoint.json'),
    'reporter_pub_links_full': os.path.join(DRIVE_ROOT, 'reporter_pub_links_full.parquet'),
    'reporter_pub_checkpoint': os.path.join(DRIVE_ROOT, 'reporter_pub_checkpoint.json'),
    'reporter_projects_full': os.path.join(DRIVE_ROOT, 'reporter_projects_full.parquet'),
    'reporter_proj_checkpoint': os.path.join(DRIVE_ROOT, 'reporter_proj_checkpoint.json'),
    'reporter_pmid_award_mn': os.path.join(DRIVE_ROOT, 'reporter_pmid_core_award_MxN.parquet'),
    'reporter_core_funding': os.path.join(DRIVE_ROOT, 'reporter_core_current_period_funding.parquet'),
    'reporter_study_rollup': os.path.join(DRIVE_ROOT, 'reporter_study_funding_rollup.parquet'),
    'reporter_verification': os.path.join(DRIVE_ROOT, 'reporter_funding_verification.parquet'),
}
PATHS.update({
    "study_counts_analysis_ready": os.path.join(DRIVE_ROOT, "study_counts_analysis_ready.parquet"),
    "study_page_metadata_analysis_ready": os.path.join(DRIVE_ROOT, "study_page_metadata_analysis_ready.parquet"),
    "sstr_summary_analysis_ready": os.path.join(DRIVE_ROOT, "sstr_summary_analysis_ready.parquet"),
})
FIGURE_DIR = os.path.join(DRIVE_ROOT, "analysis_figures_combined")
FIG_DIR = TIMING_FIGURE_DIR = FIGURE_DIR
os.makedirs(FIGURE_DIR, exist_ok=True)
COLORS = {"blue": "#2563EB", "amber": "#D97706", "grey": "#64748B",
          "slate": "#475569", "green": "#059669", "red": "#DC2626"}
HEADERS = {"Accept": "application/json", "User-Agent": "dbgap-accession-dar-research/0.5"}
TIMEOUT = 30
def get_colab_secret(name):
    """Return a Colab secret value, or None when unset/unavailable."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if isinstance(value, str) and value.strip() else None
    except Exception:
        return None
NCBI_API_KEY = get_colab_secret("NCBI")
# Retained aliases used only by optional legacy audit code.
OPENALEX_API_KEY = get_colab_secret("OPENALEX")
SCOPUS_API_KEY = get_colab_secret("SCOPUS")
WOS_API_KEY = get_colab_secret("WOS")
REPORTER_PUB_URL = "https://api.reporter.nih.gov/v1/publications/search"
REPORTER_PRJ_URL = "https://api.reporter.nih.gov/v1/projects/search"
CONTRACT_PREFIXES = ("N01", "HHSN")
FULL_GRANT_RE = re.compile(r"([A-Z]\d{2}[A-Z]{2}\d{6})")
REPORTER_SLEEP = 1.0
ICITE_API = "https://icite.od.nih.gov/api/pubs"
assert RUN_MODE in {"reuse", "check_stale", "clean"}
assert STALE_MONTHS >= 1
print(f"Mode: {RUN_MODE}; stale threshold: {STALE_MONTHS} calendar months; data: {DRIVE_ROOT}")
print("Maintenance is disabled unless explicitly enabled in the last cells.")


## 1. Shared cache policy and single-source views
One scheduler is used throughout. A successful negative response completes a lookup but does not mean there is no scientific use. Three-month age refers to a successful source check, not a local join or a notebook run.


In [ ]:
# @title
# Shared cache policy. All acquisition blocks use these functions.
VOLATILE_COLUMNS = {"fetched_at", "pubmed_fetched_utc", "icite_fetched_utc",
                    "ctgov_fetched_utc", "authorized_access_fetched_utc",
                    "validated_utc", "checkpoint_timestamp_utc"}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
STATE_PATH = os.path.join(DRIVE_ROOT, "notebook_a_refresh_state.json")
RUN_REPORT = []
_CLEANED = set()
_CHANGED_SOURCE_FILES = set()
_LAST_SOURCE_ERRORS = {}


def utc_now():
    return pd.Timestamp.now(tz="UTC")


def load_json(path, default):
    if not os.path.exists(path):
        return default
    try:
        with open(path, encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError) as exc:
        # A corrupt checkpoint must not silently trigger a full re-pull.
        raise RuntimeError(f"Cannot read checkpoint {path}. Preserve it and inspect before retrying.") from exc


def save_json(path, obj, max_retries=5):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = str(path) + ".tmp"
    for attempt in range(max_retries):
        try:
            with open(tmp, "w", encoding="utf-8") as handle:
                json.dump(obj, handle, indent=2, default=str, allow_nan=False)
            os.replace(tmp, path)
            return
        except OSError:
            if attempt + 1 == max_retries:
                raise
            time.sleep(min(2 ** attempt, 15))


PIPELINE_STATE = load_json(STATE_PATH, {"format_version": 1, "sources": {}, "derived": {}})
PIPELINE_STATE.setdefault("sources", {})
PIPELINE_STATE.setdefault("derived", {})


def persist_state():
    save_json(STATE_PATH, PIPELINE_STATE)


def sha256_file(path):
    if not os.path.isfile(path):
        return None
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def mode_for(block):
    value = BLOCK_MODES.get(block, BLOCK_MODES.get("ftp", RUN_MODE) if block == "ftp_audit" else RUN_MODE)
    if value not in {"reuse", "check_stale", "clean"}:
        raise ValueError(f"{block}: mode must be reuse, check_stale, or clean; got {value!r}")
    return value


def read_frame(key, columns=()):
    path = PATHS.get(key, key)
    if not os.path.isfile(path):
        return pd.DataFrame(columns=list(columns))
    return pd.read_parquet(path)


def require_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label}: missing columns {missing}. No automatic schema repair was applied.")


def require_files(*keys):
    missing = [PATHS.get(k, k) for k in keys if not os.path.isfile(PATHS.get(k, k))]
    if missing:
        raise FileNotFoundError("Missing prerequisites: " + ", ".join(missing))


def stable_value(value):
    if isinstance(value, dict):
        return {str(k): stable_value(v) for k, v in sorted(value.items(), key=lambda x: str(x[0]))}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [stable_value(v) for v in value]
    if isinstance(value, set):
        return sorted((stable_value(v) for v in value), key=str)
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.generic):
        value = value.item()
    try:
        if pd.isna(value):
            return None
    except (ValueError, TypeError):
        pass
    return value


def rows_equal(left, right, ignore_clock=False):
    """Compare complete rows without sorting/changing either stored dataframe."""
    if set(left.columns) != set(right.columns) or len(left) != len(right):
        return False
    cols = sorted(c for c in left.columns if not (ignore_clock and c in VOLATILE_COLUMNS))
    def signatures(frame):
        return sorted(json.dumps(stable_value(r), sort_keys=True, default=str)
                      for r in frame[cols].to_dict("records"))
    return signatures(left) == signatures(right)


def atomic_table(frame, path, compare=True):
    """Write through a sibling temp file; retain existing bytes if values/schema match."""
    if compare and os.path.exists(path):
        old = pd.read_parquet(path)
        if list(old.columns) == list(frame.columns) and rows_equal(old, frame):
            return False
    tmp = str(path) + ".tmp"
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, path)
    return True


def archive_registered(paths, label):
    """Move only named files below DRIVE_ROOT into a timestamped backup; never delete."""
    root = Path(DRIVE_ROOT).resolve()
    backup = root / "notebook_a_backups" / RUN_ID / label
    moved = []
    for raw in dict.fromkeys(map(str, paths)):
        path = Path(raw).resolve()
        if not path.is_relative_to(root) or path == root:
            raise ValueError(f"Refusing to archive outside the data directory: {path}")
        if path.is_file():
            destination = backup / path.relative_to(root)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), str(destination))
            moved.append({"original": str(path), "backup": str(destination),
                    "sha256": sha256_file(str(destination))})
    if moved:
        save_json(str(backup / "archive_manifest.json"), {"run_id": RUN_ID, "files": moved})
        print(f"ARCHIVED {len(moved)} file(s): {backup}")
    return moved


def prepare_source(block, keys):
    if mode_for(block) == "clean" and block not in _CLEANED:
        archive_registered([PATHS[k] for k in keys], block)
        PIPELINE_STATE["sources"].pop(block, None)
        _CLEANED.add(block)
        persist_state()
    return PIPELINE_STATE["sources"].setdefault(block, {"items": {}})


def checkpoint_map(key, field):
    saved = load_json(PATHS[key], {}).get(field, {})
    return {str(k): str(v) for k, v in saved.items()} if isinstance(saved,
        dict) else {str(k): "terminal" for k in saved}


def key_values(frame, cols):
    if frame.empty:
        return pd.Series(index=frame.index, dtype="string")
    require_columns(frame, cols, "Source record key")
    return frame[list(cols)].astype("string").fillna("").agg("|".join, axis=1)


def known_from_rows(frame, cols, note_column=None, accepted=("ok",), error_column=None):
    if frame.empty:
        return {}
    key = key_values(frame, cols)
    mask = pd.Series(True, index=frame.index)
    if note_column:
        if note_column not in frame:
            return {}
        mask = frame[note_column].isin(accepted)
    if error_column and error_column in frame:
        mask = frame[error_column].isna() | frame[error_column].isin(accepted)
    return dict(zip(key.loc[mask].astype(str),
                    frame.loc[mask, note_column].astype(str) if note_column else ["ok"] * int(mask.sum())))


def item_last_check(block_state, item, old, keycols, path, *, fetch_times=None, legacy_mtime=None):
    record = block_state.setdefault("items", {}).setdefault(item, {})
    stamp = pd.to_datetime(record.get("last_checked_utc"), utc=True, errors="coerce")
    if pd.notna(stamp):
        return stamp, "recorded_successful_check"
    anchor = pd.to_datetime(record.get("legacy_check_reference_utc"), utc=True, errors="coerce")
    if pd.notna(anchor):
        return anchor, record.get("legacy_check_reference_basis", "legacy_file_mtime_estimate")
    if fetch_times is None:
        # Convenience path used only by standalone checks; collect_keyed pre-indexes once.
        fetch_times = {}
        if not old.empty:
            keys = key_values(old, keycols)
            dates = pd.Series(pd.NaT, index=old.index, dtype="datetime64[ns, UTC]")
            for col in VOLATILE_COLUMNS - {"validated_utc", "checkpoint_timestamp_utc"}:
                if col in old:
                    parsed = pd.to_datetime(old[col], utc=True, errors="coerce")
                    dates = dates.where(dates.notna() & ((dates >= parsed) | parsed.isna()), parsed)
            fetch_times = dates.groupby(keys).max().dropna().to_dict()
    stamp = fetch_times.get(item, pd.NaT)
    basis = "stored_fetch_timestamp"
    if pd.isna(stamp):
        if legacy_mtime is None:
            legacy_mtime = pd.Timestamp(os.path.getmtime(path), unit="s",
                tz="UTC") if os.path.isfile(path) else pd.NaT
        stamp, basis = legacy_mtime, "legacy_file_mtime_estimate"
    if pd.notna(stamp):
        # Freeze the initial legacy reference: adding one new row must not reset
        # every old record's apparent age. This is NOT a successful check stamp.
        record["legacy_check_reference_utc"] = stamp.isoformat()
        record["legacy_check_reference_basis"] = basis
    return stamp, basis if pd.notna(stamp) else "unknown"


def replace_response_groups(old, changes, keycols):
    """Apply a batch of full-response replacements in one dataframe pass."""
    if not changes:
        return old
    keys = key_values(old, keycols)
    columns = list(old.columns)
    parts = [old.loc[~keys.isin(changes)]]
    for item, frame in changes.items():
        columns += [c for c in frame.columns if c not in columns]
        if len(frame):
            require_columns(frame, keycols, item)
            if not key_values(frame, keycols).eq(item).all():
                raise ValueError(f"{item}: replacement contains an unexpected key")
            parts.append(frame)
    nonempty = [frame for frame in parts if not frame.empty]
    return (pd.concat(nonempty, ignore_index=True, sort=False).reindex(columns=columns)
            if nonempty else old.iloc[:0].reindex(columns=columns).copy())


def needs_source_item(mode, terminal, last_check, *, same_version=True, now=None):
    if not terminal or not same_version or mode == "clean":
        return True
    if mode == "reuse":
        return False
    now = utc_now() if now is None else pd.Timestamp(now)
    return pd.isna(last_check) or last_check > now or last_check <= now - pd.DateOffset(months=STALE_MONTHS)


def update_group(old, new, item, keycols):
    """Replace the full response group, including a successful empty response."""
    mask = key_values(old, keycols).eq(item)
    previous = old.loc[mask]
    if len(new):
        require_columns(new, keycols, item)
        if not key_values(new, keycols).eq(item).all():
            raise ValueError(f"{item}: response contains an unexpected key; no data written")
    if rows_equal(previous, new, ignore_clock=True):
        return old, False
    if old.empty:
        return new.copy() if len(new.columns) else old.copy(), bool(len(new))
    if new.empty:
        return old.loc[~mask].copy(), bool(mask.any())
    # Preserve prior column ordering and append only columns present in the source payload.
    cols = list(old.columns) + [c for c in new if c not in old]
    return pd.concat([old.loc[~mask], new], ignore_index=True, sort=False).reindex(columns=cols), True


def collect_keyed(block, artifact_key, checkpoint_key, checkpoint_field, targets, keycols,
                  fetch_group, existing_terminal, *, empty_columns, batch_size=1,
                  pause=0.0, checkpoint_dict=False, versions=None, stored_version_col=None,
                  checkpoint_extra=None, extra_flush=None):
    """One scheduler for API stages; source columns and old checkpoint shapes are retained.

    fetch_group receives target objects and returns key -> (DataFrame, note, terminal).
    Only complete responses advance last_checked_utc. Failed refreshes preserve old rows.
    """
    state = prepare_source(block, [artifact_key, checkpoint_key])
    old = read_frame(artifact_key, empty_columns)
    done = existing_terminal(old)
    mode = mode_for(block)
    now = utc_now()
    selected = []
    age_sources = {}
    old_keys = key_values(old, keycols)
    old_groups = {}  # Materialize only selected response groups after scheduling.
    dates = pd.Series(pd.NaT, index=old.index, dtype="datetime64[ns, UTC]")
    for col in VOLATILE_COLUMNS - {"validated_utc", "checkpoint_timestamp_utc"}:
        if col in old:
            parsed = pd.to_datetime(old[col], utc=True, errors="coerce")
            dates = dates.where(dates.notna() & ((dates >= parsed) | parsed.isna()), parsed)
    fetch_times = dates.groupby(old_keys).max().dropna().to_dict()
    legacy_path = PATHS[artifact_key] if os.path.isfile(PATHS[artifact_key]) else PATHS[checkpoint_key]
    legacy_mtime = pd.Timestamp(os.path.getmtime(legacy_path), unit="s",
        tz="UTC") if os.path.isfile(legacy_path) else pd.NaT
    stored_versions = dict(zip(old_keys,
            old[stored_version_col].astype(str))) if stored_version_col and stored_version_col in old else {}
    before_planning = json.dumps(state, sort_keys=True, default=str)
    for item in targets:
        stamp, basis = item_last_check(state, item, old, keycols, PATHS[artifact_key], fetch_times=fetch_times,
            legacy_mtime=legacy_mtime)
        age_sources[basis] = age_sources.get(basis, 0) + 1
        same_version = True
        if versions is not None:
            previous = state["items"].get(item, {}).get("source_version")
            if previous is None and stored_version_col and stored_version_col in old:
                previous = stored_versions.get(item)
            same_version = previous == versions[item] if previous is not None else True
        pending_retry = state["items"].get(item, {}).get("last_attempt_failed",
            False) or state.get("force_check_next_run", False)
        if needs_source_item(mode, item in done and not pending_retry, stamp, same_version=same_version, now=now):
            selected.append(item)
    print(f"{block} [{mode}]: {len(targets):,} targets; {len(selected):,} need collection/check; {len(targets)-len(selected):,} reused.")
    if mode == "check_stale" and age_sources.get("legacy_file_mtime_estimate", 0):
        print("  Legacy freshness uses file mtime as an estimate, not a verified last-check timestamp.")
    if json.dumps(state, sort_keys=True, default=str) != before_planning:
        persist_state()  # Legacy clock anchors only; no source file or check timestamp changes.
    changed = False
    if selected and len(old):
        selected_rows = old.loc[old_keys.isin(selected)]
        old_groups = {str(k): part.copy() for k, part in selected_rows.groupby(old_keys.loc[selected_rows.index],
                sort=False)}
    response_changes = {}
    checked = failed = 0
    cp = load_json(PATHS[checkpoint_key], {})
    pending_state = {}
    flush = max(1, SOURCE_FLUSH_EVERY // batch_size)
    for group_number, start in enumerate(range(0, len(selected), batch_size), 1):
        ids = selected[start:start + batch_size]
        try:
            response = fetch_group({k: targets[k] for k in ids})
        except Exception as exc:
            # Software/schema failures are visible; successful cached records are not erased.
            response = {k: (pd.DataFrame(columns=empty_columns), f"exception:{type(exc).__name__}:{exc}",
                    False) for k in ids}
        for item in ids:
            new, note, terminal = response.get(item, (pd.DataFrame(columns=empty_columns), "missing_batch_result",
                    False))
            entry = dict(state["items"].get(item, {}))
            entry.update(last_attempt_utc=utc_now().isoformat(), last_attempt_status=str(note),
                last_attempt_failed=not bool(terminal))
            if terminal:
                previous_group = old_groups.get(item, old.iloc[:0])
                if len(new) and not key_values(new, keycols).eq(item).all():
                    raise ValueError(f"{item}: response contains an unexpected key; no data written")
                if not rows_equal(previous_group, new, ignore_clock=True):
                    response_changes[item] = new
                    old_groups[item] = new
                    changed = True
                entry["last_checked_utc"] = entry["last_attempt_utc"]
                if versions is not None:
                    entry["source_version"] = versions[item]
                done[item] = str(note)
                checked += 1
            else:
                failed += 1
                # Store first-attempt error rows only when there is no previous response.
                if len(new) and item not in old_groups:
                    if not key_values(new, keycols).eq(item).all():
                        raise ValueError(f"{item}: error response contains an unexpected key")
                    response_changes[item] = new
                    old_groups[item] = new
                    changed = True
                print(f"  RETRY LATER {item}: {note}")
            pending_state[item] = entry
        if group_number % flush == 0 or start + len(ids) == len(selected):
            # The data commit precedes the checkpoint. A crash cannot checkpoint unwritten data.
            if extra_flush is not None:
                extra_flush()
            if changed or not os.path.exists(PATHS[artifact_key]):
                old = replace_response_groups(old, response_changes, keycols)
                response_changes = {}
                wrote = atomic_table(old, PATHS[artifact_key])
                if wrote:
                    _CHANGED_SOURCE_FILES.add(PATHS[artifact_key])
                changed = False
            cp[checkpoint_field] = dict(sorted(done.items())) if checkpoint_dict else sorted(done)
            if checkpoint_extra:
                cp.update(checkpoint_extra)
            save_json(PATHS[checkpoint_key], cp)
            state["items"].update(pending_state)
            pending_state = {}
            state["last_attempt_utc"] = utc_now().isoformat()
            if failed == 0 and set(targets).issubset(done):
                state["last_successful_run_utc"] = state["last_attempt_utc"]
                state.pop("force_check_next_run", None)
            persist_state()
            print(f"  {min(start+len(ids),
                        len(selected)):,}/{len(selected):,} attempted; {failed:,} failures retained for retry.")
        if pause:
            time.sleep(pause)
    if not os.path.exists(PATHS[artifact_key]) and (not targets or set(targets).issubset(done)):
        atomic_table(old, PATHS[artifact_key])
        _CHANGED_SOURCE_FILES.add(PATHS[artifact_key])
    _LAST_SOURCE_ERRORS[block] = failed
    RUN_REPORT.append({"block": block, "mode": mode, "targets": len(targets),
                       "attempted": len(selected), "checked": checked, "failed": failed,
                       "remaining": len(set(targets)-set(done)), "rows": len(old),
                       "age_basis": age_sources})
    brief_stats(block, old, keycols, done=done, targets=targets)
    return old


def brief_stats(label, frame, keycols=(), done=None, targets=None):
    print(f"{label}: {len(frame):,} rows x {len(frame.columns):,} columns")
    if not frame.columns.is_unique:
        raise AssertionError(f"{label}: duplicate column labels")
    if keycols:
        require_columns(frame, keycols, label)
        print(f"  Distinct keys: {key_values(frame,
                    keycols).nunique():,}; null key cells: {int(frame[list(keycols)].isna().sum().sum()):,}")
    for col in frame:
        if col.endswith("fetch_note") or col in {"collection_status", "fetch_error"}:
            print(f"  {col}: {frame[col].fillna('<missing>').value_counts().head(8).to_dict()}")
    if done is not None and targets is not None:
        print(f"  Completed checks: {len(set(targets)&set(done)):,}/{len(targets):,}")
    if len(frame):
        display(frame.head(3))


def source_inputs_signature(keys):
    return {PATHS.get(k, k): sha256_file(PATHS.get(k, k)) for k in keys}


def run_derived(block, inputs, outputs, build, *, extra=None):
    """Rebuild local results only when absent or their recorded inputs/recipe changed."""
    paths = [PATHS.get(k, k) for k in outputs]
    signature = {"inputs": source_inputs_signature(inputs), "recipe": DERIVED_RECIPE_VERSION,
                 "options": stable_value(extra or {})}
    previous = PIPELINE_STATE["derived"].get(block)
    all_present = all(os.path.isfile(p) for p in paths)
    clean = mode_for(block) == "clean"
    integrity_ok = previous is not None and all(sha256_file(p) == previous.get("outputs", {}).get(p) for p in paths)
    reuse = all_present and not clean and previous is not None and previous.get("signature") == signature and integrity_ok
    if all_present and previous is None and not clean:
        # Adopt existing legacy products without recomputation only if no input changed/newer.
        input_paths = list(signature["inputs"])
        oldest_output = min(os.path.getmtime(p) for p in paths)
        reuse = not any(p in _CHANGED_SOURCE_FILES or (os.path.isfile(p) and os.path.getmtime(p) > oldest_output)
                        for p in input_paths)
    if reuse:
        print(f"{block}: reusing saved outputs; source hashes unchanged.")
        if previous is None:
            PIPELINE_STATE["derived"][block] = {"signature": signature, "adopted_legacy_utc": utc_now().isoformat(),
                                                "outputs": {p: sha256_file(p) for p in paths}, "recomputed": False}
            persist_state()
        return False
    print(f"{block}: building local outputs (no acquisition calls).")
    if clean and block not in _CLEANED:
        archive_registered(paths, block)
        _CLEANED.add(block)
    build()
    missing = [p for p in paths if not os.path.isfile(p)]
    if missing:
        raise RuntimeError(f"{block}: build did not produce {missing}; signature not advanced")
    PIPELINE_STATE["derived"][block] = {"signature": signature, "built_utc": utc_now().isoformat(),
                                        "outputs": {p: sha256_file(p) for p in paths}, "recomputed": True}
    persist_state()
    return True

def compact_study_level_table(df, accession_col="accession", latest_accessions=None, prefer_ok_col=None):
    """Restrict to current latest versions and retain one defensible row per accession."""
    if df.empty:
        return df.copy()
    out = df.copy()
    out[accession_col] = out[accession_col].astype(str)
    if latest_accessions is not None:
        out = out[out[accession_col].isin(latest_accessions)].copy()
    if prefer_ok_col and prefer_ok_col in out.columns:
        out["_preferred"] = out[prefer_ok_col].eq("ok").astype(int)
        out = out.sort_values([accession_col, "_preferred"], ascending=[True, True])
        out = out.drop(columns="_preferred")
    out = out.drop_duplicates(subset=[accession_col], keep="last")
    return out

def current_pages():
    roots = read_frame("root_accessions")
    page = read_frame("study_page_meta", ["accession", "pmids_found"])
    return compact_study_level_table(page, latest_accessions=set(roots.latest_accession.dropna().astype(str)))


## 2. FHIR studies and canonical acquisition universe
The root list is a one-source view needed by the collectors; analytical joins come later. A refresh uses a resumable candidate snapshot, not a destructive restart of the active complete snapshot.


In [ ]:
# @title 2. FHIR studies and canonical acquisition universe
FHIR_BASE = "https://dbgap-api.ncbi.nlm.nih.gov/fhir/x1/"
def fetch_bundle(url, max_retries=8):
    delay = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        except requests.RequestException:
            time.sleep(delay); delay = min(delay * 2, 60); continue
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code in (429, 500, 502, 503, 504):
            time.sleep(delay); delay = min(delay * 2, 60); continue
        raise RuntimeError(f"FHIR request failed: {resp.status_code} {resp.text[:300]}")
    raise RuntimeError(f"Exceeded retries fetching {url}")

def extract_next_link(bundle):
    for link in bundle.get("link", []):
        if link.get("relation") == "next":
            return link.get("url")
    return None

def extract_mesh_terms(condition_list):
    """Pulls primary MeSH descriptor(s) from a FHIR ResearchStudy.condition list.
    Confirmed shape: [{"coding": [{"system": ..., "code": "D010190", "display": ...}], "text": ...}, ...]
    Entry-term synonyms use a dbGaP-specific MeshEntryTerm system -- excluded
    here (kept as duplicate/near-duplicate labels for the same descriptor);
    only the primary MeSH descriptor (urn:oid:2.16.840.1.113883.6.177) is kept.
    """
    if not condition_list:
        return []
    terms = []
    for cond in condition_list:
        for coding in cond.get("coding", []):
            if coding.get("system") == "urn:oid:2.16.840.1.113883.6.177":
                terms.append({"mesh_id": coding.get("code"), "label": coding.get("display")})
    return terms

def parse_research_study(rs):
    identifiers = rs.get("identifier", [])
    raw_accession = None
    for ident in identifiers:
        val = ident.get("value", "")
        if re.match(r"^phs\d{6}(\.v\d+)?(\.p\d+)?$", val or ""):
            raw_accession = val; break
    if raw_accession is None:
        m = re.search(r"(phs\d{6}(?:\.v\d+)?(?:\.p\d+)?)", rs.get("id", ""))
        if m:
            raw_accession = m.group(1)

    sponsor_ref = None
    for extension in rs.get("extension", []) or []:
        if "sponsor" in (extension.get("url") or "").lower():
            sponsor_ref = extension.get("valueReference", {}).get("display") or extension.get("valueString")

    return {
        "fhir_id": rs.get("id"),
        "raw_accession": raw_accession,
        "title": rs.get("title"),
        "status": rs.get("status"),
        "sponsor": sponsor_ref,
        "mesh_terms_raw": json.dumps(extract_mesh_terms(rs.get("condition"))),
        "identifiers_raw": json.dumps(identifiers),
        "fetched_at": datetime.now(timezone.utc).isoformat(),
    }

def split_accession(raw_accession):
    if not raw_accession:
        return None, None, None
    m = re.match(r"^(phs\d{6})(?:\.v(\d+))?(?:\.p(\d+))?$", raw_accession)
    if not m:
        return None, None, None
    root, v, p = m.group(1), m.group(2), m.group(3)
    return root, (int(v) if v else None), (int(p) if p else None)

def build_root_accession_table():
    raw = pd.read_parquet(PATHS["studies_raw"])

    # Schema guard: dbgap_studies_raw.parquet may have been written by an
    # older version of parse_research_study that didn't capture MeSH terms.
    # Backfill a null column rather than crashing, and tell the user how to
    # get real MeSH data if they want it.
    if "mesh_terms_raw" not in raw.columns:
        print("WARNING: dbgap_studies_raw.parquet predates MeSH capture (no "
              "'mesh_terms_raw' column). Backfilling as empty -- MeSH fields "
              "in root_table will be empty until the FHIR crawl is re-run "
              "from scratch. To get real MeSH data: delete "
              f"{PATHS['fhir_checkpoint']} and {PATHS['studies_raw']}, then "
              "re-run Section 1's crawl_all_studies cell (full re-crawl).")
        raw["mesh_terms_raw"] = None

    split_cols = raw["raw_accession"].apply(split_accession)
    raw = raw.copy()
    raw["root_phs"] = split_cols.apply(lambda t: t[0])
    raw["version_v"] = split_cols.apply(lambda t: t[1])
    raw["version_p"] = split_cols.apply(lambda t: t[2])
    resolved = raw.dropna(subset=["root_phs"]).copy()

    def collapse(group):
        versions = sorted({(int(v), int(p)) for v, p in zip(group["version_v"], group["version_p"])
                            if pd.notna(v) and pd.notna(p)})
        latest = max(versions) if versions else (None, None)
        # MeSH terms: take the union across versions (should usually be identical/stable)
        all_mesh = []
        for raw_mesh in group["mesh_terms_raw"].dropna():
            try:
                all_mesh.extend(json.loads(raw_mesh))
            except (ValueError, TypeError):
                pass
        unique_mesh = list({m["mesh_id"]: m for m in all_mesh if m.get("mesh_id")}.values())
        return pd.Series({
            "n_versions_seen": len(versions),
            "versions": json.dumps(versions),
            "latest_v": latest[0], "latest_p": latest[1],
            "latest_accession": (f"{group.name}.v{latest[0]}.p{latest[1]}" if latest[0] is not None else None),
            "titles_seen": json.dumps(sorted(set(group["title"].dropna().unique().tolist()))),
            "sponsors_seen": json.dumps(sorted(set(group["sponsor"].dropna().unique().tolist()))),
            "statuses_seen": json.dumps(sorted(set(group["status"].dropna().unique().tolist()))),
            "mesh_terms": json.dumps(unique_mesh),
            "n_mesh_terms": len(unique_mesh),
        })

    root_table = resolved.groupby("root_phs", group_keys=True).apply(collapse, include_groups=False).reset_index()
    root_table = root_table.rename(columns={"root_phs": "root_accession"})
    atomic_table(root_table, PATHS["root_accessions"])
    print(f"Built root accession table: {len(root_table)} unique root phs numbers.")
    return root_table

def run_fhir_source():
    state = prepare_source("fhir", ["studies_raw", "fhir_checkpoint"])
    cached = read_frame("studies_raw")
    cp = load_json(PATHS["fhir_checkpoint"], {})
    complete = os.path.isfile(PATHS["studies_raw"]) and cp.get("done", True)
    stamp = pd.to_datetime(state.get("last_successful_run_utc"), utc=True, errors="coerce")
    if pd.isna(stamp) and not cached.empty and "fetched_at" in cached:
        stamp = pd.to_datetime(cached.fetched_at, utc=True, errors="coerce").max()
    if pd.isna(stamp) and os.path.isfile(PATHS["studies_raw"]):
        stamp = pd.Timestamp(os.path.getmtime(PATHS["studies_raw"]), unit="s", tz="UTC")
        print("FHIR: legacy file mtime is an estimated check date.")
    workdir = os.path.join(DRIVE_ROOT, "notebook_a_work", "fhir")
    os.makedirs(workdir, exist_ok=True)
    raw_work = os.path.join(workdir, "dbgap_studies_raw.parquet")
    cp_work = os.path.join(workdir, "fhir_fetch_checkpoint.json")
    if mode_for("fhir") == "clean":
        for path in (raw_work, cp_work):
            if os.path.isfile(path):
                archive_registered([path], "fhir_interrupted_work")
    work = load_json(cp_work, {})
    resume = os.path.isfile(raw_work) and work and not work.get("done", False)
    due = needs_source_item(mode_for("fhir"), complete, stamp) or resume or state.get("force_check_next_run", False)
    if not due:
        print(f"FHIR: reusing {len(cached):,} saved version records; no network request.")
        RUN_REPORT.append({"block": "fhir", "mode": mode_for("fhir"), "attempted": 0, "rows": len(cached)})
        return cached
    if not resume:
        if cp and not cp.get("done", True) and not cached.empty:
            candidate = cached.copy()
            work = dict(cp)
        else:
            candidate = pd.DataFrame(columns=["fhir_id", "raw_accession", "title", "status", "sponsor",
                    "mesh_terms_raw", "identifiers_raw", "fetched_at"])
            work = {"next_url": None, "pages_fetched": 0, "done": False}
    else:
        candidate = pd.read_parquet(raw_work)
    url = work.get("next_url") or urljoin(FHIR_BASE, "ResearchStudy?_count=100")
    pages = 0
    seen = set()
    try:
        while url:
            if url in seen:
                raise RuntimeError("FHIR pagination repeated a URL")
            seen.add(url)
            bundle = fetch_bundle(url)
            if not isinstance(bundle, dict) or bundle.get("resourceType", "Bundle") != "Bundle":
                raise ValueError("Unexpected FHIR response, refusing to replace snapshot")
            rows = [parse_research_study(e["resource"]) for e in bundle.get("entry", []) if "resource" in e]
            if rows:
                candidate = pd.concat([candidate, pd.DataFrame(rows)], ignore_index=True).drop_duplicates("fhir_id",
                    keep="last")
            next_url = extract_next_link(bundle)
            if next_url:
                next_url = urljoin(url, next_url)
            work.update(next_url=next_url, pages_fetched=work.get("pages_fetched",0)+1, done=next_url is None)
            atomic_table(candidate, raw_work)
            save_json(cp_work, work)
            pages += 1
            print(f"FHIR page {work['pages_fetched']}: {len(candidate):,} version records")
            url = next_url
            if url:
                time.sleep(1.0)
        if candidate.empty:
            raise ValueError("FHIR snapshot unexpectedly empty; previous snapshot retained")
        if not rows_equal(cached, candidate, ignore_clock=True):
            atomic_table(candidate, PATHS["studies_raw"])
            _CHANGED_SOURCE_FILES.add(PATHS["studies_raw"])
        save_json(PATHS["fhir_checkpoint"], work)
        state["last_successful_run_utc"] = utc_now().isoformat()
        state.pop("force_check_next_run", None)
        state["last_attempt_status"] = "ok"
        _LAST_SOURCE_ERRORS["fhir"] = 0
        persist_state()
    except Exception as exc:
        state["last_attempt_status"] = str(exc)
        state["last_attempt_utc"] = utc_now().isoformat()
        _LAST_SOURCE_ERRORS["fhir"] = 1
        persist_state()
        if cached.empty or not complete:
            raise
        print(f"FHIR refresh failed; previous complete snapshot retained: {exc}")
    RUN_REPORT.append({"block": "fhir", "mode": mode_for("fhir"), "attempted": pages,
            "failed": _LAST_SOURCE_ERRORS.get("fhir",0)})
    return read_frame("studies_raw")

studies_raw = run_fhir_source()
brief_stats("FHIR version records", studies_raw, ["fhir_id"])
run_derived("root_table", ["studies_raw"], ["root_accessions"], build_root_accession_table)
root_table = read_frame("root_accessions")
require_columns(root_table, ["root_accession", "latest_accession"], "FHIR roots")
assert root_table.root_accession.is_unique
print(f"Current root universe: {len(root_table):,}; versioned targets: {root_table.latest_accession.notna().sum():,}")


## 2b. First-observed accession ledger

Prospective discovery dates; not historical release dates.

In [ ]:
# @title Record first locally observed root accession -- no source queries
# This new ledger is prospective. Its dates must never be backfilled with publication
# dates, first DAR, file mtimes, or the day a slide was rendered.
ledger_path=os.path.join(DRIVE_ROOT,'dbgap_first_observed_ledger.parquet')
prior=pd.read_parquet(ledger_path) if os.path.exists(ledger_path) else pd.DataFrame(columns=[
    'root_accession','first_observed_utc','observation_basis','root_source_sha256'])
seen=set(prior.root_accession.astype(str))
new_roots=[r for r in root_table.root_accession.astype(str) if r not in seen]
if new_roots:
    new=pd.DataFrame({'root_accession':new_roots,'first_observed_utc':utc_now().isoformat(),
        'observation_basis':'first seen by this acquisition ledger; not historical release',
        'root_source_sha256':sha256_file(PATHS['root_accessions'])})
    atomic_table(pd.concat([prior,new],ignore_index=True),ledger_path)
print(f'First-observed ledger: {len(seen)+len(new_roots):,} roots; {len(new_roots):,} newly recorded.')


## 3. Observed DAR records
Original approved-project download parser and status labels. A successful no-requester response has zero rows and a terminal checkpoint entry; transport failures are not equivalent.


In [ ]:
# @title 3. Observed DAR records
PRIMARY_DAR_URL_TMPL = "https://dbgap.ncbi.nlm.nih.gov/beta/api/study/{accession}/approved-projects-download/"
FALLBACK_DAR_URL = "https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/GetAuthorizedRequestDownload.cgi"
def _parse_dar_text(text, accession, source):
    first_line = text.split("\n", 1)[0]
    if "wlid" not in first_line:
        if "no authorized requester" in text.lower() or len(text) < 500:
            return pd.DataFrame(), "no_requesters"
        return None, "unexpected_format"
    try:
        df = pd.read_csv(io.StringIO(text), sep="\t", dtype=str)
    except Exception as exc:
        return None, f"parse_error:{exc}"
    df["queried_accession"] = accession
    df["source"] = source
    return df, "ok"

def fetch_dar_primary(accession, max_retries=5):
    delay = 2.0
    url = PRIMARY_DAR_URL_TMPL.format(accession=accession)
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        except requests.RequestException:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 404:
            return None, "http_404"
        if resp.status_code != 200:
            return None, f"http_{resp.status_code}"
        return _parse_dar_text(resp.text, accession, "primary_beta_api")
    return None, "exceeded_retries"

def fetch_dar_fallback(accession, max_retries=5):
    delay = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.get(FALLBACK_DAR_URL, params={"study_id": accession}, headers=HEADERS, timeout=TIMEOUT)
        except requests.RequestException:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code != 200:
            return None, f"http_{resp.status_code}"
        return _parse_dar_text(resp.text, accession, "fallback_legacy_cgi")
    return None, "exceeded_retries"

def fetch_dar_tsv(accession):
    df, note = fetch_dar_primary(accession)
    if df is not None:
        return df, note
    df2, note2 = fetch_dar_fallback(accession)
    if df2 is not None:
        return df2, f"primary_failed({note})_fallback_{note2}"
    return None, f"both_failed(primary={note}, fallback={note2})"

def dar_note_terminal(note):
    text = str(note)
    return text in {"ok",
        "no_requesters"} or text.endswith("_fallback_ok") or text.endswith("_fallback_no_requesters")

def known_dar(old):
    cp = checkpoint_map("dar_checkpoint", "done_accessions")
    present = set(old.queried_accession.dropna().astype(str)) if "queried_accession" in old else set()
    # Missing raw files cannot be reconstructed from positive checkpoints.
    return {k:v for k,v in cp.items() if dar_note_terminal(v) and (k in present or v.endswith("no_requesters"))}

def fetch_dar_group(targets):
    result = {}
    for key in targets:
        frame, note = fetch_dar_tsv(key)
        if frame is None or frame.empty:
            frame = pd.DataFrame(columns=["queried_accession", "wlid", "initial_approval_date", "status", "source"])
        result[key] = (frame, note, dar_note_terminal(note))
    return result

dar_records = collect_keyed("dar", "dar_records", "dar_checkpoint", "done_accessions",
    {str(a):str(a) for a in root_table.latest_accession.dropna()}, ["queried_accession"],
    fetch_dar_group, known_dar,
    empty_columns=["queried_accession", "wlid", "initial_approval_date", "status", "source"],
    pause=0.6, checkpoint_dict=True)


## 4. Study counts
A completed page lookup without studyCounts remains unavailable metadata. No zero count is inferred.


In [ ]:
# @title 4. Study counts
BETA_STUDY_URL_TMPL = "https://dbgap.ncbi.nlm.nih.gov/beta/study/{accession}/"
def fetch_study_counts(accession, max_retries=4):
    delay = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.get(BETA_STUDY_URL_TMPL.format(accession=accession), headers=HEADERS, timeout=TIMEOUT)
        except requests.RequestException:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code != 200:
            return {"accession": accession, "fetch_error": f"http_{resp.status_code}"}
        m = re.search(r"window\.studyCounts\s*=\s*(\{.*?\});", resp.text, re.DOTALL)
        if not m:
            return {"accession": accession, "fetch_error": "studyCounts_not_found"}
        raw_json = re.sub(r"(\w+)\s*:", r'"\1":', m.group(1))
        try:
            counts = json.loads(raw_json)
        except Exception as exc:
            return {"accession": accession, "fetch_error": f"parse_error:{exc}"}
        counts["accession"] = accession
        return counts
    return {"accession": accession, "fetch_error": "exceeded_retries"}

def fetch_counts_group(targets):
    result = {}
    for key in targets:
        row = fetch_study_counts(key)
        note = row.get("fetch_error") or "ok"
        # A completed lookup with no studyCounts is unavailable metadata, NOT a zero count.
        terminal = note in {"ok", "studyCounts_not_found", "http_404"}
        result[key] = (pd.DataFrame([row]), note, terminal)
    return result

study_counts_raw = collect_keyed("counts", "study_counts", "study_counts_checkpoint", "done_accessions",
    {str(a):str(a) for a in root_table.latest_accession.dropna()}, ["accession"], fetch_counts_group,
    lambda old: known_from_rows(old, ["accession"], error_column="fetch_error", accepted=("studyCounts_not_found",
            "http_404")),
    empty_columns=["accession", "fetch_error"], pause=0.6)


## 5. Study-page metadata
Collect publications, relationships, disease, type and design once. All six endpoint statuses participate in the check outcome; status bookkeeping is sidecar-only.


In [ ]:
# @title 5. Study-page metadata
BETA_API_BASE = "https://dbgap.ncbi.nlm.nih.gov/beta/api/study/"
_PAGE_ENDPOINT_NOTES = {}
def _extract_list(payload, list_key):
    if not isinstance(payload, dict):
        return None
    data = payload.get("data")
    if isinstance(data, dict) and list_key in data and isinstance(data[list_key], list):
        return data[list_key]
    if isinstance(data, list):
        return data
    if list_key in payload and isinstance(payload[list_key], list):
        return payload[list_key]
    if "results" in payload and isinstance(payload["results"], list):
        return payload["results"]
    return None

def fetch_beta_api_json(accession, endpoint, max_retries=4):
    url = f"{BETA_API_BASE}{accession}/{endpoint}/"
    delay = 2.0
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        except requests.RequestException:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 404:
            return None, "http_404"
        if resp.status_code != 200:
            return None, f"http_{resp.status_code}"
        try:
            return resp.json(), "ok"
        except ValueError:
            return None, "non_json_response"
    return None, "exceeded_retries"

def fetch_study_page_metadata(accession):
    row = {"accession": accession}

    pubs_payload, pubs_note = fetch_beta_api_json(accession, "publications")
    row["publications_raw"] = json.dumps(pubs_payload) if pubs_payload is not None else None
    row["publications_fetch_note"] = pubs_note
    pubs_list = _extract_list(pubs_payload, "publications")
    if pubs_list is not None:
        row["n_publications"] = len(pubs_list)
        row["pmids_found"] = json.dumps(sorted({str(p.get("pmid")).strip() for p in pubs_list if isinstance(p,
                    dict) and p.get("pmid")}))
    else:
        row["n_publications"] = None
        row["pmids_found"] = None

    parent_payload, parent_note = fetch_beta_api_json(accession, "parent")
    row["parent_raw"] = json.dumps(parent_payload) if parent_payload is not None else None
    row["parent_fetch_note"] = parent_note

    children_payload, children_note = fetch_beta_api_json(accession, "children")
    row["children_raw"] = json.dumps(children_payload) if children_payload is not None else None
    row["children_fetch_note"] = children_note
    children_list = _extract_list(children_payload, "children")
    row["n_children"] = len(children_list) if children_list is not None else None

    disease_payload, disease_note = fetch_beta_api_json(accession, "disease")
    disease_data = (disease_payload or {}).get("data", {}) if isinstance(disease_payload, dict) else {}
    row["primary_disease"] = disease_data.get("primary_disease")
    row["primary_disease_mesh_id"] = disease_data.get("primary_disease_mesh_id")

    type_payload, type_note = fetch_beta_api_json(accession, "type")
    type_data = (type_payload or {}).get("data", {}) if isinstance(type_payload, dict) else {}
    row["study_types"] = json.dumps(type_data.get("study_types"))

    design_payload, design_note = fetch_beta_api_json(accession, "design")
    design_data = (design_payload or {}).get("data", {}) if isinstance(design_payload, dict) else {}
    row["study_design"] = design_data.get("study_design")

    _PAGE_ENDPOINT_NOTES[accession] = [pubs_note, parent_note, children_note, disease_note, type_note, design_note]
    return row

def known_metadata(old):
    accepted = {"ok", "http_404"}
    if old.empty:
        return {}
    fields = [c for c in ("publications_fetch_note", "parent_fetch_note", "children_fetch_note") if c in old]
    if not fields:
        return {}
    mask = old[fields].isin(accepted).all(axis=1)
    return {str(k):"ok" for k in old.loc[mask, "accession"]}

def fetch_metadata_group(targets):
    result = {}
    for key in targets:
        row = fetch_study_page_metadata(key)
        notes = _PAGE_ENDPOINT_NOTES[key]
        terminal = all(n in {"ok", "http_404"} for n in notes)
        note = "ok" if terminal else "incomplete_endpoints:" + "|".join(notes)
        result[key] = (pd.DataFrame([row]), note, terminal)
    return result

study_page_raw = collect_keyed("metadata", "study_page_meta", "study_page_checkpoint", "done_accessions",
    {str(a):str(a) for a in root_table.latest_accession.dropna()}, ["accession"], fetch_metadata_group, known_metadata,
    empty_columns=["accession", "publications_raw", "pmids_found", "publications_fetch_note", "parent_fetch_note",
        "children_fetch_note"], pause=0.6)


## 6. SSTR summaries
Collect the original subject/sample summaries and status fields. HTTP 404 is a completed unavailable lookup; transient failures remain retryable.


In [ ]:
# @title 6. SSTR summaries
SSTR_BASE = "https://www.ncbi.nlm.nih.gov/gap/sstr/api/v1"
def fetch_sstr_summary(accession, max_retries=4):
    url = f"{SSTR_BASE}/study/{accession}/summary"
    delay = 1.5
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers={"Accept": "application/json"}, timeout=TIMEOUT)
        except requests.RequestException:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(delay); delay = min(delay * 2, 20); continue
        if resp.status_code == 404:
            return None, "http_404"
        if resp.status_code != 200:
            return None, f"http_{resp.status_code}"
        try:
            return resp.json(), "ok"
        except ValueError:
            return None, "non_json_response"
    return None, "exceeded_retries"

def parse_sstr_summary(accession, payload):
    row = {"accession": accession, "sstr_raw": json.dumps(payload)}
    if not isinstance(payload, dict):
        return row
    study = payload.get("study", {})
    accver = study.get("accver", {})
    row["sstr_root_accession"] = accver.get("accession")
    row["sstr_version"] = accver.get("version")
    row["sstr_participant_set"] = accver.get("participant_set")
    row["study_name"] = study.get("name")
    row["repository"] = study.get("repository")
    row["handle"] = study.get("handle")
    row["admin_ic"] = study.get("admin_ic")
    row["study_status"] = study.get("study_status")

    bioprojects = study.get("bioproject_ids") or []
    row["bioproject_ids"] = json.dumps([bp.get("id") for bp in bioprojects if isinstance(bp,
                dict) and bp.get("id")])
    row["bioproject_ids_raw"] = json.dumps(bioprojects)

    consent_groups = study.get("consent_groups") or []
    row["consent_groups"] = json.dumps(consent_groups)
    row["n_consent_groups"] = len(consent_groups)
    row["n_subjects_by_consent_sum"] = sum(cg.get("subject_count", 0) or 0 for cg in consent_groups)
    row["n_samples_by_consent_sum"] = sum(cg.get("sample_count", 0) or 0 for cg in consent_groups)

    row["sstr_parent"] = json.dumps(study.get("parent_study") or study.get("parent"))
    row["sstr_children"] = json.dumps(study.get("child_studies") or study.get("children"))

    stats = payload.get("study_stats", {})
    cnt_subjects = stats.get("cnt_subjects", {})
    cnt_samples = stats.get("cnt_samples", {})
    row["n_subjects_total"] = cnt_subjects.get("loaded")
    row["n_subjects_has_mol_data"] = cnt_subjects.get("has_mol_data")
    row["n_subjects_has_seq_data"] = cnt_subjects.get("has_seq_data")
    row["n_samples_total"] = cnt_samples.get("loaded")
    row["n_samples_has_mol_data"] = cnt_samples.get("has_mol_data")
    row["n_samples_has_seq_data"] = cnt_samples.get("has_seq_data")
    row["sequence_status"] = stats.get("sequence_status")

    consent_sample_use = stats.get("cnt_by_consent_and_sample_use") or []
    row["data_types_raw"] = json.dumps(consent_sample_use)
    distinct_data_types = sorted({r.get("sample_use") for r in consent_sample_use if isinstance(r,
                dict) and r.get("sample_use")})
    row["data_types"] = json.dumps(distinct_data_types)
    row["n_distinct_data_types"] = len(distinct_data_types)

    return row

def fetch_sstr_group(targets):
    result = {}
    for key in targets:
        payload, note = fetch_sstr_summary(key)
        row = parse_sstr_summary(key, payload) if payload is not None else {"accession":key,"sstr_raw":None}
        row["sstr_fetch_note"] = note
        result[key] = (pd.DataFrame([row]), note, note in {"ok", "http_404"})
    return result

sstr_raw = collect_keyed("sstr", "sstr_summary", "sstr_checkpoint", "done_accessions",
    {str(a):str(a) for a in root_table.latest_accession.dropna()}, ["accession"], fetch_sstr_group,
    lambda old: known_from_rows(old, ["accession"], "sstr_fetch_note", ("ok", "http_404")),
    empty_columns=["accession", "sstr_raw", "sstr_fetch_note"], pause=0.15, checkpoint_dict=True)


## 7. Direct ClinicalTrials.gov records
Candidate NCT mentions come from the current raw study-page view. This collection no longer depends on the joined analysis table.


In [ ]:
# @title 7. Direct ClinicalTrials.gov records
CTGOV_BASE = "https://clinicaltrials.gov/api/v2/studies"
NCT_PATTERN = re.compile(r"\bNCT\d{8}\b", re.IGNORECASE)
def extract_nct_ids(value):
    if value is None or pd.isna(value):
        return []
    return sorted({match.upper() for match in NCT_PATTERN.findall(str(value))})

def direct_nct_candidates_from_dbgaP(page_df):
    fields = ["publications_raw", "parent_raw", "children_raw", "primary_disease", "study_types", "study_design"]
    rows = []
    for _, row in page_df.iterrows():
        latest = str(row.get("accession", ""))
        root = latest.split(".")[0]
        for field in fields:
            if field not in page_df.columns:
                continue
            text = row.get(field)
            for nct_id in extract_nct_ids(text):
                rows.append({
                    "root_accession": root,
                    "latest_accession": latest,
                    "nct_id": nct_id,
                    "linkage_method": "direct_dbGaP_page_mention",
                    "dbgap_evidence_field": field,
                    "dbgap_evidence_text": str(text)[:1000],
                })
    columns = ["root_accession", "latest_accession", "nct_id", "linkage_method", "dbgap_evidence_field",
        "dbgap_evidence_text"]
    return pd.DataFrame(rows, columns=columns).drop_duplicates()

def fetch_ctgov_study(nct_id, max_retries=4):
    delay = 1.0
    url = f"{CTGOV_BASE}/{nct_id}"
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers={"Accept": "application/json"}, timeout=30)
        except requests.RequestException as exc:
            note = f"request_error:{type(exc).__name__}"
        else:
            if response.status_code == 200:
                try: return response.json(), "ok"
                except ValueError: return None, "non_json_response"
            if response.status_code in (429, 500, 502, 503, 504): note = f"http_{response.status_code}"
            else: return None, f"http_{response.status_code}"
        if attempt < max_retries - 1:
            time.sleep(delay); delay = min(delay * 2, 20)
    return None, note

def parse_ctgov_payload(payload):
    protocol = payload.get("protocolSection", {}) if isinstance(payload, dict) else {}
    ident = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    conditions = protocol.get("conditionsModule", {})
    return {
        "ctgov_brief_title": ident.get("briefTitle"),
        "ctgov_official_title": ident.get("officialTitle"),
        "ctgov_overall_status": status.get("overallStatus"),
        "ctgov_study_type": design.get("studyType"),
        "ctgov_phases": json.dumps(design.get("phases", [])),
        "ctgov_enrollment": (design.get("enrollmentInfo") or {}).get("count"),
        "ctgov_lead_sponsor": (sponsor.get("leadSponsor") or {}).get("name"),
        "ctgov_conditions": json.dumps(conditions.get("conditions", [])),
        "ctgov_start_date": (status.get("startDateStruct") or {}).get("date"),
        "ctgov_completion_date": (status.get("completionDateStruct") or {}).get("date"),
    }

def fetch_ct_group(targets):
    result = {}
    for key, candidate in targets.items():
        payload, note = fetch_ctgov_study(candidate["nct_id"])
        row = dict(candidate)
        row.update(ctgov_fetch_note=note, ctgov_fetched_utc=utc_now().isoformat())
        row.update(parse_ctgov_payload(payload or {}))
        result[key] = (pd.DataFrame([row]), note, note in {"ok", "http_404"})
    return result

clinical_trial_candidates = direct_nct_candidates_from_dbgaP(current_pages())
ct_targets = {r["latest_accession"]+"|"+r["nct_id"]:r for r in clinical_trial_candidates.to_dict("records")}
ct_columns = list(clinical_trial_candidates.columns) + ["ctgov_fetch_note",
    "ctgov_fetched_utc"] + list(parse_ctgov_payload({}))
clinical_trial_links = collect_keyed("clinical_trials", "clinical_trials_direct", "clinical_trials_direct_checkpoint", "done_pairs",
    ct_targets, ["latest_accession", "nct_id"], fetch_ct_group,
    lambda old: known_from_rows(old, ["latest_accession", "nct_id"], "ctgov_fetch_note", ("ok", "http_404")),
    empty_columns=ct_columns, pause=0.25)
if len(clinical_trial_links):
    print("Direct NCT links:", clinical_trial_links.root_accession.nunique(), "roots;",
        clinical_trial_links.nct_id.nunique(), "NCT identifiers")


## 8. PubMed records
PMIDs come directly from the current study-page metadata. The enriched root-PMID linkage is created later in the join section.


In [ ]:
# @title 8. PubMed records
PUBMED_BATCH_SIZE = 100
PUBMED_REQUEST_INTERVAL = 0.12 if NCBI_API_KEY else 0.36
PUBMED_EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
PUBMED_TOOL = "dbgap_accession_linkage"
def extract_pmids_from_page_metadata(page_df):
    rows = []
    for _, row in page_df.iterrows():
        raw = row.get("pmids_found")
        if raw is None or (not isinstance(raw, (list, tuple, dict)) and pd.isna(raw)):
            continue
        try:
            pmids = json.loads(raw) if isinstance(raw, str) else raw
        except (TypeError, ValueError):
            pmids = []
        if not isinstance(pmids, list):
            continue
        latest = str(row.get("accession", ""))
        root = latest.split(".")[0]
        for pmid in pmids:
            pmid = str(pmid).strip()
            if pmid.isdigit():
                rows.append({"root_accession": root, "latest_accession": latest, "pmid": pmid})
    return pd.DataFrame(rows, columns=["root_accession", "latest_accession", "pmid"]).drop_duplicates()

def xml_text(element, path):
    node = element.find(path)
    return "".join(node.itertext()).strip() if node is not None else None

def fetch_pubmed_metadata(pmids, max_retries=4):
    pmids = [str(p).strip() for p in pmids if str(p).strip().isdigit()]
    if not pmids:
        return pd.DataFrame()
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "xml", "tool": PUBMED_TOOL, "email": PUBMED_EMAIL}
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    delay = 1.0
    for attempt in range(max_retries):
        try:
            response = requests.get(PUBMED_EFETCH_URL, params=params, timeout=40)
        except requests.RequestException as exc:
            note = f"request_error:{type(exc).__name__}"
        else:
            if response.status_code == 200:
                try:
                    import xml.etree.ElementTree as ET
                    root = ET.fromstring(response.content)
                    rows = []
                    for article in root.findall(".//PubmedArticle"):
                        pmid = xml_text(article, "./MedlineCitation/PMID")
                        art = article.find("./MedlineCitation/Article")
                        journal = art.find("Journal") if art is not None else None
                        authors = []
                        if art is not None:
                            for author in art.findall("./AuthorList/Author"):
                                name = xml_text(author, "LastName") or xml_text(author, "CollectiveName")
                                if name: authors.append(name)
                        types = ["".join(x.itertext()).strip() for x in article.findall("./MedlineCitation/Article/PublicationTypeList/PublicationType")]
                        rows.append({"pmid": str(pmid), "pubmed_title": xml_text(article,
                                    "./MedlineCitation/Article/ArticleTitle"), "pubmed_journal": xml_text(journal,
                                    "Title") if journal is not None else None, "pubmed_pub_date": xml_text(article,
                                    "./MedlineCitation/Article/Journal/JournalIssue/PubDate/Year") or xml_text(article,
                                    "./MedlineCitation/Article/Journal/JournalIssue/PubDate/MedlineDate"),
                                "pubmed_authors": json.dumps(authors),
                                "pubmed_publication_types": json.dumps(types), "pubmed_fetch_note": "ok",
                                "pubmed_fetched_utc": datetime.now(timezone.utc).isoformat()})
                    returned = {r["pmid"] for r in rows}
                    rows.extend({"pmid": pmid, "pubmed_fetch_note": "not_returned",
                            "pubmed_fetched_utc": datetime.now(timezone.utc).isoformat()} for pmid in pmids if pmid not in returned)
                    return pd.DataFrame(rows)
                except Exception as exc:
                    return pd.DataFrame([{"pmid": pmid, "pubmed_fetch_note": f"parse_error:{type(exc).__name__}",
                                "pubmed_fetched_utc": datetime.now(timezone.utc).isoformat()} for pmid in pmids])
            note = f"http_{response.status_code}"
        if attempt < max_retries - 1:
            time.sleep(delay); delay = min(delay * 2, 20)
    return pd.DataFrame([{"pmid": pmid, "pubmed_fetch_note": note,
                "pubmed_fetched_utc": datetime.now(timezone.utc).isoformat()} for pmid in pmids])

def current_pmid_mapping():
    return extract_pmids_from_page_metadata(current_pages())

def corpus_pmids():
    return sorted(set(current_pmid_mapping().pmid.dropna().astype(str)))

def fetch_pubmed_group(targets):
    frame = fetch_pubmed_metadata(list(targets))
    result = {}
    for key in targets:
        part = frame.loc[frame.pmid.astype(str).eq(key)].copy() if "pmid" in frame else pd.DataFrame()
        note = str(part.pubmed_fetch_note.iloc[-1]) if len(part) else "missing_batch_result"
        result[key] = (part, note, note in {"ok", "not_returned"})
    return result

pubmed_metadata = collect_keyed("pubmed", "pubmed_metadata", "pubmed_checkpoint", "done_pmids",
    {p:p for p in corpus_pmids()}, ["pmid"], fetch_pubmed_group,
    lambda old: known_from_rows(old, ["pmid"], "pubmed_fetch_note", ("ok", "not_returned")),
    empty_columns=["pmid", "pubmed_title", "pubmed_journal", "pubmed_pub_date", "pubmed_authors",
        "pubmed_publication_types", "pubmed_fetch_note", "pubmed_fetched_utc"],
    batch_size=PUBMED_BATCH_SIZE, pause=PUBMED_REQUEST_INTERVAL)


## 8b. Public attribution and NIH acknowledgement evidence

Exact public study sections are captured with source hashes. Failed parsing is not a negative acknowledgement. Name/affiliation splits are displayed-label proxies, not validated identities.

In [ ]:
# @title Extra collection for the ten-slide deck: public attribution and funding evidence
# New sidecars only: existing analytical tables and checkpoint schemas are unchanged.
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import gzip

PATHS.update({
    'public_attribution': os.path.join(DRIVE_ROOT, 'dbgap_public_attribution.parquet'),
    'public_attribution_checkpoint': os.path.join(DRIVE_ROOT, 'dbgap_public_attribution_checkpoint.json'),
    'pubmed_grants': os.path.join(DRIVE_ROOT, 'pubmed_grant_acknowledgements.parquet'),
    'pubmed_grants_checkpoint': os.path.join(DRIVE_ROOT, 'pubmed_grant_acknowledgements_checkpoint.json'),
})
VOLATILE_COLUMNS.update({'attribution_fetched_utc', 'grant_fetched_utc'})
PUBLIC_EVIDENCE_DIR = Path(DRIVE_ROOT) / 'public_evidence'
PUBLIC_EVIDENCE_DIR.mkdir(exist_ok=True)
NIH_EVIDENCE_RE = re.compile(
    r'\b(?:National Institutes of Health|NIH|NHLBI|NHGRI|NIAID|NIGMS|NIDDK|NINDS|NIMH|NICHD|NIDA|NIAAA|NCI|NEI|NIEHS|NIDCR|NIDCD|NIBIB|NIMHD|NINR|NCCIH|NCATS|NCRR|NIA)\b', re.I)

def record_public_response(content, suffix):
    checksum = hashlib.sha256(content).hexdigest()
    path = PUBLIC_EVIDENCE_DIR / (checksum + suffix + '.gz')
    if not path.exists():
        with gzip.open(path, 'wb') as f: f.write(content)
    return checksum, str(path.relative_to(Path(DRIVE_ROOT)))

def attribution_identity(text):
    """Mechanical displayed-label split, not person/institution resolution."""
    text = re.sub(r'\s+', ' ', text).strip()
    split = re.split(r'\.\s+(?=[A-Z])', text, maxsplit=1)
    if len(split) == 2 and len(split[0].split()) >= 2 and re.search(
        r'University|Institute|Hospital|Center|Centre|College|School|Department|Laborator|Foundation|Consortium', split[1], re.I):
        return split[0], split[1], 'displayed_label_split_not_entity_resolution'
    return text, '', 'unsplit_displayed_label'

def parse_public_attribution(markup, accession):
    soup = BeautifulSoup(markup, 'html.parser')
    # Only role-labelled list items are eligible; never scan NIH navigation/footer text.
    records = []
    for item in soup.find_all('li'):
        nested = item.find(['ul','ol'], recursive=False)
        if nested is None: continue
        role = ' '.join(x.get_text(' ', strip=True) if getattr(x, 'name', None) else str(x)
                        for x in item.contents if x is not nested)
        role = re.sub(r'\s+', ' ', role).strip(' :\n')
        if not re.search(r'principal investigator|co.investigator|submitter|submitting|funding source|collaborat|consortium|data coordinating', role, re.I):
            continue
        for child in nested.find_all('li', recursive=False):
            text = child.get_text(' ', strip=True)
            if not text: continue
            person, organization, rule = attribution_identity(text)
            records.append({'role':role, 'text':text, 'person_or_group_label':person,
                            'organization_label':organization, 'parse_rule':rule})
    # Explicitly headed acknowledgement paragraphs; retain exact displayed text.
    acknowledgements = []
    for tag in soup.find_all(['p','dd','div']):
        text = tag.get_text(' ', strip=True)
        if len(text) > 20000: continue
        if re.match(r'^Acknowledg(?:e)?ment Statement\s*:', text, re.I):
            if not any(re.match(r'^Acknowledg(?:e)?ment Statement\s*:', c.get_text(' ',strip=True), re.I)
                       for c in tag.find_all(['p','dd','div'])):
                acknowledgements.append(text)
    funding = [r['text'] for r in records if re.search('funding', r['role'], re.I)]
    evidence = funding + acknowledgements
    hits = [s for s in evidence if NIH_EVIDENCE_RE.search(s)]
    # Positive = explicit mention in a funding/acknowledgement section. Negative is only
    # no such mention in an observed section; never a claim that the study was unfunded.
    indicator = True if hits else (False if evidence else pd.NA)
    return {'accession':accession, 'root_accession':accession.split('.')[0],
            'attributions_json':json.dumps(records, ensure_ascii=False),
            'funding_sources_json':json.dumps(funding, ensure_ascii=False),
            'acknowledgement_text': '\n'.join(acknowledgements),
            'nih_funding_acknowledgement':indicator,
            'nih_funding_evidence_json':json.dumps(hits, ensure_ascii=False),
            'attribution_section_parsed':bool(records),
            'funding_section_observed':bool(evidence)}

def fetch_attribution_group(targets):
    output = {}
    for accession in targets:
        url = 'https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id=' + accession
        try:
            r = requests.get(url, timeout=40, headers={'User-Agent':HEADERS['User-Agent']})
            if r.status_code != 200:
                output[accession] = (pd.DataFrame(), f'http_{r.status_code}', False); continue
            if accession.split('.')[0].lower() not in r.text.lower():
                output[accession] = (pd.DataFrame(), 'wrong_page_or_blocked', False); continue
            row = parse_public_attribution(r.text, accession)
            row['response_sha256'], row['raw_response_file'] = record_public_response(r.content, '.html')
            row.update(source_url=url, attribution_fetch_note='ok' if row['attribution_section_parsed'] else 'ok_unparsed',
                       attribution_fetched_utc=utc_now().isoformat())
            output[accession] = (pd.DataFrame([row]), row['attribution_fetch_note'], True)
        except requests.RequestException as exc:
            output[accession] = (pd.DataFrame(), 'request_error:'+type(exc).__name__, False)
    return output

if ENABLE_PUBLIC_ATTRIBUTION:
    public_attribution = collect_keyed('public_attribution','public_attribution',
        'public_attribution_checkpoint','done_accessions',
        {str(a):str(a) for a in root_table.latest_accession.dropna()}, ['accession'], fetch_attribution_group,
        lambda old:known_from_rows(old,['accession'],'attribution_fetch_note',('ok','ok_unparsed')),
        empty_columns=['accession','root_accession','attributions_json','nih_funding_acknowledgement','attribution_fetch_note'], pause=0.36)
    brief_stats('Public study attribution', public_attribution, ['accession'])
    if len(public_attribution):
        print('Parsed attribution:', int(public_attribution.attribution_section_parsed.fillna(False).sum()),
              '; explicit NIH mentions:', int(public_attribution.nih_funding_acknowledgement.fillna(False).sum()))
        print('PI/affiliation labels are NOT verified uploaders or canonical institutions.')
else:
    print('Public attribution collection disabled; existing sidecars remain available to B.')

# Optional independent evidence stream. Never silently relabel it as study funding.
def parse_grant_xml(content):
    tree=ET.fromstring(content); result={}
    for article in tree.findall('.//PubmedArticle'):
        pmid=article.findtext('./MedlineCitation/PMID')
        if not pmid: continue
        grants=[]
        for grant in article.findall('./MedlineCitation/Article/GrantList/Grant'):
            grants.append({key:grant.findtext(tag) for key,tag in [('grant_id','GrantID'),('agency','Agency'),('country','Country')]})
        # NIH agency names/NIH-labelled agencies only, not arbitrary contract patterns.
        nih=[g for g in grants if NIH_EVIDENCE_RE.search(g.get('agency') or '')]
        result[pmid]={'pmid':pmid,'grants_json':json.dumps(grants),'has_nih_grantlist':bool(nih),
                      'n_grants':len(grants),'grant_fetch_note':'ok'}
    return result

def fetch_grant_group(targets):
    params={'db':'pubmed','id':','.join(targets),'retmode':'xml','tool':PUBMED_TOOL}
    if PUBMED_EMAIL and PUBMED_EMAIL != 'research@example.org':params['email']=PUBMED_EMAIL
    if NCBI_API_KEY: params['api_key']=NCBI_API_KEY
    try:
        r=requests.get(PUBMED_EFETCH_URL,params=params,timeout=45);r.raise_for_status()
        parsed=parse_grant_xml(r.content)
        sha,rawfile=record_public_response(r.content,'.xml')
    except (requests.RequestException,ET.ParseError) as exc:
        return {p:(pd.DataFrame(),'failed:'+type(exc).__name__,False) for p in targets}
    output={}
    for p in targets:
        if p not in parsed:
            output[p]=(pd.DataFrame(),'not_returned',False);continue
        row=parsed[p];row.update(response_sha256=sha,raw_response_file=rawfile,grant_fetched_utc=utc_now().isoformat())
        output[p]=(pd.DataFrame([row]),'ok',True)
    return output

if ENABLE_PUBMED_GRANTS:
    grant_records=collect_keyed('pubmed_grants','pubmed_grants','pubmed_grants_checkpoint','done_pmids',
        {p:p for p in corpus_pmids()},['pmid'],fetch_grant_group,
        lambda old:known_from_rows(old,['pmid'],'grant_fetch_note',('ok',)),
        empty_columns=['pmid','grants_json','has_nih_grantlist','n_grants','grant_fetch_note'],
        batch_size=100,pause=PUBMED_REQUEST_INTERVAL)
    brief_stats('PubMed GrantList acknowledgements',grant_records,['pmid'])
else:
    print('Optional PubMed GrantList collection off; RePORTER is not substituted for acknowledgement.')


## 9. Authorized Access

Retains the optimized one-attempt collector, persistent failure cooldowns and per-run failure budget. Successful cached rows and original schemas are preserved.


In [ ]:
# @title 9. Authorized Access -- cached failures, bounded requests, unchanged output schema
# Replace the whole old section 9 cell with this cell; keep the shared setup above it.
# Only this source is affected. No A/B/C table or checkpoint field is renamed.
AA_API_TEMPLATE = "https://dbgap.ncbi.nlm.nih.gov/beta/api/study/{accession}/authorized-access/"
AA_API_HEADERS = {"Accept": "application/json", "User-Agent": "dbgap-accession-dar-research/1.2"}
AA_API_TIMEOUT = (6.1, 30)  # Connect/read inactivity timeouts; NOT a wall-clock deadline.
AA_API_SLEEP = 1.0         # Keep conservative pacing; no concurrent requests.
AA_RETRY_COOLDOWN_DAYS = (1, 7, 30)  # First / second / subsequent failed runs, same version.
AA_MAX_FAILURES_PER_RUN = 10        # Stop this source, not the notebook. None disables the cap.
AA_RETRY_FAILED_NOW = False        # Bypass local failure cooldown only; never Retry-After.
AA_SAVE_EVERY = 10                 # Save source data BEFORE successful checkpoints.

from email.utils import parsedate_to_datetime

def normalize_accession(value):
    """
    Use a versioned latest accession when available. The beta API endpoint
    accepts versioned forms such as phs001345.v1.p1.
    """
    if value is None or (
        isinstance(value, float) and pd.isna(value)
    ):
        return None

    value = str(value).strip()

    return value if value else None

def normalize_unique_strings(values):
    """Return ordered unique nonempty strings."""
    seen = set()
    output = []

    for value in values:
        if value is None:
            continue

        value = str(value).strip()

        if (
            value
            and value.lower() not in {
                "none",
                "nan",
                "null",
            }
            and value.lower() not in seen
        ):
            output.append(value)
            seen.add(value.lower())

    return output

def aa_retry_after(value, now):
    """Interpret either HTTP Retry-After form; invalid/past values impose no delay."""
    if value is None:
        return pd.NaT
    value = str(value).strip()
    try:
        if re.fullmatch(r"[0-9]+", value):
            return now + pd.Timedelta(seconds=int(value))
        stamp = pd.Timestamp(parsedate_to_datetime(value))
        stamp = stamp.tz_localize("UTC") if stamp.tzinfo is None else stamp.tz_convert("UTC")
        return stamp if stamp > now else pd.NaT
    except (ValueError, TypeError, OverflowError):
        return pd.NaT


def fetch_authorized_access_once(session, accession):
    """One request, with no hidden adapter retries or in-cell exponential sleeps.

    Return payload, original-style fetch note, URL, and optional endpoint pause.
    A failed response remains an error/unknown, never a platform-negative result.
    """
    url = AA_API_TEMPLATE.format(accession=accession)
    try:
        with session.get(url, timeout=AA_API_TIMEOUT) as response:
            code = response.status_code
            now = utc_now()
            retry_at = aa_retry_after(response.headers.get("Retry-After"), now)
            if code == 200:
                try:
                    payload = response.json()
                except ValueError:
                    return None, "non_json_response", url, pd.NaT
                # An HTTP 200 error envelope must not turn missing fields into False.
                if not (isinstance(payload, dict) and isinstance(payload.get("data"), dict)
                        and isinstance(payload["data"].get("study_info"), dict)
                        and payload["data"]["study_info"]):
                    return None, "invalid_payload", url, pd.NaT
                return payload, "ok", url, pd.NaT
            if code == 404:
                # Existing terminal HTTP-404 policy retained; no platform flags are filled.
                return None, "http_404", url, pd.NaT
            if code in {429, 503} and pd.isna(retry_at):
                retry_at = now + pd.Timedelta(minutes=15)
            if code != 429 and code < 500:
                retry_at = pd.NaT
            return None, f"http_{code}", url, retry_at
    except requests.RequestException as exc:
        return None, f"request_error:{type(exc).__name__}", url, pd.NaT


def aa_failure_state(entry, row, accession):
    """Recover prior failure without inventing repeated-run counts for legacy logs."""
    entry = dict(entry or {})
    row = row or {}
    note = str(entry.get("last_attempt_status", row.get("authorized_access_fetch_note", "")))
    failed = bool(entry.get("last_attempt_failed", bool(note and note not in {"ok", "http_404"})))
    version = entry.get("last_attempt_source_version")
    if version is None:
        version = row.get("latest_accession_used", entry.get("source_version"))
    # No evidence that the error was for this version => do not suppress its first check.
    failed = failed and version is not None and str(version) == str(accession)
    stamp = pd.to_datetime(entry.get("last_attempt_utc", row.get("authorized_access_fetched_utc")),
                           utc=True, errors="coerce")
    try:
        runs = max(1, int(entry.get("consecutive_failed_runs", 1))) if failed else 0
    except (ValueError, TypeError):
        runs = 1 if failed else 0
    return failed, stamp, runs


def aa_failure_deadline(entry, row, accession, now):
    failed, stamp, runs = aa_failure_state(entry, row, accession)
    if not failed or pd.isna(stamp):
        return pd.NaT
    # A future legacy clock is suspect; recheck rather than suppress indefinitely.
    if stamp > now:
        return pd.NaT
    saved = pd.to_datetime(entry.get("next_retry_utc"), utc=True, errors="coerce")
    if pd.notna(saved):
        return saved
    days = AA_RETRY_COOLDOWN_DAYS[min(runs - 1, len(AA_RETRY_COOLDOWN_DAYS) - 1)]
    return stamp + pd.Timedelta(days=days)


def extract_platform_flags(external_datasources):
    """
    Classify platforms only from the API's explicit external_datasources list.
    dbGaP is separately added as the native platform for released studies.
    """
    names = [
        item.get("name")
        for item in external_datasources
        if isinstance(item, dict)
    ]

    urls = [
        item.get("download_url")
        for item in external_datasources
        if isinstance(item, dict)
    ]

    combined = " | ".join(
        normalize_unique_strings(names + urls)
    ).lower()

    return {
        "platform_biodata_catalyst": bool(
            "biodata catalyst" in combined
            or "biodatacatalyst.nhlbi.nih.gov" in combined
        ),
        "platform_anvil": bool(
            "anvil" in combined
            or "anvilproject.org" in combined
            or "terra.bio" in combined
        ),
        "platform_kids_first": bool(
            "kids first" in combined
            or "kidsfirstdrc.org" in combined
        ),
        "platform_crdc": bool(
            "cancer research data commons" in combined
            or "crdc" in combined
            or "cancerdata.nih.gov" in combined
        ),
        "platform_gdc": bool(
            "genomic data commons" in combined
            or "gdc.cancer.gov" in combined
        ),
        "platform_ctdc": bool(
            "clinical and translational data commons" in combined
            or "ctdc" in combined
        ),
        "platform_all_of_us": bool(
            "all of us" in combined
            or "researchallofus.org" in combined
        ),
        "platform_ega": bool(
            "european genome-phenome archive" in combined
            or "ega-archive.org" in combined
        ),
        "platform_immport": bool(
            "immport" in combined
            or "immport.org" in combined
        ),
        "platform_nda": bool(
            "nimh data archive" in combined
            or "nda.nih.gov" in combined
        ),
    }

def parse_authorized_access_payload(
    root_accession,
    latest_accession,
    payload,
    fetch_note,
    url,
):
    """
    Flatten one successful or unsuccessful API result to one root-accession row.
    """
    base = {
        "root_accession": str(root_accession),
        "latest_accession_used": latest_accession,
        "authorized_access_api_url": url,
        "authorized_access_fetch_note": fetch_note,
        "authorized_access_fetched_utc": (
            datetime.now(timezone.utc).isoformat()
        ),
    }

    if fetch_note != "ok" or payload is None:
        return base

    data = payload.get("data", {}) or {}

    study_info = data.get("study_info", {}) or {}

    consent_groups = (
        study_info.get("consent_group", []) or []
    )

    external_datasources = (
        study_info.get("external_datasources", []) or []
    )

    if not isinstance(consent_groups, list):
        consent_groups = []

    if not isinstance(external_datasources, list):
        external_datasources = []

    # Consent-level SRA availability is the strongest structured signal
    # available in this endpoint for SRA-associated data availability.
    sra_flags = [
        bool(group.get("sra_available"))
        for group in consent_groups
        if isinstance(group, dict)
    ]

    consent_accessions = normalize_unique_strings(
        [
            group.get("accession")
            for group in consent_groups
            if isinstance(group, dict)
        ]
    )

    consent_names = normalize_unique_strings(
        [
            group.get("name")
            for group in consent_groups
            if isinstance(group, dict)
        ]
    )

    dac_names = normalize_unique_strings(
        [
            group.get("long_name")
            for group in consent_groups
            if isinstance(group, dict)
        ]
    )

    dac_emails = normalize_unique_strings(
        [
            group.get("email")
            for group in consent_groups
            if isinstance(group, dict)
        ]
    )

    external_names = normalize_unique_strings(
        [
            item.get("name")
            for item in external_datasources
            if isinstance(item, dict)
        ]
    )

    external_download_urls = normalize_unique_strings(
        [
            item.get("download_url")
            for item in external_datasources
            if isinstance(item, dict)
        ]
    )

    external_help_urls = normalize_unique_strings(
        [
            item.get("help_url")
            for item in external_datasources
            if isinstance(item, dict)
        ]
    )

    external_ftp_codes = normalize_unique_strings(
        [
            item.get("ftp")
            for item in external_datasources
            if isinstance(item, dict)
        ]
    )

    release_status = study_info.get("release_status")

    is_released = (
        isinstance(release_status, str)
        and release_status.strip().lower() == "released"
    )

    platform_flags = extract_platform_flags(
        external_datasources
    )

    # dbGaP is treated as the native distribution platform for released studies.
    # This is a documented convention from the frontend logic, which prepends
    # dbGaP to external_datasources for released studies.
    platform_dbgap = is_released

    return {
        **base,
        "release_status": release_status,
        "is_released": is_released,
        "gsr_mode": data.get("gsr_mode"),
        "ega_accession": data.get("ega_accession"),
        "has_substudies": data.get("has_substudies"),
        "is_collection": data.get("is_collection"),
        "alternate_comment": data.get(
            "alternate_comment"
        ),
        "additional_comment": data.get(
            "additional_comment"
        ),
        "release_date": study_info.get("release_date"),
        "n_consent_groups_authorized_access": len(
            consent_groups
        ),
        "any_consent_group_sra_available": any(
            sra_flags
        ),
        "all_consent_groups_sra_available": (
            bool(sra_flags) and all(sra_flags)
        ),
        "n_consent_groups_sra_available": int(
            sum(sra_flags)
        ),
        "consent_group_accessions": json.dumps(
            consent_accessions
        ),
        "consent_group_names": json.dumps(consent_names),
        "dac_names": json.dumps(dac_names),
        "dac_emails": json.dumps(dac_emails),
        "external_datasource_names": json.dumps(
            external_names
        ),
        "external_datasource_download_urls": json.dumps(
            external_download_urls
        ),
        "external_datasource_help_urls": json.dumps(
            external_help_urls
        ),
        "external_datasource_ftp_codes": json.dumps(
            external_ftp_codes
        ),
        "n_external_datasources": len(
            external_datasources
        ),
        "platform_dbgap": platform_dbgap,
        **platform_flags,
    }

def collect_authorized_access(targets):
    """AA-only scheduling; reuses A's storage helpers and existing control files."""
    if not AA_RETRY_COOLDOWN_DAYS or any(float(x) < 0 for x in AA_RETRY_COOLDOWN_DAYS):
        raise ValueError("AA_RETRY_COOLDOWN_DAYS must contain nonnegative day intervals.")
    if AA_MAX_FAILURES_PER_RUN is not None and (not isinstance(AA_MAX_FAILURES_PER_RUN, int)
            or isinstance(AA_MAX_FAILURES_PER_RUN, bool) or AA_MAX_FAILURES_PER_RUN < 1):
        raise ValueError("AA_MAX_FAILURES_PER_RUN must be a positive integer or None.")
    if AA_SAVE_EVERY < 1 or AA_API_SLEEP < 0:
        raise ValueError("AA_SAVE_EVERY must be positive; AA_API_SLEEP must be nonnegative.")
    block = "authorized_access"
    mode = mode_for(block)
    artifact_key = "authorized_access_metadata"
    checkpoint_key = "authorized_access_checkpoint"
    artifact_path, checkpoint_path = PATHS[artifact_key], PATHS[checkpoint_key]
    # Even a clean rebuild does not override an active server-requested pause.
    prior_state = PIPELINE_STATE["sources"].get(block, {})
    prior_pause = prior_state.get("endpoint_retry_after_utc")
    state = prepare_source(block, [artifact_key, checkpoint_key])
    if prior_pause:
        state["endpoint_retry_after_utc"] = prior_pause
    empty_columns = list(parse_authorized_access_payload("", "", {"data": {"study_info": {}}}, "ok", "").keys())
    old = read_frame(artifact_key, empty_columns)
    require_columns(old, ["root_accession"], block)
    if old["root_accession"].isna().any() or old["root_accession"].astype(str).duplicated().any():
        raise ValueError("Authorized Access cache must contain one non-null row per root; no data changed.")
    row_lookup = {str(r["root_accession"]): r for r in old.to_dict("records")}
    done = known_from_rows(old, ["root_accession"], "authorized_access_fetch_note", ("ok", "http_404"))
    cp = load_json(checkpoint_path, {})
    start_now = utc_now()
    fetch_times = {k: pd.to_datetime(r.get("authorized_access_fetched_utc"), utc=True, errors="coerce")
                   for k, r in row_lookup.items()}
    fetch_times = {k: v for k, v in fetch_times.items() if pd.notna(v)}
    mtime_path = artifact_path if os.path.isfile(artifact_path) else checkpoint_path
    legacy_mtime = (pd.Timestamp(os.path.getmtime(mtime_path), unit="s", tz="UTC")
                   if os.path.isfile(mtime_path) else pd.NaT)
    before_planning = json.dumps(state, sort_keys=True, default=str)
    if state.get("force_check_next_run") and not state.get("force_check_started_utc"):
        state["force_check_started_utc"] = start_now.isoformat()
    force_started = pd.to_datetime(state.get("force_check_started_utc"), utc=True, errors="coerce")
    selected, cooldown, age_sources = [], {}, {}
    reused = 0
    for item, accession in targets.items():
        entry = state.setdefault("items", {}).setdefault(item, {})
        row = row_lookup.get(item, {})
        stamp, basis = item_last_check(state, item, old, ["root_accession"], artifact_path,
                                      fetch_times=fetch_times, legacy_mtime=legacy_mtime)
        age_sources[basis] = age_sources.get(basis, 0) + 1
        version = entry.get("source_version", row.get("latest_accession_used"))
        same_version = version is None or str(version) == str(accession)
        prior_failed, attempt_date, _ = aa_failure_state(entry, row, accession)
        force_due = bool(state.get("force_check_next_run")) and (
            pd.isna(stamp) or pd.isna(force_started) or stamp < force_started)
        needed = needs_source_item(mode, item in done and not prior_failed and not force_due,
                                   stamp, same_version=same_version, now=start_now)
        if not needed:
            reused += 1
            continue
        deadline = aa_failure_deadline(entry, row, accession, start_now)
        if mode != "clean" and not AA_RETRY_FAILED_NOW and pd.notna(deadline) and deadline > start_now:
            cooldown[item] = deadline
            continue
        # New versions and never-attempted items first; then oldest attempts.
        # A failed prefix cannot permanently starve the rest of the accession list.
        rank = 0 if not same_version or pd.isna(attempt_date) else 1
        order_time = attempt_date.value if pd.notna(attempt_date) else 0
        selected.append((rank, order_time, item))
    selected = [x[2] for x in sorted(selected)]
    print(f"{block} [{mode}]: {len(targets):,} targets; {reused:,} reused; "
          f"{len(cooldown):,} deferred by failure cooldown; {len(selected):,} due.")
    if cooldown:
        print(f"  Earliest local retry: {min(cooldown.values()).isoformat()}")
    if age_sources.get("legacy_file_mtime_estimate", 0) and mode == "check_stale":
        print("  Legacy freshness uses file mtime as an estimate, not a verified check date.")
    if json.dumps(state, sort_keys=True, default=str) != before_planning:
        persist_state()  # Freezes legacy anchors only; does not advance successful checks.

    changes, pending_entries = {}, {}
    checked = failed = attempted = 0
    stop_reason = None

    def flush():
        nonlocal old, changes, pending_entries
        if not changes and not pending_entries:
            return
        if changes:
            candidate = replace_response_groups(old, changes, ["root_accession"])
            if atomic_table(candidate, artifact_path):
                _CHANGED_SOURCE_FILES.add(artifact_path)
            old = candidate
        # Never checkpoint a row before its successful write.
        new_cp = dict(cp)
        new_cp["done_roots"] = sorted(done)
        if new_cp != cp or not os.path.exists(checkpoint_path):
            save_json(checkpoint_path, new_cp)
            cp.clear()
            cp.update(new_cp)
        state["items"].update(pending_entries)
        if pending_entries:
            state["last_attempt_utc"] = max(x["last_attempt_utc"] for x in pending_entries.values())
        persist_state()
        changes, pending_entries = {}, {}

    try:
        if selected:
            with requests.Session() as session:
                session.headers.update(AA_API_HEADERS)
                session.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
                for item in selected:
                    pause_until = pd.to_datetime(state.get("endpoint_retry_after_utc"), utc=True, errors="coerce")
                    if pd.notna(pause_until) and pause_until > utc_now():
                        stop_reason = f"endpoint pause until {pause_until.isoformat()}"
                        break
                    if AA_MAX_FAILURES_PER_RUN is not None and failed >= AA_MAX_FAILURES_PER_RUN:
                        stop_reason = f"per-run limit of {AA_MAX_FAILURES_PER_RUN} failed accessions"
                        break
                    if attempted and AA_API_SLEEP:
                        time.sleep(AA_API_SLEEP)
                    accession = targets[item]
                    payload, note, url, retry_at = fetch_authorized_access_once(session, accession)
                    attempted += 1
                    try:
                        row = parse_authorized_access_payload(item, accession, payload, note, url)
                    except (ValueError, TypeError, KeyError, AttributeError) as exc:
                        note = f"parse_error:{type(exc).__name__}"
                        row = parse_authorized_access_payload(item, accession, None, note, url)
                    new = pd.DataFrame([row])
                    terminal = note in {"ok", "http_404"}
                    entry = dict(state["items"].get(item, {}))
                    was_failed, _, count = aa_failure_state(entry, row_lookup.get(item, {}), accession)
                    attempt_now = utc_now()
                    entry.update(last_attempt_utc=attempt_now.isoformat(), last_attempt_status=note,
                                 last_attempt_failed=not terminal, last_attempt_source_version=accession)
                    if terminal:
                        previous = old.loc[old["root_accession"].astype(str).eq(item)]
                        if not rows_equal(previous, new, ignore_clock=True):
                            changes[item] = new
                        entry.update(last_checked_utc=attempt_now.isoformat(), source_version=accession,
                                     consecutive_failed_runs=0)
                        entry.pop("next_retry_utc", None)
                        done[item] = note
                        checked += 1
                    else:
                        count = count + 1 if was_failed else 1
                        days = AA_RETRY_COOLDOWN_DAYS[min(count - 1, len(AA_RETRY_COOLDOWN_DAYS) - 1)]
                        deadline = attempt_now + pd.Timedelta(days=days)
                        if pd.notna(retry_at) and retry_at > deadline:
                            deadline = retry_at
                        entry.update(consecutive_failed_runs=count, next_retry_utc=deadline.isoformat())
                        failed += 1
                        # Existing rows, especially good prior data, remain byte-for-byte
                        # unchanged when a refresh fails. Persist only first-ever error rows.
                        if item not in row_lookup:
                            changes[item] = new
                        print(f"  DEFER {item}: {note}; retry not before {deadline.isoformat()}")
                    pending_entries[item] = entry
                    if pd.notna(retry_at) and retry_at > attempt_now:
                        state["endpoint_retry_after_utc"] = retry_at.isoformat()
                        stop_reason = f"endpoint pause until {retry_at.isoformat()}"
                        flush()
                        break
                    if attempted % AA_SAVE_EVERY == 0:
                        flush()
                        print(f"  {attempted:,}/{len(selected):,} due targets attempted; {failed:,} failed.")
    finally:
        # Preserve completed work on interruption. No false attempt is written for
        # queued targets that were never requested.
        flush()

    deferred_queue = len(selected) - attempted
    unresolved = sum(k not in done for k in targets)
    retry_pending = sum(aa_failure_state(state["items"].get(k, {}), row_lookup.get(k, {}), v)[0]
                        for k, v in targets.items())
    if attempted and not failed and not cooldown and not deferred_queue and not unresolved:
        state["last_successful_run_utc"] = utc_now().isoformat()
        state.pop("force_check_next_run", None)
        state.pop("force_check_started_utc", None)
        persist_state()
    if not targets and not os.path.isfile(artifact_path):
        if atomic_table(old, artifact_path):
            _CHANGED_SOURCE_FILES.add(artifact_path)
    _LAST_SOURCE_ERRORS[block] = failed
    report = {"block": block, "mode": mode, "targets": len(targets), "attempted": attempted,
              "checked": checked, "failed": failed, "remaining": unresolved, "rows": len(old),
              "age_basis": age_sources, "reused": reused, "deferred_cooldown": len(cooldown),
              "deferred_queue": deferred_queue, "retry_pending": retry_pending, "stop_reason": stop_reason}
    RUN_REPORT.append(report)
    if stop_reason:
        print(f"  STOPPED AA only: {stop_reason}; {deferred_queue:,} due targets not attempted.")
    print(f"  Target GET calls: {attempted:,}; successful/terminal checks: {checked:,}; "
          f"failed: {failed:,}; unresolved (no terminal cache): {unresolved:,}.")
    brief_stats(block, old, ["root_accession"], done=done, targets=targets)
    return old


aa_targets = {str(r.root_accession): (normalize_accession(r.latest_accession) or str(r.root_accession))
              for r in root_table.itertuples()}
authorized_access = collect_authorized_access(aa_targets)
for field in ["platform_biodata_catalyst", "platform_anvil", "platform_crdc", "any_consent_group_sra_available"]:
    if field in authorized_access:
        value = authorized_access[field].astype("boolean")
        print(f"  {field}: {int(value.fillna(False).sum()):,} positive / "
              f"{int(value.notna().sum()):,} non-null recorded labels")


## 10. iCite records
Uses the same PMID universe without requiring a prior join. Retains the supplied schema version and score/clinical-field parsing.


In [ ]:
# @title 10. iCite records
ICITE_SCHEMA_VERSION = "2026-10-01-v3-current-api"
BATCH = 200
SLEEP = 0.34
def bool_or_na(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return pd.NA
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    return True if s in {'true', 'yes', '1', 'y'} else False if s in {'false', 'no', '0', 'n'} else pd.NA

def clinical_count(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return 0
    if isinstance(v, (list, tuple, set)):
        return sum((bool(str(x).strip()) for x in v))
    return len([x for x in re.split('[,\\s]+', str(v).strip()) if x])

def parse_record(x):
    return {'pmid': str(x.get('pmid')), 'icite_year': x.get('year'), 'icite_title': x.get('title'),
        'icite_journal': x.get('journal'), 'doi': x.get('doi'), 'rcr': x.get('relative_citation_ratio'),
        'nih_percentile': x.get('nih_percentile'), 'citation_count': x.get('citation_count'),
        'citations_per_year': x.get('citations_per_year'),
        'expected_citations_per_year': x.get('expected_citations_per_year'),
        'field_citation_rate': x.get('field_citation_rate'), 'apt': x.get('apt'),
        'is_research_article': bool_or_na(x.get('is_research_article')),
        'is_clinical': bool_or_na(x.get('is_clinical')), 'provisional': bool_or_na(x.get('provisional')),
        'n_clinical_citations': clinical_count(x.get('cited_by_clin')), 'icite_fetch_note': 'ok',
        'icite_schema_version': ICITE_SCHEMA_VERSION, 'icite_fetched_utc': datetime.now(timezone.utc).isoformat()}

def fetch_batch(pmids, retries=4):
    pmids = [str(p).strip() for p in pmids if str(p).strip().isdigit()]
    if not pmids:
        return (pd.DataFrame(), 'empty')
    delay = 1
    for _ in range(retries):
        try:
            r = requests.get(ICITE_API, params={'pmids': ','.join(pmids)}, headers=HEADERS, timeout=TIMEOUT)
        except requests.RequestException:
            r = None
        if r is None or r.status_code == 429 or r.status_code >= 500:
            time.sleep(delay)
            delay = min(delay * 2, 20)
            continue
        if r.status_code != 200:
            return (None, f'http_{r.status_code}')
        try:
            data = r.json().get('data', []) or []
        except ValueError:
            return (None, 'non_json')
        rows = [parse_record(x) for x in data if x.get('pmid') is not None]
        returned = {x['pmid'] for x in rows}
        now = datetime.now(timezone.utc).isoformat()
        rows += [{'pmid': p, 'icite_fetch_note': 'not_returned', 'icite_schema_version': ICITE_SCHEMA_VERSION,
                'icite_fetched_utc': now} for p in pmids if p not in returned]
        return (pd.DataFrame(rows), 'ok')
    return (None, 'exceeded_retries')

def fetch_icite_group(targets):
    frame, note = fetch_batch(list(targets))
    if frame is None:
        return {p:(pd.DataFrame(columns=["pmid"]), note, False) for p in targets}
    result = {}
    for key in targets:
        part = frame.loc[frame.pmid.astype(str).eq(key)].copy()
        row_note = str(part.icite_fetch_note.iloc[-1]) if len(part) else "missing_batch_result"
        result[key] = (part, row_note, row_note in {"ok", "not_returned"})
    return result

def known_icite(old):
    if old.empty or "icite_schema_version" not in old:
        return {}
    return known_from_rows(old.loc[old.icite_schema_version.eq(ICITE_SCHEMA_VERSION)], ["pmid"], "icite_fetch_note",
        ("ok", "not_returned"))

icite = collect_keyed("icite", "icite_metrics", "icite_checkpoint", "done_pmids",
    {p:p for p in corpus_pmids()}, ["pmid"], fetch_icite_group, known_icite,
    empty_columns=list(parse_record({}).keys()), batch_size=BATCH, pause=SLEEP,
    checkpoint_extra={"schema_version":ICITE_SCHEMA_VERSION})
if "rcr" in icite:
    values = pd.to_numeric(icite.rcr, errors="coerce")
    print(f"RCR defined: {values.notna().sum():,}/{len(values):,}; median: {values.median():.3f}; negatives: {values.lt(0).sum():,}")


## 11. RePORTER publication-to-project links
Every page in a PMID batch must succeed before checkpoint advancement. A successful no-link response is recorded as a completed lookup, not proof of no NIH support.


In [ ]:
# @title 11. RePORTER publication-to-project links
def _reporter_search_all(url, payload, endpoint_kind, max_retries=5):
    """Fetch a complete batch; raise on failure before its checkpoint advances.

    Pagination/rate limits: https://api.reporter.nih.gov/
    No rows are persisted until all pages of this batch have been retrieved.
    Existing endpoint versions, search criteria and output row fields are retained.
    """
    offset = 0
    limit = int(payload.get("limit", 500))
    offset_cap = 9999 if endpoint_kind == "publications" else 14999
    all_results = []
    seen_pages = set()
    initial_total = None
    while True:
        if offset > offset_cap:
            raise RuntimeError("RePORTER offset limit reached. Reduce the batch chunk size and rerun; this batch was not checkpointed.")
        body = dict(payload, offset=offset, limit=limit)
        response_json = None
        last_error = "request failed"
        delay = 1.0
        for attempt in range(max_retries):
            # Applies to each page and retry, not only once per batch.
            time.sleep(max(float(REPORTER_SLEEP), 1.0))
            try:
                response = requests.post(url, json=body, timeout=TIMEOUT,
                                         headers={"Content-Type": "application/json"})
            except requests.RequestException as exc:
                last_error = f"{type(exc).__name__}: {exc}"
            else:
                if response.status_code == 200:
                    try:
                        candidate = response.json()
                        if not isinstance(candidate, dict) or not isinstance(candidate.get("results"), list):
                            raise ValueError("expected a JSON object containing a results list")
                        if any(not isinstance(item, dict) for item in candidate["results"]):
                            raise ValueError("results entries must be objects")
                        response_json = candidate
                        break
                    except ValueError as exc:
                        last_error = f"invalid JSON response: {exc}"
                elif response.status_code != 429 and response.status_code < 500:
                    raise RuntimeError(f"RePORTER HTTP {response.status_code}; batch not checkpointed. Check endpoint/configuration before retrying.")
                else:
                    last_error = f"HTTP {response.status_code}"
            if attempt < max_retries - 1:
                time.sleep(delay)
                delay = min(delay * 2, 20)
        if response_json is None:
            raise RuntimeError(f"RePORTER retries exhausted ({last_error}); batch not checkpointed.")
        page = response_json["results"]
        meta = response_json.get("meta") or {}
        total_raw = meta.get("total") if isinstance(meta, dict) else None
        total = int(total_raw) if total_raw is not None else None
        if total is not None:
            if total < 0:
                raise RuntimeError("RePORTER returned a negative total; batch not checkpointed.")
            if initial_total is None:
                initial_total = total
            elif total != initial_total:
                raise RuntimeError("RePORTER total changed while paging; batch not checkpointed. Retry this batch.")
        signature = json.dumps(page, sort_keys=True)
        if page and signature in seen_pages:
            raise RuntimeError("RePORTER repeated a page; batch not checkpointed.")
        seen_pages.add(signature)
        all_results.extend(page)
        if total is not None and len(all_results) >= total:
            if len(all_results) != total:
                raise RuntimeError("RePORTER returned more rows than its total; batch not checkpointed.")
            return all_results
        if not page:
            if total is not None and len(all_results) < total:
                raise RuntimeError("RePORTER ended before its stated total; batch not checkpointed.")
            return all_results
        if total is None and len(page) < limit:
            return all_results
        offset += len(page)

def to_core(num):
    if not num:
        return None
    m = FULL_GRANT_RE.search(str(num).upper().replace(" ", ""))
    return m.group(1) if m else str(num).upper().strip()

def classify_core(core):
    c = (core or "").upper()
    if not c:
        return "unknown"
    if c.startswith("Z"):
        return "intramural"
    if c.startswith(CONTRACT_PREFIXES):
        return "contract"
    return "grant"

def current_period_funding(projects):
    if projects.empty:
        return pd.DataFrame(columns=["core_project_num", "current_period_award",
                                     "current_period_years", "latest_fy", "admin_ic", "activity_code"])
    p = projects.dropna(subset=["core_project_num"]).copy()
    # One award_amount per (core, FY): competing supplements in the same FY -> sum.
    fy = (p.groupby(["core_project_num", "fiscal_year"], as_index=False)
            .agg(award_amount=("award_amount", "sum")))
    out = []
    for core, g in fy.groupby("core_project_num"):
        g = g.dropna(subset=["fiscal_year"]).sort_values("fiscal_year")
        if g.empty:
            out.append({"core_project_num": core, "current_period_award": np.nan,
                        "current_period_years": 0, "latest_fy": np.nan}); continue
        years = g["fiscal_year"].astype(int).tolist()
        amts = g.set_index(g["fiscal_year"].astype(int))["award_amount"]
        # Walk back from the latest FY while years stay consecutive.
        latest = years[-1]; period = [latest]; y = latest
        yset = set(years)
        while (y - 1) in yset:
            y -= 1; period.append(y)
        period = sorted(period)
        award = amts.loc[period].sum(min_count=1)
        out.append({"core_project_num": core,
                    "current_period_award": award,
                    "current_period_years": len(period),
                    "latest_fy": latest})
    cp = pd.DataFrame(out)
    meta = (projects.sort_values("fiscal_year")
            .dropna(subset=["core_project_num"])
            .drop_duplicates("core_project_num", keep="last")[["core_project_num", "admin_ic", "activity_code"]])
    return cp.merge(meta, on="core_project_num", how="left")

assert CONTRACT_PREFIXES == ("N01", "HHSN")
assert [classify_core(v) for v in ["R01NS047347", "N01AA000001", "HHSN000000000001", "ZIAAA000001",
        None]] == ["grant", "contract", "contract", "intramural", "unknown"]
assert to_core("5R01NS047347-05") == "R01NS047347"

def known_reporter(old, checkpoint_key, field, cols):
    # A successfully probed identifier with no returned link is a completed negative lookup.
    # A missing artifact invalidates positive-only old checkpoints: do not claim lost rows complete.
    if not os.path.exists(PATHS["reporter_pub_links_full" if field == "done_pmids" else "reporter_projects_full"]):
        return {}
    return checkpoint_map(checkpoint_key, field)

def fetch_reporter_pub_group(targets):
    payload = {"criteria":{"pmids":[int(p) for p in targets]}, "limit":500, "offset":0}
    records = _reporter_search_all(REPORTER_PUB_URL, payload, "publications")
    rows = [{"pmid":str(rec.get("pmid")), "core_project_num":to_core(rec.get("coreproject") or rec.get("core_project_num")),
             "appl_id":rec.get("applid") or rec.get("appl_id")} for rec in records]
    frame = pd.DataFrame(rows, columns=["pmid", "core_project_num", "appl_id"])
    if not frame.empty and not frame.pmid.isin(targets).all():
        raise ValueError("RePORTER returned an unrequested PMID")
    return {p:(frame.loc[frame.pmid.eq(p)].copy(), "ok" if frame.pmid.eq(p).any() else "no_links",
            True) for p in targets}

if ENABLE_REPORTER:
    reporter_pub_links = collect_keyed("reporter_publications", "reporter_pub_links_full", "reporter_pub_checkpoint", "done_pmids",
        {p:p for p in corpus_pmids()}, ["pmid"], fetch_reporter_pub_group,
        lambda old: known_reporter(old, "reporter_pub_checkpoint", "done_pmids", ["pmid"]),
        empty_columns=["pmid", "core_project_num", "appl_id"], batch_size=250, pause=REPORTER_SLEEP)
    print(f"RePORTER: {reporter_pub_links.pmid.nunique():,}/{len(corpus_pmids()):,} corpus PMIDs linked; {len(reporter_pub_links):,} raw relationship rows.")
else:
    reporter_pub_links = read_frame("reporter_pub_links_full", ["pmid", "core_project_num", "appl_id"])
    print("RePORTER acquisition disabled; saved inputs retained.")


## 12. RePORTER project records
Requests all available fiscal-year rows for linked cores, with the original subproject exclusion. Refreshes replace a core response as a group rather than appending its old rows again.


In [ ]:
# @title 12. RePORTER project records
def fetch_reporter_project_group(targets):
    payload = {"criteria":{"project_nums":list(targets)},
               "include_fields":["CoreProjectNum", "FiscalYear", "AwardAmount", "AgencyIcAdmin", "ActivityCode",
                   "SubprojectId"],
               "limit":500, "offset":0}
    records = _reporter_search_all(REPORTER_PRJ_URL, payload, "projects")
    rows = []
    for rec in records:
        if rec.get("subproject_id") not in (None, "", 0, "0"):
            continue
        ic = rec.get("agency_ic_admin") if isinstance(rec.get("agency_ic_admin"), dict) else {}
        rows.append({"core_project_num":to_core(rec.get("core_project_num") or rec.get("core_project_number")),
                     "fiscal_year":pd.to_numeric(rec.get("fiscal_year"), errors="coerce"),
                     "award_amount":pd.to_numeric(rec.get("award_amount"), errors="coerce"),
                     "activity_code":rec.get("activity_code"), "admin_ic":ic.get("abbreviation") or ic.get("name")})
    frame = pd.DataFrame(rows, columns=["core_project_num", "fiscal_year", "award_amount", "activity_code",
            "admin_ic"])
    if not frame.empty and not frame.core_project_num.isin(targets).all():
        raise ValueError("RePORTER returned an unrequested core")
    return {p:(frame.loc[frame.core_project_num.eq(p)].copy(),
            "ok" if frame.core_project_num.eq(p).any() else "no_projects", True) for p in targets}

if ENABLE_REPORTER:
    cores = sorted(reporter_pub_links.core_project_num.dropna().apply(to_core).unique())
    reporter_projects = collect_keyed("reporter_projects", "reporter_projects_full", "reporter_proj_checkpoint", "done_cores",
        {c:c for c in cores}, ["core_project_num"], fetch_reporter_project_group,
        lambda old: known_reporter(old, "reporter_proj_checkpoint", "done_cores", ["core_project_num"]),
        empty_columns=["core_project_num", "fiscal_year", "award_amount", "activity_code",
            "admin_ic"], batch_size=50, pause=REPORTER_SLEEP)
else:
    reporter_projects = read_frame("reporter_projects_full", ["core_project_num", "fiscal_year", "award_amount",
            "activity_code", "admin_ic"])


## 13a. FTP timing collector definitions
The supplied algorithm selects the earliest eligible timestamp in the first numerically ordered version containing a preferred file, with later-version fallback. This is not a global scan of every version and not a release/minting date. Schemas and filename rules are retained.


In [ ]:
# @title 13a. FTP timing collector definitions
# FTP means current surviving public artifact, never accession minting/public release.
FTP_HOST = "ftp.ncbi.nlm.nih.gov"
FTP_STUDIES_DIR = "/dbgap/studies"
REQUEST_PAUSE_SECONDS = 0.15
CHECKPOINT_EVERY = 25
FTP_TIMEOUT_SECONDS = 90
MAX_ROOT_RETRIES = 2
RETRY_PAUSE_SECONDS = 2.0
MIN_VALID_YEAR, MAX_VALID_YEAR = 1990, 2035
MIN_FTP_COLLECTION_COVERAGE = 0.90
MIN_PREFERRED_ARTIFACT_COVERAGE = 0.80
PLOT_AUDIT_FTP_TIMELINES = False
FTP_LISTING_COLUMNS = ["parent_path", "name", "path", "object_type", "modify_raw", "size", "modify_datetime"]
SENTINEL_ROOTS = [
    "phs000001",
    "phs000006",
    "phs000007",
    "phs000016",
    "phs000049",
    "phs000050",
    "phs000100",
    "phs000327",
    "phs000420",
    "phs000974",
    "phs001000",
    "phs001412",
    "phs001600",
    "phs001603",
    "phs001635",
    "phs001644",
    "phs002000",
    "phs002685",
    "phs003000",
    "phs003360",
    "phs004768",
    "phs004891",
]

TIMING_ROOT = DRIVE_ROOT if FTP_RUN_MODE == "full" else os.path.join(DRIVE_ROOT, "accession_timing_audit")
os.makedirs(TIMING_ROOT, exist_ok=True)
TIMING_FIGURE_DIR = FIGURE_DIR if FTP_RUN_MODE == "full" else os.path.join(TIMING_ROOT, "figures")
os.makedirs(TIMING_FIGURE_DIR, exist_ok=True)
# Optional independently supplied family-deduplicated rows; no invented family map.
DEDUP_ACCESS_PATH = os.path.join(DRIVE_ROOT, "dedup_access.parquet")
# Change this to your saved Notebook B artifact when named differently.
artifact_paths = {
    "ftp_root_results": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_root_results.parquet",
    ),
    "ftp_shallow_objects": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_shallow_objects.parquet",
    ),
    "ftp_dar_validation": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_dar_validation.parquet",
    ),
    "ftp_validation_summary": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_validation_summary.parquet",
    ),
    "ftp_collection_status_summary": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_collection_status_summary.parquet",
    ),
    "ftp_access_audit_summary": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_access_audit_summary.parquet",
    ),
    "ftp_public_directory_absences": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_public_directory_absences.parquet",
    ),
    "ftp_true_collector_failures": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_true_collector_failures.parquet",
    ),
    "ftp_no_preferred_artifact_with_directory": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_no_preferred_artifact_with_directory.parquet",
    ),
    "corrected_coverage_summary": os.path.join(
        TIMING_ROOT,
        "accession_timing_corrected_coverage_summary.parquet",
    ),
    "manual_review_cases": os.path.join(
        TIMING_ROOT,
        "accession_timing_ftp_manual_review_cases.parquet",
    ),
    "annual_dar_timeline": os.path.join(
        TIMING_ROOT,
        "accession_timing_annual_dar_timeline.parquet",
    ),
    "first_observed_dar": os.path.join(
        TIMING_ROOT,
        "accession_timing_first_observed_dar_by_root.parquet",
    ),
    "deck_readiness": os.path.join(
        TIMING_ROOT,
        "accession_timing_deck_readiness.parquet",
    ),
    "next_steps": os.path.join(
        TIMING_ROOT,
        "accession_timing_next_steps.parquet",
    ),
    "matched_ftp_dar_timeline": os.path.join(
        TIMING_ROOT,
        "accession_timing_matched_ftp_dar_timeline.parquet",
    ),
}
PATHS.update({"timing_"+k:v for k,v in artifact_paths.items()})
PATHS["ftp_root_results"] = artifact_paths["ftp_root_results"]
PATHS["ftp_shallow_objects"] = artifact_paths["ftp_shallow_objects"]
PATHS["ftp_checkpoint_metadata"] = os.path.join(TIMING_ROOT, "accession_timing_ftp_checkpoint_metadata.parquet")
PATHS["ftp_check_state"] = os.path.join(TIMING_ROOT, "accession_timing_ftp_checked_roots.json")

def bool_condition(series):
    """
    Convert pandas nullable booleans to standard bool ndarray for np.select.
    Missing values are interpreted as False.
    """
    return pd.Series(series).fillna(False).astype(bool).to_numpy()

def empty_ftp_listing():
    """
    Produce a stable-schema empty FTP listing.

    This is crucial: a successfully listed but empty FTP directory must still
    carry all expected columns so later references to object_type cannot fail.
    """
    return pd.DataFrame(columns=FTP_LISTING_COLUMNS)

def parse_ftp_modify_timestamp(value):
    """
    Parse RFC 3659 MLSD modify timestamp as timezone-aware UTC.

    Expected source format: YYYYMMDDHHMMSS
    """
    if value is None or pd.isna(value):
        return pd.NaT
    try:
        return pd.Timestamp(datetime.strptime(str(value), '%Y%m%d%H%M%S').replace(tzinfo=timezone.utc))
    except (TypeError, ValueError):
        return pd.NaT

def ftp_connect():
    """
    Start anonymous NCBI public FTP session.
    """
    ftp = FTP(FTP_HOST, timeout=FTP_TIMEOUT_SECONDS)
    ftp.login()
    return ftp

def ftp_list_mlsd(ftp, path):
    """
    List one public FTP directory using MLSD.

    Returns:
        Stable-schema DataFrame and error text (or None).

    A successfully listed empty directory returns:
        empty_ftp_listing(), None

    An inaccessible path or transport error returns:
        empty_ftp_listing(), error_text
    """
    records = []
    try:
        for name, facts in ftp.mlsd(path):
            if name in {'.', '..'}:
                continue
            records.append({'parent_path': path, 'name': name, 'path': f"{path.rstrip('/')}/{name}",
                    'object_type': facts.get('type', pd.NA), 'modify_raw': facts.get('modify', pd.NA),
                    'size': facts.get('size', pd.NA)})
    except all_errors as exc:
        return (empty_ftp_listing(), str(exc))
    if not records:
        return (empty_ftp_listing(), None)
    listing = pd.DataFrame(records)
    listing['modify_datetime'] = listing['modify_raw'].apply(parse_ftp_modify_timestamp)
    listing = listing.reindex(columns=FTP_LISTING_COLUMNS)
    return (listing, None)

def is_ftp_550_missing_directory(error_text):
    """
    Return True for FTP 550 responses that indicate absent current path.

    This is a current-public-coverage absence, not a retryable technical
    collector failure.
    """
    if error_text is None or pd.isna(error_text):
        return False
    normalized = str(error_text).lower()
    return '550' in normalized and any((phrase in normalized for phrase in ['no such file', 'not found',
                'cannot find', 'file unavailable']))

def version_sort_key(folder_name, root_accession):
    """
    Numeric version key for folders such as:
    phs000001.v1.p1
    phs000001.v10.p5
    """
    match = re.match(f'^{re.escape(root_accession)}\\.v(\\d+)\\.p(\\d+)$', str(folder_name), flags=re.IGNORECASE)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return (999999, 999999)

def classify_ftp_object(name, path):
    """
    Classify a file and decide its timing-proxy eligibility.

    Preferred:
    - Study-specific file.
    - Not generic template/asset.
    - Not obvious administrative/interchange artifact.

    Sensitivity:
    - Not generic template/asset.
    - Administrative/interchange objects allowed.
    """
    filename = str(name).lower()
    full_path = str(path).lower()
    is_generic_template = bool(re.search('(?:varreports?(_v\\d+)?\\.xsl$|datadict(_v\\d+)?\\.xsl$|\\.xsl$|\\.xslt$|\\.css$|\\.js$|index\\.html?$|readme(\\.txt)?$|\\.ds_store$)',
            filename, flags=re.IGNORECASE))
    is_interchange = bool(re.search('gapexchange|exchange.*\\.xml$', filename, flags=re.IGNORECASE))
    if is_interchange:
        artifact_category = 'administrative_or_interchange_metadata'
    elif is_generic_template:
        artifact_category = 'generic_template_or_asset'
    elif re.search('data[_\\.-]?dict|datadict|var[_\\.-]?report|variable[_\\.-]?report|\\.datadict\\.xml$',
        filename, flags=re.IGNORECASE):
        artifact_category = 'metadata_or_dictionary'
    elif re.search('release[_\\.-]?notes?|consent|protocol|phd\\d+.*\\.xml$|\\.pdf$|\\.docx?$', filename,
        flags=re.IGNORECASE):
        artifact_category = 'documents_or_release_material'
    elif re.search('geno|genotype|snp|vcf|bcf|bam|cram|cel|idat|plink|\\.bed$|\\.bim$|\\.fam$|sample[_\\.-]?info|sample[_\\.-]?annotation|qc[_\\.-]?|call[_\\.-]?threshold|array[_\\.-]?annotation',
        filename, flags=re.IGNORECASE):
        artifact_category = 'genotype_or_technical_sample_metadata'
    elif re.search('phenotype|clinical|subject[_\\.-]?data|participant[_\\.-]?data|case[_\\.-]?control|baseline|follow[_\\.-]?up|pht\\d+',
        filename, flags=re.IGNORECASE):
        artifact_category = 'phenotype_or_clinical_metadata'
    else:
        artifact_category = 'other_study_specific_file'
    return {'artifact_category': artifact_category, 'is_generic_template_artifact': is_generic_template,
        'is_administrative_interchange': is_interchange, 'is_substantive_study_artifact': not is_generic_template,
        'is_preferred_timeline_artifact': not is_generic_template and (not is_interchange)}

def select_earliest_candidate(objects, preferred_only=True):
    """
    Select earliest eligible timestamped file from a version scan.
    """
    if objects.empty:
        return pd.Series(dtype='object')
    candidates = objects.copy().loc[lambda frame: frame['object_type'].astype('string').str.lower().eq('file')].dropna(subset=['modify_datetime'])
    if preferred_only:
        candidates = candidates.loc[candidates['is_preferred_timeline_artifact']]
    else:
        candidates = candidates.loc[candidates['is_substantive_study_artifact']]
    if candidates.empty:
        return pd.Series(dtype='object')
    return candidates.sort_values(['modify_datetime', 'path']).iloc[0]

def empty_classified_ftp_objects():
    """
    Stable-schema empty object table for valid but empty version folders.
    """
    empty_objects = empty_ftp_listing().copy()
    empty_objects['root_accession'] = pd.Series(dtype='string')
    empty_objects['version_folder'] = pd.Series(dtype='string')
    empty_objects['artifact_category'] = pd.Series(dtype='string')
    empty_objects['is_generic_template_artifact'] = pd.Series(dtype='boolean')
    empty_objects['is_administrative_interchange'] = pd.Series(dtype='boolean')
    empty_objects['is_substantive_study_artifact'] = pd.Series(dtype='boolean')
    empty_objects['is_preferred_timeline_artifact'] = pd.Series(dtype='boolean')
    return empty_objects

def scan_one_version_folder(ftp, root_accession, root_path, version_folder):
    """
    Shallow-scan one version folder and immediate child directories.

    Empty successful listings are valid and return stable empty data rather
    than raising a KeyError. Child-directory errors are returned as warnings
    while preserving usable parent-level objects.
    """
    version_path = f'{root_path}/{version_folder}'
    version_listing, version_error = ftp_list_mlsd(ftp, version_path)
    if version_error is not None:
        return (empty_classified_ftp_objects(), version_error)
    if version_listing.empty:
        return (empty_classified_ftp_objects(), None)
    listing_tables = [version_listing]
    child_listing_errors = []
    immediate_child_dirs = version_listing.loc[version_listing['object_type'].astype('string').str.lower().eq('dir'),
        'path'].tolist()
    for child_path in immediate_child_dirs:
        child_listing, child_error = ftp_list_mlsd(ftp, child_path)
        if child_error is not None:
            child_listing_errors.append(f'{child_path}: {child_error}')
            continue
        if not child_listing.empty:
            listing_tables.append(child_listing)
        time.sleep(REQUEST_PAUSE_SECONDS)
    objects = pd.concat(listing_tables, ignore_index=True)
    if objects.empty:
        return (empty_classified_ftp_objects(), ' | '.join(child_listing_errors) if child_listing_errors else None)
    objects['root_accession'] = root_accession
    objects['version_folder'] = version_folder
    classifications = objects.apply(lambda row: classify_ftp_object(row['name'], row['path']), axis=1,
        result_type='expand')
    objects = pd.concat([objects, classifications], axis=1)
    warning_text = ' | '.join(child_listing_errors) if child_listing_errors else None
    return (objects, warning_text)

def collect_root_ftp_proxy(ftp, root_accession):
    """
    Collect public FTP timing metadata for a root accession.

    Root outcomes:
    - preferred_artifact_found
    - sensitivity_only_artifact_found
    - no_current_public_ftp_root_directory
    - no_version_folder
    - no_dated_substantive_artifact
    - root_listing_error
    - version_scan_error_no_artifact
    """
    root_path = f'{FTP_STUDIES_DIR}/{root_accession}'
    result = {'root_accession': root_accession, 'ftp_root_path': root_path, 'collection_status': 'unknown',
        'collection_error': pd.NA, 'has_current_public_ftp_root_directory': pd.NA, 'requires_ftp_retry': False,
        'n_version_folders': 0, 'version_folders_seen': [], 'selected_version_folder': pd.NA,
        'used_later_version_fallback': False, 'n_shallow_objects': 0, 'n_files_with_dates': 0,
        'first_preferred_public_versioned_ftp_timestamp': pd.NaT,
        'first_preferred_public_versioned_ftp_object_name': pd.NA,
        'first_preferred_public_versioned_ftp_object_path': pd.NA,
        'first_preferred_public_versioned_ftp_category': pd.NA,
        'first_any_substantive_public_versioned_ftp_timestamp': pd.NaT,
        'first_any_substantive_public_versioned_ftp_object_name': pd.NA,
        'first_any_substantive_public_versioned_ftp_object_path': pd.NA,
        'first_any_substantive_public_versioned_ftp_category': pd.NA}
    root_listing, root_error = ftp_list_mlsd(ftp, root_path)
    if root_error is not None:
        if is_ftp_550_missing_directory(root_error):
            result['collection_status'] = 'no_current_public_ftp_root_directory'
            result['collection_error'] = 'Current public FTP root directory not found (FTP 550).'
            result['has_current_public_ftp_root_directory'] = False
            result['requires_ftp_retry'] = False
        else:
            result['collection_status'] = 'root_listing_error'
            result['collection_error'] = root_error
            result['has_current_public_ftp_root_directory'] = pd.NA
            result['requires_ftp_retry'] = True
        return (result, empty_classified_ftp_objects())
    result['has_current_public_ftp_root_directory'] = True
    version_folders = root_listing.loc[root_listing['object_type'].astype('string').str.lower().eq('dir') & root_listing['name'].astype('string').str.match(f'^{re.escape(root_accession)}\\.v\\d+\\.p\\d+$',
            case=False, na=False), 'name'].tolist()
    version_folders = sorted(version_folders, key=lambda value: version_sort_key(value, root_accession))
    result['n_version_folders'] = len(version_folders)
    result['version_folders_seen'] = version_folders
    if not version_folders:
        result['collection_status'] = 'no_version_folder'
        return (result, empty_classified_ftp_objects())
    scanned_object_tables = []
    version_scan_errors = []
    selected_version_folder = None
    selected_preferred_candidate = pd.Series(dtype='object')
    selected_sensitivity_candidate = pd.Series(dtype='object')
    for version_index, version_folder in enumerate(version_folders):
        objects, version_warning = scan_one_version_folder(ftp=ftp, root_accession=root_accession,
            root_path=root_path, version_folder=version_folder)
        if version_warning is not None:
            version_scan_errors.append(f'{version_folder}: {version_warning}')
        if objects.empty:
            continue
        scanned_object_tables.append(objects)
        preferred_candidate = select_earliest_candidate(objects, preferred_only=True)
        sensitivity_candidate = select_earliest_candidate(objects, preferred_only=False)
        if not preferred_candidate.empty:
            selected_version_folder = version_folder
            selected_preferred_candidate = preferred_candidate
            selected_sensitivity_candidate = sensitivity_candidate
            result['used_later_version_fallback'] = version_index > 0
            break
        if selected_sensitivity_candidate.empty and (not sensitivity_candidate.empty):
            selected_version_folder = version_folder
            selected_sensitivity_candidate = sensitivity_candidate
            result['used_later_version_fallback'] = version_index > 0
        time.sleep(REQUEST_PAUSE_SECONDS)
    all_scanned_objects = pd.concat(scanned_object_tables,
        ignore_index=True) if scanned_object_tables else empty_classified_ftp_objects()
    result['n_shallow_objects'] = len(all_scanned_objects)
    if not all_scanned_objects.empty:
        result['n_files_with_dates'] = int((all_scanned_objects['object_type'].astype('string').str.lower().eq('file') & all_scanned_objects['modify_datetime'].notna()).sum())
    result['selected_version_folder'] = selected_version_folder if selected_version_folder is not None else pd.NA
    if version_scan_errors:
        result['collection_error'] = ' | '.join(version_scan_errors)
        result['requires_ftp_retry'] = True
    if not selected_preferred_candidate.empty:
        result['first_preferred_public_versioned_ftp_timestamp'] = selected_preferred_candidate['modify_datetime']
        result['first_preferred_public_versioned_ftp_object_name'] = selected_preferred_candidate['name']
        result['first_preferred_public_versioned_ftp_object_path'] = selected_preferred_candidate['path']
        result['first_preferred_public_versioned_ftp_category'] = selected_preferred_candidate['artifact_category']
    if not selected_sensitivity_candidate.empty:
        result['first_any_substantive_public_versioned_ftp_timestamp'] = selected_sensitivity_candidate['modify_datetime']
        result['first_any_substantive_public_versioned_ftp_object_name'] = selected_sensitivity_candidate['name']
        result['first_any_substantive_public_versioned_ftp_object_path'] = selected_sensitivity_candidate['path']
        result['first_any_substantive_public_versioned_ftp_category'] = selected_sensitivity_candidate['artifact_category']
    if not selected_preferred_candidate.empty:
        result['collection_status'] = 'preferred_artifact_found'
    elif not selected_sensitivity_candidate.empty:
        result['collection_status'] = 'sensitivity_only_artifact_found'
    elif version_scan_errors:
        result['collection_status'] = 'version_scan_error_no_artifact'
        result['collection_error'] = ' | '.join(version_scan_errors)
        result['requires_ftp_retry'] = True
    else:
        result['collection_status'] = 'no_dated_substantive_artifact'
    return (result, all_scanned_objects)

def normalize_ftp_results(frame):
    ftp_root_results = frame.copy()
    ftp_root_results["root_accession"] = ftp_root_results["root_accession"].astype("string").str.strip().str.lower()
    previous_retry = ftp_root_results.get('requires_ftp_retry', pd.Series(False,
            index=ftp_root_results.index)).astype('boolean').fillna(False)
    text_columns = ['collection_status', 'collection_error', 'selected_version_folder',
        'first_preferred_public_versioned_ftp_category', 'first_preferred_public_versioned_ftp_object_name',
        'first_preferred_public_versioned_ftp_object_path', 'first_any_substantive_public_versioned_ftp_category',
        'first_any_substantive_public_versioned_ftp_object_name',
        'first_any_substantive_public_versioned_ftp_object_path']
    for column in text_columns:
        if column not in ftp_root_results.columns:
            ftp_root_results[column] = pd.NA
    integer_columns = ['n_version_folders', 'n_shallow_objects', 'n_files_with_dates', 'collection_attempts']
    for column in integer_columns:
        if column not in ftp_root_results.columns:
            ftp_root_results[column] = 0
    boolean_columns = ['used_later_version_fallback', 'requires_ftp_retry']
    for column in boolean_columns:
        if column not in ftp_root_results.columns:
            ftp_root_results[column] = False
    if 'has_current_public_ftp_root_directory' not in ftp_root_results.columns:
        ftp_root_results['has_current_public_ftp_root_directory'] = pd.NA
    timestamp_columns = ['first_preferred_public_versioned_ftp_timestamp',
        'first_any_substantive_public_versioned_ftp_timestamp']
    for column in timestamp_columns:
        if column not in ftp_root_results.columns:
            ftp_root_results[column] = pd.NaT
        ftp_root_results[column] = pd.to_datetime(ftp_root_results[column], errors='coerce', utc=True)
    ftp_root_results['collection_status'] = ftp_root_results['collection_status'].astype('string')
    ftp_root_results['collection_error'] = ftp_root_results['collection_error'].astype('string')
    legacy_ftp_550_mask = ftp_root_results['collection_status'].eq('root_listing_error') & ftp_root_results['collection_error'].apply(is_ftp_550_missing_directory)
    ftp_root_results.loc[legacy_ftp_550_mask, 'collection_status'] = 'no_current_public_ftp_root_directory'
    ftp_root_results.loc[legacy_ftp_550_mask,
        'collection_error'] = 'Current public FTP root directory not found (FTP 550).'
    ftp_root_results['has_current_public_ftp_root_directory'] = np.where(ftp_root_results['collection_status'].eq('no_current_public_ftp_root_directory'),
        False, ftp_root_results['has_current_public_ftp_root_directory'])
    ftp_root_results['requires_ftp_retry'] = ftp_root_results['collection_status'].astype('string').str.contains('listing_error|scan_error|timeout|connection|temporary',
        case=False, regex=True,
        na=False) & ~ftp_root_results['collection_status'].eq('no_current_public_ftp_root_directory')
    ftp_root_results['first_preferred_public_versioned_ftp_year'] = ftp_root_results['first_preferred_public_versioned_ftp_timestamp'].dt.year
    ftp_root_results['first_any_substantive_public_versioned_ftp_year'] = ftp_root_results['first_any_substantive_public_versioned_ftp_timestamp'].dt.year
    ftp_root_results['phs_numeric_id'] = ftp_root_results['root_accession'].str.extract('phs(\\d+)',
        expand=False).astype('Int64')
    ftp_root_results['primary_missing_sensitivity_present'] = ftp_root_results['first_preferred_public_versioned_ftp_timestamp'].isna() & ftp_root_results['first_any_substantive_public_versioned_ftp_timestamp'].notna()
    ftp_root_results['manual_review_low_phs_late_ftp'] = ftp_root_results['phs_numeric_id'].lt(1000) & ftp_root_results['first_preferred_public_versioned_ftp_year'].ge(2020)
    ftp_root_results['requires_ftp_retry'] = ftp_root_results['requires_ftp_retry'] | previous_retry & ~legacy_ftp_550_mask.fillna(False)
    return ftp_root_results


## 13b. FTP timing acquisition
The collector needs only the root list. It opens no FTP connection on a completed cached run. Missing directories are distinct from transport/partial-scan errors. Audit mode writes to its own directory.


In [ ]:
# @title 13b. FTP timing acquisition
def empty_ftp_roots():
    columns = ["root_accession", "ftp_root_path", "collection_status", "collection_error",
               "has_current_public_ftp_root_directory", "requires_ftp_retry", "n_version_folders",
               "version_folders_seen", "selected_version_folder", "used_later_version_fallback",
               "n_shallow_objects", "n_files_with_dates",
               "first_preferred_public_versioned_ftp_timestamp", "first_preferred_public_versioned_ftp_object_name",
               "first_preferred_public_versioned_ftp_object_path", "first_preferred_public_versioned_ftp_category",
               "first_any_substantive_public_versioned_ftp_timestamp", "first_any_substantive_public_versioned_ftp_object_name",
               "first_any_substantive_public_versioned_ftp_object_path", "first_any_substantive_public_versioned_ftp_category"]
    frame = pd.DataFrame(columns=columns)
    frame["root_accession"] = frame.root_accession.astype("string")
    return normalize_ftp_results(frame)


def run_ftp_collection():
    block = "ftp" if FTP_RUN_MODE == "full" else "ftp_audit"
    prepare_source(block, ["ftp_root_results", "ftp_shallow_objects", "ftp_checkpoint_metadata", "ftp_check_state"])
    roots = sorted(root_table.root_accession.dropna().astype(str))
    if FTP_RUN_MODE == "audit":
        rng = np.random.default_rng(20261005)
        roots = sorted(set(rng.choice(roots, size=min(FTP_AUDIT_RANDOM_N,len(roots)), replace=False))
                       | set(roots[:10]+roots[-10:]) | (set(SENTINEL_ROOTS)&set(roots)))
    elif FTP_RUN_MODE != "full":
        raise ValueError("FTP_RUN_MODE must be audit or full")
    objects = read_frame("ftp_shallow_objects", list(empty_classified_ftp_objects().columns))
    pending_objects = {}
    connection = None
    def close():
        nonlocal connection
        if connection is not None:
            try:
                connection.quit()
            except Exception:
                try: connection.close()
                except Exception: pass
            connection = None
    def fetch_group(targets):
        nonlocal connection
        responses = {}
        for root in targets:
            for attempt in range(1, MAX_ROOT_RETRIES+1):
                if connection is None:
                    connection = ftp_connect()
                row, shallow = collect_root_ftp_proxy(connection, root)
                row["collection_attempts"] = attempt
                if not row["requires_ftp_retry"]:
                    break
                close()
                if attempt < MAX_ROOT_RETRIES:
                    time.sleep(RETRY_PAUSE_SECONDS * attempt)
            terminal = not row["requires_ftp_retry"]
            if terminal:
                pending_objects[root] = shallow
            frame = normalize_ftp_results(pd.DataFrame([row]))
            responses[root] = (frame, row["collection_status"], terminal)
        return responses
    def commit_objects():
        nonlocal objects
        objects = replace_response_groups(objects, pending_objects, ["root_accession"])
        if pending_objects or not os.path.isfile(PATHS["ftp_shallow_objects"]):
            if atomic_table(objects, PATHS["ftp_shallow_objects"]):
                _CHANGED_SOURCE_FILES.add(PATHS["ftp_shallow_objects"])
        pending_objects.clear()
    def known(old):
        if old.empty or not os.path.isfile(PATHS["ftp_shallow_objects"]):
            return {}
        old = normalize_ftp_results(old)
        return dict(zip(old.loc[~old.requires_ftp_retry.fillna(True),"root_accession"].astype(str),
                        old.loc[~old.requires_ftp_retry.fillna(True),"collection_status"].astype(str)))
    try:
        result = collect_keyed(block, "ftp_root_results", "ftp_check_state", "done_roots",
            {r:r for r in roots}, ["root_accession"], fetch_group, known,
            empty_columns=list(empty_ftp_roots().columns), pause=REQUEST_PAUSE_SECONDS,
            checkpoint_dict=True, extra_flush=commit_objects)
    finally:
        close()
    stat = RUN_REPORT[-1]
    if stat.get("attempted",0):
        metadata = pd.DataFrame({"run_mode":[FTP_RUN_MODE], "checkpoint_timestamp_utc":[utc_now()],
            "roots_newly_processed_this_run":[stat["checked"]], "roots_requested_this_run":[stat["attempted"]],
            "roots_in_checkpoint":[result.root_accession.nunique()]})
        atomic_table(metadata, PATHS["ftp_checkpoint_metadata"])
    return result

if ENABLE_FTP_TIMING:
    ftp_root_results = run_ftp_collection()
    print("FTP field is a retained public-artifact timestamp, not a release or minting date.")
else:
    ftp_root_results = read_frame("ftp_root_results", list(empty_ftp_roots().columns))
    print("FTP collection disabled; existing files retained.")


## 13c. Native MeSH recovery and assay evidence

Uses the complete native MeSH section of each current public study page, independently collected trial annotations, a fixed-year MeSH XML snapshot and OBI. Saved public pages are reused first. Raw labels, scope and hashes are retained; no term is copied from a linked entity. First use adds only missing sidecars and vocabulary files.


**Repair:** a host is paused only when an explicit future pause is recorded. Cached successful responses, actual Retry-After delays, and failure cooldowns are preserved. Rerun setup and Section 13c in `reuse`; do not clean or reset the cache.


In [ ]:
# @title 13c. Native MeSH recovery, trial annotations, ontology snapshots and assay evidence
# New sidecars only. Existing A/B source schemas and checkpoints are not changed.
import gzip, io, ast, unicodedata, xml.etree.ElementTree as SX
from collections import Counter, defaultdict
from urllib.parse import urlparse, urljoin, parse_qs
from email.utils import parsedate_to_datetime
from bs4 import BeautifulSoup

SEMANTIC_CODE_SHA256 = 'e24a875fa2b985b5197283af7c97d2fcf89c1231e60faa279a285563f54572df'
SEMANTIC_SCHEMA = 'dbgap-semantics-1.0'
SEMANTIC_REFERENCES = {
    'mesh': 'https://www.nlm.nih.gov/databases/download/mesh.html',
    'mesh_tree': 'https://www.nlm.nih.gov/mesh/intro_trees.html',
    'mesh_entry': 'https://www.nlm.nih.gov/mesh/intro_entry.html',
    'obi': 'https://obi-ontology.org/',
    'sra': 'https://www.ncbi.nlm.nih.gov/sra/docs/submitmeta/',
    'rcdc': 'https://report.nih.gov/funding/categorical-spending/rcdc-process',
    'ctgov': 'https://clinicaltrials.gov/data-api/api',
}

def sem_clean(x):
    if isinstance(x, dict): return {str(k):sem_clean(v) for k,v in x.items()}
    if isinstance(x, (list, tuple, set, np.ndarray)): return [sem_clean(v) for v in x]
    if x is None or x is pd.NA or x is pd.NaT: return None
    if isinstance(x, (datetime, pd.Timestamp)): return x.isoformat()
    if isinstance(x, np.generic): x=x.item()
    if isinstance(x, float) and not np.isfinite(x): return None
    return x

def sem_text(x):
    x=sem_clean(x)
    return '' if x is None else str(x).strip()

def sem_unpack(x):
    if isinstance(x, (list, tuple, dict, np.ndarray)): return sem_clean(x)
    for fun in (json.loads, ast.literal_eval):
        try:return fun(sem_text(x))
        except (ValueError, SyntaxError, TypeError):pass
    return None

def sem_norm(x):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', sem_text(x))).strip().casefold()

def sem_id(x):
    # CT.gov uses D plus six or nine digits. Never map an arbitrary number to a descriptor.
    m=re.search(r'(?<![A-Za-z0-9])(D(?:\d{9}|\d{6}))(?!\d)',sem_text(x),re.I)
    return m.group(1).upper() if m else None

def sem_root(x):
    m=re.fullmatch(r'(phs\d+)(?:\.v\d+\.p\d+(?:\.c\d+)?)?',sem_text(x),re.I)
    return m.group(1).lower() if m else None

def sem_sha(raw):return hashlib.sha256(raw).hexdigest()
def sem_file_sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def sem_write(path,obj,jsonl=False):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if jsonl:
        s=''.join(json.dumps(sem_clean(r),ensure_ascii=False,sort_keys=True,allow_nan=False)+'\n' for r in obj)
    else:s=json.dumps(sem_clean(obj),ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)
    raw=s.encode('utf-8')
    if path.exists() and path.read_bytes()==raw:return False
    tmp=path.with_name(path.name+'.tmp');tmp.write_bytes(raw);os.replace(tmp,path);return True

def sem_read_rows(path):
    p=Path(path)
    if not p.exists():return []
    if p.suffix=='.parquet':return sem_clean(pd.read_parquet(p).to_dict('records'))
    if p.suffix=='.csv':return sem_clean(pd.read_csv(p,keep_default_na=False).to_dict('records'))
    if p.suffix=='.jsonl':return [json.loads(s) for s in p.read_text().splitlines() if s.strip()]
    raise ValueError('Unsupported semantic source: '+str(p))

class SemanticCache:
    """Content-addressed public metadata. Failed refreshes retain previous good payloads."""
    def __init__(self,base,mode='reuse',months=3,max_calls=5000,max_failures=10,session=None,clock=None):
        self.base=Path(base);self.base.mkdir(parents=True,exist_ok=True)
        self.objects=self.base/'objects';self.objects.mkdir(exist_ok=True)
        self.path=self.base/'cache_state.json'
        self.state=json.loads(self.path.read_text()) if self.path.exists() else {}
        self.mode=mode;self.months=months;self.max_calls=max_calls;self.max_failures=max_failures
        self.session=session or requests.Session();self.clock=clock or (lambda:pd.Timestamp.now(tz='UTC'))
        self.calls=0;self.failures=Counter();self.host_pause={};self.attempted=set();self.used={}
    def now(self):return self.clock()
    def valid(self,r):
        p=self.base/r.get('path','__none__')
        return p.is_file() and sem_file_sha(p)==r.get('sha256')
    def read(self,key):
        r=self.state.get(key,{})
        if not self.valid(r):return None
        self.used[key]={k:v for k,v in r.items() if k not in ('last_attempt','failure_count','next_retry','last_error')}
        return (self.base/r['path']).read_bytes(),dict(r)
    def store(self,key,raw,url,checked=None,basis='remote_success'):
        sha=sem_sha(raw);p=self.objects/sha
        if not p.exists():p.write_bytes(raw)
        r={'path':str(p.relative_to(self.base)),'sha256':sha,'url':url,
           'last_success':sem_text(checked) or None,'basis':basis}
        self.state[key]=r;sem_write(self.path,self.state);self.used[key]=dict(r)
        return raw,dict(r)
    def adopt(self,key,raw,url,checked=None):
        if self.read(key) is None:self.store(key,raw,url,checked,'saved_public_response')
    def get(self,key,url,validator=None,timeout=(10,45),max_bytes=500_000_000):
        old=self.read(key);r=self.state.get(key,{})
        checked=pd.to_datetime(r.get('last_success'),utc=True,errors='coerce')
        recent=pd.notna(checked) and self.now()<checked+pd.DateOffset(months=self.months)
        if old and (self.mode in ('reuse','offline') or (self.mode=='check_stale' and recent)):return old,'cached'
        if self.mode=='offline':return old,'offline_missing' if not old else 'cached'
        retry=pd.to_datetime(r.get('next_retry'),utc=True,errors='coerce')
        host=urlparse(url).hostname
        if key in self.attempted or (pd.notna(retry) and self.now()<retry):return old,'cooldown'
        # No pause exists unless the server actually supplied one.
        # Comparing two successive self.now() calls falsely pauses an unlisted host.
        pause_until=self.host_pause.get(host)
        if pause_until is not None and self.now()<pause_until:return old,'server_pause'
        if self.calls>=self.max_calls or self.failures[host]>=self.max_failures:return old,'run_budget'
        self.calls+=1;self.attempted.add(key)
        err='unknown_failure';pause=None
        try:
            resp=self.session.get(url,timeout=timeout,headers={'User-Agent':'dbgap-semantic-profile/1.0','Accept':'*/*'},stream=True)
            with resp:
                if resp.status_code!=200:
                    err='http_'+str(resp.status_code)
                    ra=resp.headers.get('Retry-After')
                    if ra:
                        try:pause=self.now()+pd.Timedelta(seconds=int(ra))
                        except ValueError:
                            try:pause=pd.Timestamp(parsedate_to_datetime(ra)).tz_convert('UTC')
                            except (ValueError,TypeError):pass
                        if pause is not None:self.host_pause[host]=pause
                    if resp.status_code==429 and pause is None:self.host_pause[host]=self.now()+pd.Timedelta(hours=1)
                else:
                    chunks=[];size=0
                    for chunk in resp.iter_content(1024*1024):
                        size+=len(chunk)
                        if size>max_bytes:raise ValueError('payload_size_limit')
                        chunks.append(chunk)
                    raw=b''.join(chunks)
                    if validator:validator(raw)
                    return self.store(key,raw,url,self.now().isoformat()),'fetched'
        except (requests.RequestException,ValueError,SX.ParseError) as exc:err=type(exc).__name__+':'+str(exc)[:200]
        self.failures[host]+=1
        # 1/7/30-day failure cooldown; success is never advanced on a failure.
        count=int(r.get('failure_count',0))+1
        wait=self.now()+pd.Timedelta(days=(1,7,30)[min(count-1,2)])
        if pause is not None:wait=max(wait,pause)
        r.update(last_attempt=self.now().isoformat(),last_error=err,failure_count=count,next_retry=wait.isoformat())
        self.state[key]=r;sem_write(self.path,self.state)
        print('  semantic retry deferred:',key,err)
        return old,'failed_using_previous' if old else 'failed'


def sem_xml_bytes(raw):
    return gzip.decompress(raw) if raw[:2]==b'\x1f\x8b' else raw

def sem_validate_mesh(raw):
    # A full parse is done by parse_mesh below. Fail before replacing good cache on obvious wrong response.
    header=sem_xml_bytes(raw)[:10000]
    if b'DescriptorRecordSet' not in header:raise ValueError('Not a MeSH DescriptorRecordSet')
    # Verify the whole XML before advancing a successful-fetch clock or replacing prior good data.
    parsed=SX.fromstring(sem_xml_bytes(raw))
    if parsed.tag!='DescriptorRecordSet' or parsed.find('DescriptorRecord') is None:raise ValueError('Empty/wrong MeSH vocabulary')

def parse_mesh(raw):
    """Read annual XML locally. Preserve concept distinctions; only the preferred concept supplies strict synonyms."""
    descriptors={};release='';root_seen=False
    for event,e in SX.iterparse(io.BytesIO(sem_xml_bytes(raw)),events=('start','end')):
        if event=='start' and not root_seen:
            release=e.attrib.get('LanguageCode','');root_seen=True
        if event!='end' or e.tag!='DescriptorRecord':continue
        ident=e.findtext('DescriptorUI');label=e.findtext('DescriptorName/String')
        if not ident or not label:raise ValueError('Incomplete MeSH descriptor')
        terms=[]
        for c in e.findall('ConceptList/Concept'):
            preferred=c.attrib.get('PreferredConceptYN')=='Y'
            for t in c.findall('TermList/Term'):
                s=t.findtext('String')
                if s:terms.append({'label':s,'concept_id':c.findtext('ConceptUI'),'preferred_concept':preferred})
        descriptors[ident]={'mesh_id':ident,'label':label,
            'tree_numbers':[x.text for x in e.findall('TreeNumberList/TreeNumber') if x.text],
            'terms':terms,'scope_note':e.findtext('ConceptList/Concept/ScopeNote')}
        e.clear()
    if not descriptors:raise ValueError('Empty MeSH descriptor vocabulary')
    return descriptors

def parse_obi(raw):
    """Named OBI classes only, not an OWL reasoner. IDs/labels must exist in this saved snapshot."""
    ns={'owl':'http://www.w3.org/2002/07/owl#','rdf':'http://www.w3.org/1999/02/22-rdf-syntax-ns#','rdfs':'http://www.w3.org/2000/01/rdf-schema#'}
    tree=SX.fromstring(raw);terms={};version=''
    for ont in tree.findall('owl:Ontology',ns):
        v=ont.find('owl:versionIRI',ns)
        if v is not None:version=v.get('{'+ns['rdf']+'}resource','')
    for c in tree.findall('owl:Class',ns):
        uri=c.get('{'+ns['rdf']+'}about','');label=c.findtext('rdfs:label',namespaces=ns)
        if not re.search(r'/OBI_\d+$',uri) or not label:continue
        if c.findtext('owl:deprecated',namespaces=ns)=='true':continue
        synonyms=[x.text for x in c if x.tag.endswith('hasExactSynonym') and x.text]
        terms[uri]={'uri':uri,'label':label,'exact_synonyms':synonyms,
            'parents':[x.get('{'+ns['rdf']+'}resource') for x in c.findall('rdfs:subClassOf',ns) if x.get('{'+ns['rdf']+'}resource')]}
    if not terms:raise ValueError('No named active OBI classes')
    return terms,version


def parse_semantic_page(raw,expected_accession):
    soup=BeautifulSoup(raw,'html.parser');visible=soup.get_text(' ',strip=True)
    m=re.search(r'dbGaP Study Accession\s*:\s*(phs\d+(?:\.v\d+\.p\d+)?)',visible,re.I)
    if not m or sem_root(m.group(1))!=sem_root(expected_accession):raise ValueError('study_page_identity_mismatch')
    displayed=m.group(1).lower()
    if '.v' in expected_accession and displayed!=expected_accession.lower():raise ValueError('study_page_version_mismatch')
    heads=[t for t in soup.find_all(['h1','h2','h3','h4','div','strong','b','span'])
        if 'Diseases/Traits Related to Study' in t.get_text(' ',strip=True) and len(t.get_text(' ',strip=True))<180]
    terms=[]
    if heads:
        head=min(heads,key=lambda t:len(list(t.descendants)))
        for node in head.next_elements:
            tag=getattr(node,'name',None)
            if not tag:continue
            text=node.get_text(' ',strip=True)
            if tag in ('h1','h2','h3','h4') and node is not head and 'Diseases/Traits Related to Study' not in text:break
            if tag in ('h1','h2','h3','h4','strong','b','div') and len(text)<150 and re.match(r'(?:Links to Related Genes|Study Attribution|Authorized Data Access Requests|Selected Publications|Study History|Study Documents|Study Datasets)\b',text):break
            if tag!='a':continue
            link=urljoin('https://www.ncbi.nlm.nih.gov',node.get('href',''));u=urlparse(link)
            if u.hostname not in {'www.ncbi.nlm.nih.gov','ncbi.nlm.nih.gov','id.nlm.nih.gov','meshb.nlm.nih.gov'}:continue
            if '/mesh' not in u.path and '/record/ui' not in u.path:continue
            if text:terms.append({'source_id':sem_id(link),'source_label':text,'term_url':link})
    # Candidate text is the study's own title/description, never its bibliography.
    descriptions=[]
    for t in soup.find_all(['h1','h2','h3','h4','b','strong','div']):
        if t.get_text(' ',strip=True).strip(':')!='Study Description':continue
        for sib in t.next_siblings:
            if getattr(sib,'name',None) in ('h1','h2','h3','h4'):break
            text=sib.get_text(' ',strip=True) if hasattr(sib,'get_text') else str(sib).strip()
            if re.match(r'^(Study Attribution|Selected Publications|Diseases/Traits|Authorized Data)',text):break
            if text:descriptions.append(text)
            if sum(map(len,descriptions))>20000:break
        break
    unique=list({(t['source_id'],t['source_label']):t for t in terms}.values())
    return {'accession':displayed,'terms':unique,'status':('native_mesh_present' if unique else 'native_mesh_section_empty' if heads else 'native_mesh_section_absent'),
        'description':' '.join(descriptions)[:25000]}


def sem_mesh_atoms(value):
    v=sem_unpack(value)
    if v is None:
        if sem_text(value) and sem_text(value) not in ('[]','null'):return [{'source_id':sem_id(value),'source_label':sem_text(value)}]
        return []
    if isinstance(v,dict):
        ident=sem_id(v.get('mesh_id') or v.get('id') or v.get('code') or v.get('uri'))
        label=v.get('label') or v.get('display') or v.get('term') or v.get('name')
        if ident or label:return [{'source_id':ident,'source_label':sem_text(label)}]
        return [a for k in ('coding','condition','mesh_terms','terms') for a in sem_mesh_atoms(v.get(k)) if v.get(k) is not None]
    if isinstance(v,list):return [a for item in v for a in sem_mesh_atoms(item)]
    return []


def sem_native_record(entity_type,entity_id,atom,source,field,sha,accession=None,role='dataset',checked=None):
    row={'entity_type':entity_type,'entity_id':entity_id,'source_id':atom.get('source_id'),
        'source_label':atom.get('source_label',''),'source':source,'source_field':field,'source_sha256':sha,
        'source_accession':accession,'role':role,'source_checked_utc':checked,'origin':'native_annotation',
        'source_scope':('retained_root_union_not_version_resolved' if field=='mesh_terms' else 'version_specific_native_annotation' if entity_type=='dataset' else 'registry_record')}
    if atom.get('term_url'):row['term_url']=atom['term_url']
    row['evidence_id']=sem_sha(json.dumps(row,sort_keys=True).encode())[:24]
    return row


def parse_trial_semantics(raw,nct):
    data=json.loads(raw);protocol=data.get('protocolSection',{});ident=protocol.get('identificationModule',{})
    if ident.get('nctId')!=nct:raise ValueError('trial_identifier_mismatch')
    derived=data.get('derivedSection',{});annotations=[];module_states={}
    for name,role in [('conditionBrowseModule','condition'),('interventionBrowseModule','intervention')]:
        mod=derived.get(name)
        module_states[role]='module_absent' if not isinstance(mod,dict) else 'direct_terms_present' if mod.get('meshes') else 'module_present_no_direct_terms'
        for fld in ('meshes','ancestors'):
            for m in (mod or {}).get(fld,[]) or []:
                if isinstance(m,dict):annotations.append({'source_id':sem_id(m.get('id')),'source_label':m.get('term',''),
                    'role':role if fld=='meshes' else role+'_ancestor','field':'derivedSection.'+name+'.'+fld})
    return annotations,{'nct_id':nct,**module_states,'title':ident.get('officialTitle') or ident.get('briefTitle'),
        'condition_text':protocol.get('conditionsModule',{}).get('conditions',[]),
        'brief_summary':protocol.get('descriptionModule',{}).get('briefSummary','')}


def extract_assays(record,source,sha,current=None):
    """Keep version/participant/consent and the child root actually owning the count."""
    accession=sem_text(record.get('accession') or record.get('accession_sstr') or record.get('latest_accession'))
    host=sem_root(record.get('root_accession') or record.get('sstr_root_accession') or accession)
    raw=sem_unpack(record.get('sstr_raw'))
    if isinstance(raw,dict):
        av=raw.get('study',{}).get('accver',{});host=sem_root(av.get('accession')) or host
        accession=host+'.v'+str(av.get('version'))+'.p'+str(av.get('participant_set')) if host and av.get('version') and av.get('participant_set') else accession
        details=raw.get('study_stats',{}).get('cnt_by_consent_and_sample_use',[])
    else:details=sem_unpack(record.get('data_types_raw'))
    output=[]
    if isinstance(details,list):
        for d in details:
            if not isinstance(d,dict) or not sem_text(d.get('sample_use')):continue
            rawphs=d.get('phs');owned=sem_root(rawphs)
            if not owned and rawphs is not None:
                try:owned='phs'+str(int(rawphs)).zfill(6)
                except (ValueError,TypeError):pass
            owned=owned or host
            row={'root_accession':owned,'reported_under_root':host,'source_accession':accession,
                'data_version':d.get('version'),'participant_set':d.get('participant_set'),'consent_code':d.get('consent_code'),
                'raw_label':d['sample_use'],'is_sequence':d.get('is_sequence'),
                'sample_count':d.get('cnt_samples'),'subject_count':d.get('cnt_subjects'),
                'source':source,'source_sha256':sha,'source_field':'sstr_raw.study_stats.cnt_by_consent_and_sample_use' if isinstance(raw,dict) else 'data_types_raw',
                'scope':'self' if owned==host else 'explicit_child','raw_detail':d}
            row['evidence_id']=sem_sha(json.dumps(sem_clean(row),sort_keys=True).encode())[:24];output.append(row)
    if not output:
        for label in sem_unpack(record.get('data_types')) or []:
            if not isinstance(label,str):continue
            row={'root_accession':host,'reported_under_root':host,'source_accession':accession,'raw_label':label,
                'source':source,'source_sha256':sha,'source_field':'data_types','scope':'unspecified_root_summary',
                'data_version':None,'participant_set':None,'consent_code':None,'is_sequence':None,
                'sample_count':None,'subject_count':None,'raw_detail':None}
            row['evidence_id']=sem_sha(json.dumps(row,sort_keys=True).encode())[:24];output.append(row)
    return output


def semantic_sources(base,frames=None):
    stems={'roots':'dbgap_root_accessions','fhir':'dbgap_studies_raw','sstr':'sstr_study_summary',
        'metadata':'study_page_metadata','attribution':'dbgap_public_attribution','trials':'dbgap_ctgov_direct_links',
        'sra_metadata':'sra_experiment_metadata'}
    out={};manifest=[]
    for key,stem in stems.items():
        if frames is not None and key in frames:
            out[key]=sem_clean(frames[key]);raw=json.dumps(out[key],sort_keys=True).encode()
            manifest.append({'key':key,'path':None,'sha256':sem_sha(raw),'rows':len(out[key]),'test_fixture':True});continue
        p=next((Path(base)/(stem+s) for s in ('.parquet','.csv') if (Path(base)/(stem+s)).exists()),None)
        out[key]=sem_read_rows(p) if p else []
        if p:manifest.append({'key':key,'path':str(p),'sha256':sem_file_sha(p),'rows':len(out[key])})
    return out,manifest


def run_semantic_collection(base,mode='reuse',months=3,mesh_year=2026,frames=None,cache=None):
    base=Path(base);dest=base/'semantic_sources';dest.mkdir(exist_ok=True)
    cache=cache or SemanticCache(dest/'public_cache',mode,months,SEMANTIC_MAX_REQUESTS,SEMANTIC_MAX_FAILURES_PER_HOST)
    sources,inputs=semantic_sources(base,frames);input_lookup={x['key']:x for x in inputs}
    roots={r['root_accession']:r for r in sources['roots'] if sem_root(r.get('root_accession'))}
    if not roots:raise ValueError('Semantic collection requires the canonical root_accessions table.')
    def src(key):return input_lookup.get(key,{}).get('path') or key
    def sha(key):return input_lookup.get(key,{}).get('sha256')
    vocab={};vocab_status={};vocab_used=[]
    for key,url,parse,validate in [
        ('mesh_'+str(mesh_year),f'https://nlmpubs.nlm.nih.gov/projects/mesh/MESH_FILES/xmlmesh/desc{mesh_year}.gz',parse_mesh,sem_validate_mesh),
        ('obi','https://purl.obolibrary.org/obo/obi.owl',parse_obi,lambda b:parse_obi(b))]:
        cached,result=cache.get(key,url,validator=validate,timeout=(10,120))
        if not cached:
            vocab_status[key]=result
            print('  Vocabulary unavailable:',key,result,'(native labels retained; hierarchy not evaluated)')
            continue
        raw,meta=cached;parserkey=sem_sha(raw);parsedpath=dest/(key+'_'+parserkey[:16]+'.parsed.json')
        if parsedpath.exists():parsed=json.loads(parsedpath.read_text())
        else:
            try:
                v=parse(raw);parsed={'terms':v[0],'version_iri':v[1]} if key=='obi' else {'descriptors':v,'year':mesh_year}
            except (ValueError,SX.ParseError) as exc:
                vocab_status[key]='parse_failed:'+str(exc);continue
            sem_write(parsedpath,parsed)
        vocab['obi' if key=='obi' else 'mesh']=parsed;vocab_status[key]='available'
        vocab_used.append({'key':key,'url':url,'raw_path':str(cache.base/meta['path']),'sha256':meta['sha256'],
            'parsed_path':str(parsedpath),'parsed_sha256':sem_file_sha(parsedpath),'checked_utc':meta.get('last_success'),
            'year':mesh_year if key.startswith('mesh') else None,'version_iri':parsed.get('version_iri')})
    native=[];coverage=[];texts=[];assays=[];pairs=[];trial_info=[]
    # All explicit FHIR terms from retained CURRENT versions; root summary only as fallback.
    fhir_by_root=defaultdict(list)
    for r in sources['fhir']:
        root=sem_root(r.get('raw_accession') or r.get('root_accession'))
        if root:fhir_by_root[root].append(r)
    attr_by_acc={sem_text(r.get('accession')):r for r in sources['attribution']}
    md_by_root=defaultdict(list)
    for r in sources['metadata']:
        root=sem_root(r.get('accession') or r.get('root_accession'))
        if root:md_by_root[root].append(r)
    for i,(root,record) in enumerate(sorted(roots.items())):
        accession=sem_text(record.get('latest_accession')) or root
        fr=[r for r in fhir_by_root[root] if sem_text(r.get('raw_accession'))==accession]
        atoms=[]
        if fr:
            for r in fr:atoms.extend((x,src('fhir'),'mesh_terms_raw',sha('fhir')) for x in sem_mesh_atoms(r.get('mesh_terms_raw')))
        else:atoms.extend((x,src('roots'),'mesh_terms',sha('roots')) for x in sem_mesh_atoms(record.get('mesh_terms')))
        for a,s,f,h in atoms:native.append(sem_native_record('dataset',root,a,s,f,h,accession))
        title=sem_unpack(record.get('titles_seen')) or record.get('title') or ''
        if isinstance(title,list):title=' | '.join(map(str,title))
        texts.append({'entity_type':'dataset','entity_id':root,'source_field':'title','text':sem_text(title or record.get('title')),
            'source':src('roots'),'source_sha256':sha('roots'),'source_accession':accession})
        # Adopt the exact archived version from the attribution collector; no repeated web page request.
        key='study_page:'+accession;url='https://www.ncbi.nlm.nih.gov/projects/gap/cgi-bin/study.cgi?study_id='+accession
        a=attr_by_acc.get(accession,{})
        rel=a.get('raw_response_file')
        if rel:
            p=(base/rel).resolve()
            if p.is_relative_to(base.resolve()) and p.is_file():
                try:
                    packed=p.read_bytes();raw=gzip.decompress(packed) if p.suffix=='.gz' else packed
                    expected=a.get('response_sha256')
                    if expected and expected not in (sem_sha(raw),sem_sha(packed)):raise ValueError('archived_page_hash_mismatch')
                    parse_semantic_page(raw,accession)
                    cache.adopt(key,raw,a.get('source_url') or url,a.get('attribution_fetched_utc'))
                except (ValueError,OSError,EOFError):pass
        got,action=cache.get(key,url,lambda b:parse_semantic_page(b,accession))
        page_status='page_unavailable';page_count=0
        if got:
            raw,meta=got
            try:
                parsed=parse_semantic_page(raw,accession);page_status=parsed['status'];page_count=len(parsed['terms'])
                for x in parsed['terms']:native.append(sem_native_record('dataset',root,x,url,'Diseases/Traits Related to Study (MeSH terms)',meta['sha256'],accession,checked=meta.get('last_success')))
                if parsed['description']:texts.append({'entity_type':'dataset','entity_id':root,'source_field':'study_description','text':parsed['description'],
                    'source':url,'source_sha256':meta['sha256'],'source_accession':accession})
            except ValueError as exc:page_status=str(exc)
        for r in md_by_root[root]:
            if sem_text(r.get('accession'))!=accession:continue
            if sem_text(r.get('primary_disease')):texts.append({'entity_type':'dataset','entity_id':root,'source_field':'primary_disease','text':r['primary_disease'],
                'source':src('metadata'),'source_sha256':sha('metadata'),'source_accession':accession})
        coverage.append({'entity_type':'dataset','entity_id':root,'source_accession':accession,
            'native_field_terms':len(atoms),'native_page_terms':page_count,'page_status':page_status,'payload_available':bool(got),
            'last_remote_check':got[1].get('last_success') if got else None})
        if (i+1)%100==0:print(f'  semantic pages: {i+1:,}/{len(roots):,}; remote requests {cache.calls:,}')
    # One full public registry payload per unique NCT, not one request per accession-NCT pair.
    ncts=sorted({sem_text(r.get('nct_id')).upper() for r in sources['trials'] if re.fullmatch(r'NCT\d{8}',sem_text(r.get('nct_id')),re.I)})
    for nct in ncts:
        url='https://clinicaltrials.gov/api/v2/studies/'+nct
        got,action=cache.get('trial:'+nct,url,lambda b:parse_trial_semantics(b,nct))
        if got:
            raw,meta=got;anns,info=parse_trial_semantics(raw,nct)
            for x in anns:native.append(sem_native_record('trial',nct,x,url,x['field'],meta['sha256'],role=x['role'],checked=meta.get('last_success')))
            info.update(source=url,source_sha256=meta['sha256']);trial_info.append(info)
            for field in ('title','condition_text','brief_summary'):
                values=info[field] if isinstance(info[field],list) else [info[field]]
                for val in values:
                    if sem_text(val):texts.append({'entity_type':'trial','entity_id':nct,'source_field':field,'text':val,'source':url,'source_sha256':meta['sha256'],'source_accession':None})
            coverage.append({'entity_type':'trial','entity_id':nct,'condition_state':info['condition'],'intervention_state':info['intervention'],'payload_available':bool(got)})
        else:coverage.append({'entity_type':'trial','entity_id':nct,'condition_state':'retrieval_unavailable','intervention_state':'retrieval_unavailable','payload_available':bool(got)})
    for row in sources['trials']:
        root=sem_root(row.get('root_accession'));nct=sem_text(row.get('nct_id')).upper()
        if root not in roots or nct not in ncts:continue
        field=sem_text(row.get('dbgap_evidence_field'))
        pairs.append({'root_accession':root,'nct_id':nct,'source':src('trials'),'source_sha256':sha('trials'),
            'original_linkage_method':row.get('linkage_method'),'source_field':field,
            'relationship_basis':'publication_mediated_reference' if field=='publications_raw' else 'recorded_reference_unadjudicated',
            'role_adjudicated':False,'original_evidence_excerpt':row.get('dbgap_evidence_text')})
    # Preserve all SSTR observations; B selects current root/version context and deduplicates repeated evidence.
    for row in sources['sstr']:
        if sem_text(row.get('sstr_fetch_note')).lower() not in ('ok','success'):continue
        assays.extend(extract_assays(row,src('sstr'),sha('sstr')))
    # Optional pre-existing explicit-root SRA metadata is an independent comparison source.
    for row in sources.get('sra_metadata',[]):
        root=sem_root(row.get('root_accession'))
        label=row.get('library_strategy') or row.get('LibraryStrategy')
        if root not in roots or not sem_text(label):continue
        evidence={'root_accession':root,'reported_under_root':root,
            'source_accession':row.get('dbgap_accession') or (row.get('accession') if sem_root(row.get('accession'))==root else None),
            'raw_label':label,'source':src('sra_metadata'),'source_sha256':sha('sra_metadata'),
            'source_field':'library_strategy','scope':'self','data_version':None,'participant_set':None,'consent_code':None,
            'is_sequence':None,'sample_count':None,'subject_count':None,'raw_detail':row,
            'library_source':row.get('library_source') or row.get('LibrarySource'),
            'library_selection':row.get('library_selection') or row.get('LibrarySelection'),
            'instrument_platform':row.get('platform') or row.get('Platform'),
            'experiment_accession':row.get('experiment_accession') or row.get('Experiment')}
        evidence['evidence_id']=sem_sha(json.dumps(sem_clean(evidence),sort_keys=True).encode())[:24];assays.append(evidence)
    # Full lexical outputs, not normalized into the existing tables.
    native=list({r['evidence_id']:r for r in native}.values());assays=list({r['evidence_id']:r for r in assays}.values())
    outputs={}
    for name,rows in [('native_annotations',native),('source_coverage',coverage),('entity_text',texts),('assay_observations',assays),('dataset_trial_evidence',pairs),('trial_info',trial_info)]:
        p=dest/(name+'.jsonl');sem_write(p,rows,True);outputs[p.name]=sem_file_sha(p)
    manifest={'schema':SEMANTIC_SCHEMA,'scope':'Native public annotations and assay evidence; not official RCDC classifications.',
        'mesh_year':mesh_year,'collector_code_sha256':SEMANTIC_CODE_SHA256,'inputs':inputs,'outputs':outputs,'vocabularies':vocab_used,'vocabulary_status':vocab_status,
        'counts':{'roots':len(roots),'trials':len(ncts),'native_annotations':len(native),'assay_observations':len(assays)},
        'references':SEMANTIC_REFERENCES}
    # Do not touch manifest on an identical cached run; source check clocks live only in cache_state.json.
    sem_write(dest/'manifest.json',manifest)
    print('Semantic collection:',manifest['counts']);print('Vocabulary status:',vocab_status)
    print('Remote GET calls this run:',cache.calls,'| source tables unchanged |',dest)
    if any(i.get('path') and sem_file_sha(i['path'])!=i['sha256'] for i in inputs):raise RuntimeError('Source changed during semantic collection; rerun to obtain a consistent snapshot.')
    return manifest

if ENABLE_SEMANTIC_COLLECTION:
    semantic_collection_manifest=run_semantic_collection(DRIVE_ROOT,mode_for('semantics'),STALE_MONTHS,MESH_YEAR)
else:
    print('Semantic collection disabled. Existing semantic_sources files were not modified.')


In [ ]:
# @title
# Semantic sanity checks: sidecars and unchanged canonical source universe.
if ENABLE_SEMANTIC_COLLECTION:
    _sm=semantic_collection_manifest
    assert _sm['counts']['roots']==root_table.root_accession.nunique()
    assert all(sem_file_sha(Path(DRIVE_ROOT)/'semantic_sources'/name)==sha for name,sha in _sm['outputs'].items())
    print('Semantic sidecars verified; source-root universe unchanged. Ontology gaps remain explicit.')


## 13d. Participation and replication instruments
No new API requests. This block inventories optional dated administrative records, creates header-only templates outside the data-input directory, and retains a prospective first-observed data-type ledger. It never reconstructs historical submissions from FTP timestamps.

In [ ]:
# @title 13d. Optional administrative inputs and prospective observation ledgers (no API calls)
# All historical event tables are optional. Header templates are examples of required instruments,
# not collected records. Original source tables and checkpoints are never overwritten here.
ENABLE_PARTICIPATION_LEDGER = True
WRITE_ADMIN_INPUT_TEMPLATES = True
from pathlib import Path as _P
import csv as _csv
_pbase=_P(DRIVE_ROOT);_pcontrol=_pbase/'participation_sources';_pcontrol.mkdir(exist_ok=True)
_optional_schemas = {'submission_events': ['submission_id', 'root_accession', 'entity_id', 'entity_name', 'institution_id', 'institution_name', 'submitted_at', 'role', 'source_url', 'validated'], 'assay_submission_events': ['root_accession', 'data_type', 'submitted_at', 'source_url', 'validated'], 'rep_lifecycle': ['study_id', 'root_accession', 'study_type', 'release_date', 'followup_end', 'release_verified', 'request_history_complete', 'publication_history_complete', 'n_participants', 'completion_year', 'purpose', 'research_area', 'source_url'], 'rep_access': ['request_id', 'study_id', 'requester_id', 'institution_id', 'consent_group_id', 'request_date', 'decision_date', 'approved', 'fulfilled_date', 'download_date', 'download_observation_complete', 'request_unit', 'purpose_category', 'purpose_validated', 'external_funding', 'external_funding_known', 'source_url', 'validated'], 'rep_publications': ['study_id', 'request_id', 'pmid', 'publication_date', 'secondary_use_verified', 'request_link_verified', 'source_url', 'validated'], 'rep_institutions': ['institution_id', 'institution_label', 'profit_status', 'institution_type', 'country', 'continent', 'latitude', 'longitude', 'source_url', 'validated'], 'rep_samples': ['study_id', 'root_accession', 'sample_id', 'broad_phenotype', 'technology', 'snp_array_size', 'snapshot_date', 'source_url', 'validated'], 'rep_topics': ['pmid', 'topic', 'source_url', 'validated'], 'rep_metrics': ['pmid', 'journal', 'citation_count', 'citation_percentile', 'percentile_direction', 'metric_system', 'metric_as_of', 'publication_date', 'cohort', 'source_url', 'validated']}
_source_names={'rep_lifecycle':'study_lifecycle','rep_access':'access_events','rep_publications':'publication_events',
 'rep_institutions':'institutions','rep_samples':'sample_assays','rep_topics':'publication_topics','rep_metrics':'publication_metrics',
 'submission_events':'submission_events','assay_submission_events':'assay_submission_events'}
_template_dir=_pcontrol/'input_templates_NOT_DATA';_template_dir.mkdir(exist_ok=True)
for _key,_fields in _optional_schemas.items():
    _stem=_source_names[_key]
    _real=next((p for p in [_pbase/'replication_inputs'/(_stem+'.parquet'),_pbase/'replication_inputs'/(_stem+'.csv'),_pbase/(_stem+'.parquet'),_pbase/(_stem+'.csv')] if p.is_file()),None)
    if _real:
        _f=pd.read_parquet(_real) if _real.suffix=='.parquet' else pd.read_csv(_real,dtype=str)
        print(f'{_stem}: {len(_f):,} rows; missing expected columns: {sorted(set(_fields)-set(_f))}')
    else:print(f'{_stem}: not supplied; related historical analyses will be withheld or explicitly labelled proxies.')
    if WRITE_ADMIN_INPUT_TEMPLATES:
        _target=_template_dir/(_stem+'.csv')
        if not _target.exists():
            with open(_target,'w',newline='') as _h:_csv.writer(_h).writerow(_fields)

# A first-observed ledger provides a future longitudinal record without backdating observations.
# It is NOT automatically used as the date the data type was submitted or became accessible.
if ENABLE_PARTICIPATION_LEDGER:
    _af=_pbase/'semantic_sources'/'assay_observations.jsonl'
    _ledger=_pcontrol/'data_types_first_observed.csv'
    _seen=pd.read_csv(_ledger,dtype=str).fillna('') if _ledger.exists() else pd.DataFrame(columns=['root_accession','raw_label','data_version','participant_set','first_observed_utc','observation_basis'])
    _existing=set(tuple(str(r.get(k,'')) for k in ['root_accession','raw_label','data_version','participant_set']) for r in _seen.to_dict('records'))
    _new=[]
    if _af.is_file():
        for _line in _af.read_text().splitlines():
            if not _line.strip():continue
            _r=json.loads(_line);_key=tuple(str(_r.get(k) if _r.get(k) is not None else '') for k in ['root_accession','raw_label','data_version','participant_set'])
            if not _key[0] or not _key[1] or _key in _existing:continue
            _existing.add(_key);_new.append(dict(zip(['root_accession','raw_label','data_version','participant_set'],_key))|{'first_observed_utc':datetime.now(timezone.utc).isoformat(),'observation_basis':'prospective first observation in local cached source; not historical submission'})
    if _new:
        _all=pd.concat([_seen,pd.DataFrame(_new)],ignore_index=True);_tmp=_ledger.with_suffix('.csv.tmp');_all.to_csv(_tmp,index=False);os.replace(_tmp,_ledger)
    print(f'Data-type observation ledger: {len(_seen)+len(_new):,} records; {len(_new):,} newly observed (never backdated).')
print('Optional administrative tables should be placed in DRIVE_ROOT/replication_inputs/. No download or collection was attempted for them.')


## 14. Collection inventory and brief checks
Inspect source coverage before combining tables. A file hash identifies the saved snapshot, not its scientific validity.


In [ ]:
# @title 14. Collection inventory and brief checks
source_specs = [
    ("FHIR roots", "root_accessions", "root_accession"),
    ("DAR", "dar_records", "queried_accession"),
    ("Study counts", "study_counts", "accession"),
    ("Study page", "study_page_meta", "accession"),
    ("SSTR", "sstr_summary", "accession"),
    ("Direct NCT", "clinical_trials_direct", "nct_id"),
    ("PubMed", "pubmed_metadata", "pmid"),
    ("Authorized Access / SRA", "authorized_access_metadata", "root_accession"),
    ("iCite", "icite_metrics", "pmid"),
    ("RePORTER PMID links", "reporter_pub_links_full", "pmid"),
    ("RePORTER projects", "reporter_projects_full", "core_project_num"),
    ("FTP artifact provenance", "ftp_root_results", "root_accession"),
]
source_inventory = []
for label, key, unit in source_specs:
    frame = read_frame(key)
    source_inventory.append({"source":label, "rows":len(frame), "distinct_ids":frame[unit].nunique() if unit in frame else None,
                             "columns":len(frame.columns), "file_exists":os.path.isfile(PATHS[key]),
                             "sha256":sha256_file(PATHS[key])})
display(pd.DataFrame(source_inventory))
if RUN_REPORT:
    display(pd.DataFrame(RUN_REPORT).drop(columns="age_basis",errors="ignore"))
print("A completed negative API lookup is not evidence of no real-world access, publications, or funding.")


## 15. Joins and existing rollups
Rebuild only when source hashes, recipe/options or required outputs change. The base join, clinical fields, full PubMed links, and the existing base/robust iCite and funding rollups are assembled in a workspace before final publication. This section makes no source API calls.


In [ ]:
# @title 15. Joins and existing rollups
def dar_summary_per_accession():
    if not os.path.exists(PATHS["dar_records"]):
        return pd.DataFrame(columns=["queried_accession", "n_dar_total", "n_dar_approved", "n_dar_closed",
                "n_dar_rejected", "n_unique_orgs"])
    dar = pd.read_parquet(PATHS["dar_records"])
    if dar.empty:
        return pd.DataFrame(columns=["queried_accession", "n_dar_total", "n_dar_approved", "n_dar_closed",
                "n_dar_rejected", "n_unique_orgs"])
    def summarize(group):
        statuses = group["status"].fillna("").astype(str).str.lower().value_counts() if "status" in group else pd.Series(dtype=int)
        return pd.Series({
            "n_dar_total": len(group),
            "n_dar_approved": statuses.get("approved", 0),
            "n_dar_closed": statuses.get("closed", 0),
            "n_dar_rejected": statuses.get("rejected", 0),
            "n_unique_orgs": group["org_name"].nunique() if "org_name" in group else np.nan,
        })
    return dar.groupby("queried_accession", group_keys=True).apply(summarize, include_groups=False).reset_index()

def _extract_child_accessions(children_raw):
    if children_raw is None or pd.isna(children_raw):
        return []
    try:
        payload = json.loads(children_raw)
    except (TypeError, ValueError):
        return []
    data = payload.get("data", {}) if isinstance(payload, dict) else {}
    items = data.get("children", []) if isinstance(data, dict) else []
    result = []
    for item in items:
        if isinstance(item, str): result.append(item)
        elif isinstance(item, dict):
            for key in ("accession", "study_id", "phs_accession", "id"):
                if item.get(key): result.append(str(item[key])); break
    return result

BASE_JOIN_OUTPUTS = ["study_counts_analysis_ready", "study_page_metadata_analysis_ready", "sstr_summary_analysis_ready",
                     "pubmed_links", "icite_study_rollup", "joined"]
FUNDING_OUTPUTS = ["reporter_pmid_award_mn", "reporter_core_funding", "reporter_study_rollup",
    "reporter_verification"]
JOIN_INPUTS = ["root_accessions", "dar_records", "dar_checkpoint", "study_counts", "study_page_meta", "sstr_summary",
               "clinical_trials_direct", "pubmed_metadata", "icite_metrics", "reporter_pub_links_full", "reporter_projects_full"]

def build_join_products(include_funding=True):
    staging = Path(DRIVE_ROOT) / "notebook_a_work" / ("joins_" + RUN_ID)
    staging.mkdir(parents=True, exist_ok=True)
    output_keys = BASE_JOIN_OUTPUTS + (FUNDING_OUTPUTS if include_funding else [])
    original_paths = {key: PATHS[key] for key in output_keys}
    for key, path in original_paths.items():
        PATHS[key] = str(staging / Path(path).name)
    try:
        require_files("root_accessions", "study_counts", "study_page_meta", "sstr_summary")
        root_table = pd.read_parquet(PATHS["root_accessions"])
        current_latest_accessions = set(root_table["latest_accession"].dropna().astype(str))
        study_counts_raw = pd.read_parquet(PATHS["study_counts"]) if os.path.exists(PATHS["study_counts"]) else pd.DataFrame()
        study_page_raw = pd.read_parquet(PATHS["study_page_meta"]) if os.path.exists(PATHS["study_page_meta"]) else pd.DataFrame()
        sstr_raw = pd.read_parquet(PATHS["sstr_summary"]) if os.path.exists(PATHS["sstr_summary"]) else pd.DataFrame()
        study_counts_clean = compact_study_level_table(study_counts_raw,
            latest_accessions=current_latest_accessions)
        study_page_clean = compact_study_level_table(study_page_raw, latest_accessions=current_latest_accessions)
        sstr_clean = compact_study_level_table(sstr_raw, latest_accessions=current_latest_accessions,
            prefer_ok_col="sstr_fetch_note")
        analysis_ready_paths = {
            "study_counts_analysis_ready": PATHS["study_counts_analysis_ready"],
            "study_page_metadata_analysis_ready": PATHS["study_page_metadata_analysis_ready"],
            "sstr_summary_analysis_ready": PATHS["sstr_summary_analysis_ready"],
        }
        study_counts_clean.to_parquet(analysis_ready_paths["study_counts_analysis_ready"], index=False)
        study_page_clean.to_parquet(analysis_ready_paths["study_page_metadata_analysis_ready"], index=False)
        sstr_clean.to_parquet(analysis_ready_paths["sstr_summary_analysis_ready"], index=False)
        print(f"Study counts: {len(study_counts_raw):,} raw -> {len(study_counts_clean):,} analysis-ready")
        print(f"Study-page metadata: {len(study_page_raw):,} raw -> {len(study_page_clean):,} analysis-ready")
        print(f"SSTR: {len(sstr_raw):,} raw -> {len(sstr_clean):,} analysis-ready")
        dar_summary = dar_summary_per_accession()
        joined = root_table.merge(dar_summary, left_on="latest_accession", right_on="queried_accession", how="left")
        joined = joined.merge(study_counts_clean, left_on="latest_accession", right_on="accession", how="left",
            suffixes=("", "_counts"))
        joined = joined.merge(study_page_clean, left_on="latest_accession", right_on="accession", how="left",
            suffixes=("", "_page"))
        joined = joined.merge(sstr_clean, left_on="latest_accession", right_on="accession", how="left",
            suffixes=("", "_sstr"))
        root_of = lambda accession: str(accession).split(".")[0] if accession else None
        pub_by_root = {root_of(row["latest_accession"]): row["n_publications"] for _,
            row in joined.iterrows() if "n_publications" in joined.columns and pd.notna(row.get("n_publications"))}
        rollups = {}
        if {"children_raw", "n_children", "n_publications"}.issubset(joined.columns):
            for _, row in joined.loc[joined["n_children"].fillna(0).gt(0)].iterrows():
                if row.get("n_publications", 0) == 0:
                    child_total = sum(pub_by_root.get(root_of(child),
                            0) for child in _extract_child_accessions(row.get("children_raw")))
                    if child_total: rollups[row["root_accession"]] = child_total
        joined["n_publications_rollup_adjustment"] = joined["root_accession"].map(rollups).fillna(0)
        joined["n_publications_corrected"] = pd.to_numeric(joined.get("n_publications", 0),
            errors="coerce").fillna(0) + joined["n_publications_rollup_adjustment"]
        assert len(joined) == len(root_table), "Joined table cardinality differs from root table."
        assert not joined["root_accession"].duplicated().any(), "Duplicate root accessions remain after analysis-ready compaction."
        joined.to_parquet(PATHS["joined"], index=False)
        print(f"Analysis-ready joined table: {len(joined):,} unique roots; rollup applied to {len(rollups):,} umbrella studies.")
        # Enriched PMID linkage is a local join, not another acquisition stage.
        mapping = extract_pmids_from_page_metadata(study_page_clean)
        metadata = read_frame("pubmed_metadata")
        metadata = metadata.copy()
        metadata["pmid"] = metadata["pmid"].astype(str)
        require_columns(metadata, ["pmid", "pubmed_fetch_note"], "PubMed metadata")
        assert metadata.pmid.is_unique, "Duplicate PubMed IDs; inspect raw inputs before joining"
        pubmed_links = mapping.merge(metadata, on="pmid", how="left", validate="many_to_one")
        pubmed_links.to_parquet(PATHS["pubmed_links"], index=False)
        print("PubMed linkage:", len(pubmed_links), "root-publication rows")
        ct_links = pd.read_parquet(PATHS["clinical_trials_direct"])

        ct_root_summary = (
            ct_links.groupby("root_accession")
            .agg(
                n_direct_nct_links=("nct_id", "nunique"),
                n_direct_ctgov_pairs=("nct_id", "size"),
                direct_ctgov_link_present=("nct_id", lambda x: x.notna().any()),
                ctgov_any_interventional=("ctgov_study_type", lambda x: x.eq("INTERVENTIONAL").any()),
                ctgov_any_observational=("ctgov_study_type", lambda x: x.eq("OBSERVATIONAL").any()),
                ctgov_any_completed=("ctgov_overall_status", lambda x: x.eq("COMPLETED").any()),
                ctgov_any_active_not_recruiting=("ctgov_overall_status",
                    lambda x: x.eq("ACTIVE_NOT_RECRUITING").any()),
                ctgov_nct_ids=("nct_id", lambda x: json.dumps(sorted(set(x.dropna().astype(str))))),
                ctgov_lead_sponsors=("ctgov_lead_sponsor",
                    lambda x: json.dumps(sorted(set(x.dropna().astype(str))))),
                ctgov_conditions=("ctgov_conditions", lambda x: json.dumps(sorted(set(x.dropna().astype(str))))),
            )
            .reset_index()
        )

        joined = pd.read_parquet(PATHS["joined"])
        joined = joined.drop(columns=[c for c in ct_root_summary.columns if c != "root_accession" and c in joined.columns],
            errors="ignore")
        joined = joined.merge(ct_root_summary, on="root_accession", how="left")
        for col in ["direct_ctgov_link_present", "ctgov_any_interventional", "ctgov_any_observational",
            "ctgov_any_completed", "ctgov_any_active_not_recruiting"]:
            joined[col] = joined[col].astype("boolean").fillna(False).astype(bool)
        for col in ["n_direct_nct_links", "n_direct_ctgov_pairs"]:
            joined[col] = joined[col].fillna(0).astype(int)
        assert not joined["root_accession"].duplicated().any()
        joined.to_parquet(PATHS["joined"], index=False)


        icite = pd.read_parquet(PATHS["icite_metrics"])
        links = pd.read_parquet(PATHS['pubmed_links'])[['root_accession', 'pmid']].dropna().copy()
        links.pmid = links.pmid.astype(str)
        ok = icite[icite.icite_fetch_note.eq('ok')].copy()
        ok.pmid = ok.pmid.astype(str)
        for col in ['rcr', 'citation_count', 'apt', 'nih_percentile', 'n_clinical_citations']:
            ok[col] = pd.to_numeric(ok.get(col), errors='coerce')
        for col in ['is_clinical', 'is_research_article', 'provisional']:
            if col in ok:
                ok[col] = ok[col].astype('boolean')
        m = links.merge(ok, on='pmid', how='left').drop_duplicates(['root_accession', 'pmid'])
        all_agg = m.groupby('root_accession').agg(n_pmids_linked=('pmid', 'nunique'),
            n_pmids_icite_ok=('icite_fetch_note', lambda s: int(s.eq('ok').sum())),
            total_citations=('citation_count', 'sum'), n_clinical_citations=('n_clinical_citations', 'sum'),
            any_clinical=('is_clinical', lambda s: bool(s.fillna(False).any())), mean_apt=('apt', 'mean'))
        rcr_agg = m[m.rcr.notna()].groupby('root_accession').agg(n_pmids_with_rcr=('rcr', 'size'), mean_rcr=('rcr',
                'mean'), median_rcr=('rcr', 'median'), max_rcr=('rcr', 'max'), sum_rcr=('rcr', 'sum'),
            mean_nih_percentile=('nih_percentile', 'mean'))
        rollup = all_agg.join(rcr_agg, how='left').reset_index()
        rollup.to_parquet(PATHS['icite_study_rollup'], index=False)
        joined = pd.read_parquet(PATHS['joined'])
        cols = [col for col in rollup if col != 'root_accession']
        joined = joined.drop(columns=[col for col in cols if col in joined], errors='ignore').merge(rollup,
            on='root_accession', how='left')
        for col in ['n_pmids_linked', 'n_pmids_icite_ok', 'n_pmids_with_rcr', 'total_citations',
            'n_clinical_citations']:
            joined[col] = joined[col].fillna(0)
        joined['any_clinical'] = joined.any_clinical.astype('boolean').fillna(False)
        assert joined.root_accession.is_unique
        joined.to_parquet(PATHS['joined'], index=False)
        print(f'Rollup: {len(rollup):,} studies; {(rollup.n_pmids_with_rcr.fillna(0) > 0).sum():,} with ≥1 RCR-scored PMID.')
        icite = pd.read_parquet(PATHS["icite_metrics"]).copy()
        links = pd.read_parquet(PATHS["pubmed_links"])[
            ["root_accession", "pmid"]
        ].dropna().copy()
        icite["pmid"] = icite["pmid"].astype(str)
        links["pmid"] = links["pmid"].astype(str)
        icite_ok = icite.loc[
            icite["icite_fetch_note"].eq("ok")
        ].copy()
        icite_ok["rcr"] = pd.to_numeric(
            icite_ok["rcr"],
            errors="coerce",
        )
        impact_long = (
            links
            .merge(
                icite_ok[
                    [
                        "pmid",
                        "rcr",
                        "citation_count",
                        "nih_percentile",
                        "apt",
                        "is_research_article",
                        "is_clinical",
                    ]
                ],
                on="pmid",
                how="left",
            )
            .drop_duplicates(["root_accession", "pmid"])
        )
        impact_long["rcr_at_least_1"] = impact_long["rcr"].ge(1)
        impact_long["rcr_at_least_2"] = impact_long["rcr"].ge(2)
        impact_long["rcr_at_least_5"] = impact_long["rcr"].ge(5)
        robust_impact_rollup = (
            impact_long
            .groupby("root_accession")
            .agg(
                n_pmids_with_rcr=("rcr", lambda s: int(s.notna().sum())),
                median_rcr=("rcr", "median"),
                p25_rcr=("rcr", lambda s: s.quantile(0.25)),
                p75_rcr=("rcr", lambda s: s.quantile(0.75)),
                share_rcr_ge_1=("rcr_at_least_1", "mean"),
                share_rcr_ge_2=("rcr_at_least_2", "mean"),
                share_rcr_ge_5=("rcr_at_least_5", "mean"),
                median_nih_percentile=("nih_percentile", "median"),
                median_apt=("apt", "median"),
            )
            .reset_index()
        )
        rollup = pd.read_parquet(PATHS["icite_study_rollup"])
        replace_columns = [
            column
            for column in robust_impact_rollup.columns
            if column != "root_accession"
        ]
        rollup = rollup.drop(
            columns=[
                column
                for column in replace_columns
                if column in rollup.columns
            ],
            errors="ignore",
        )
        rollup = rollup.merge(
            robust_impact_rollup,
            on="root_accession",
            how="left",
            validate="one_to_one",
        )
        rollup.to_parquet(
            PATHS["icite_study_rollup"],
            index=False,
        )
        joined = pd.read_parquet(PATHS["joined"])
        joined = joined.drop(
            columns=[
                column
                for column in replace_columns
                if column in joined.columns
            ],
            errors="ignore",
        )
        joined = joined.merge(
            robust_impact_rollup,
            on="root_accession",
            how="left",
            validate="one_to_one",
        )
        assert joined["root_accession"].is_unique
        joined.to_parquet(
            PATHS["joined"],
            index=False,
        )
        if include_funding:
            require_files("pubmed_links", "joined")
            pub_links = pd.read_parquet(PATHS["reporter_pub_links_full"])
            projects = pd.read_parquet(PATHS["reporter_projects_full"])
            require_columns(pub_links, ["pmid", "core_project_num"], "RePORTER publication links")
            require_columns(projects, ["core_project_num", "fiscal_year", "award_amount", "admin_ic",
                    "activity_code"], "RePORTER projects")
            if pub_links.empty or projects.empty:
                    raise ValueError("Funding inputs are empty; existing outputs were not overwritten.")
            pub_links = pub_links.dropna(subset=["core_project_num"]).drop_duplicates(["pmid",
                    "core_project_num"]).copy()
            pub_links["pmid"] = pub_links["pmid"].astype(str)
            pub_links["funding_class"] = pub_links["core_project_num"].apply(classify_core)
            if pub_links.empty:
                    raise ValueError("No non-null linked cores; existing funding outputs were not overwritten.")
            n_corpus_pmids = pd.read_parquet(PATHS["pubmed_links"])["pmid"].dropna().astype(str).nunique()
            projects["core_project_num"] = projects["core_project_num"].apply(to_core)
            core_funding = current_period_funding(projects)
            core_funding["funding_class"] = core_funding["core_project_num"].apply(classify_core)
            core_funding["priceable"] = core_funding["funding_class"].eq("grant") & core_funding["current_period_award"].notna()
            assert core_funding["core_project_num"].is_unique, "Funding must have one row per core."
            core_funding.to_parquet(PATHS["reporter_core_funding"], index=False)
            links = pd.read_parquet(PATHS["pubmed_links"])
            links = links[["root_accession", "pmid"]].dropna().copy()
            links["pmid"] = links["pmid"].astype(str)
            pub_links_slim = pub_links.drop(columns=[c for c in ["funding_class", "priceable",
                                                                     "award_amount", "admin_ic"]
                                                         if c in pub_links.columns])
            core_cols = [c for c in ["core_project_num", "current_period_award", "current_period_years",
                                         "funding_class", "priceable", "admin_ic", "activity_code"]
                             if c in core_funding.columns]
            mn = (pub_links_slim.merge(core_funding[core_cols], on="core_project_num", how="left")
                                    .merge(links, on="pmid", how="left"))
            for req in ["funding_class", "priceable", "current_period_award"]:
                    if req not in mn.columns:
                        raise KeyError(f"'{req}' missing from mn; core_funding columns were {list(core_funding.columns)}")
            mn.to_parquet(PATHS["reporter_pmid_award_mn"], index=False)
            awards_per_pmid = pub_links.groupby("pmid")["core_project_num"].nunique()
            pmids_per_core = pub_links.groupby("core_project_num")["pmid"].nunique()
            print("\nM:N structure:")
            print(f"  Awards per PMID  -- median {awards_per_pmid.median():.0f}, "
                      f"max {awards_per_pmid.max()}, PMIDs citing >1 award: {(awards_per_pmid>1).sum():,}")
            print(f"  PMIDs per award  -- median {pmids_per_core.median():.0f}, "
                      f"max {pmids_per_core.max()}, awards cited by >1 corpus PMID: {(pmids_per_core>1).sum():,}")
            awards_per_pmid = (mn.dropna(subset=["core_project_num"])
                                     .groupby("pmid")["core_project_num"].nunique())
            clean_pmids = set(awards_per_pmid[awards_per_pmid == 1].index)
            print(f"1:1 PMID->award edges: {len(clean_pmids):,} PMIDs "
                      f"({len(clean_pmids)/max(mn['pmid'].nunique(),
                              1):.1%} of linked PMIDs) used for dollar attribution.")
            rc_all = (mn.dropna(subset=["root_accession", "core_project_num"])
                            .drop_duplicates(["root_accession", "core_project_num"])).copy()
            for cls in ["grant", "contract", "intramural"]:
                    rc_all[f"is_{cls}"] = rc_all["funding_class"].eq(cls).astype(int)
            counts = rc_all.groupby("root_accession", as_index=False).agg(
                    n_linked_cores=("core_project_num", "nunique"),
                    n_grant_cores=("is_grant", "sum"),
                    n_contract_cores=("is_contract", "sum"),
                    n_intramural_cores=("is_intramural", "sum"),
                )
            admin_ics = (rc_all.dropna(subset=["admin_ic"]).groupby("root_accession")["admin_ic"]
                             .agg(lambda s: json.dumps(sorted(set(s)))).rename("admin_ics").reset_index())
            rc_11 = (mn[mn["pmid"].isin(clean_pmids)]
                           .dropna(subset=["root_accession", "core_project_num"])
                           .drop_duplicates(["root_accession", "core_project_num"])).copy()
            rc_11["is_priceable"] = rc_11["priceable"].fillna(False).astype(bool)
            rc_11["priceable_award"] = np.where(rc_11["is_priceable"],
                                                    pd.to_numeric(rc_11["current_period_award"],
                                                        errors="coerce"), 0.0)
            dollars = rc_11.groupby("root_accession", as_index=False).agg(
                    n_priceable_cores_1to1=("is_priceable", "sum"),
                    current_period_priceable_award_1to1=("priceable_award", "sum"),
                )
            rollup = (counts.merge(admin_ics, on="root_accession", how="left")
                                .merge(dollars, on="root_accession", how="left"))
            rollup["admin_ics"] = rollup["admin_ics"].fillna("[]")
            rollup["n_priceable_cores_1to1"] = rollup["n_priceable_cores_1to1"].fillna(0).astype(int)
            rollup["current_period_priceable_award_1to1"] = rollup["current_period_priceable_award_1to1"].fillna(0.0)
            rollup["n_priceable_cores"] = rollup["n_priceable_cores_1to1"]
            rollup["current_period_priceable_award"] = rollup["current_period_priceable_award_1to1"]
            rollup["share_cores_structurally_unpriced"] = (
                    (rollup["n_contract_cores"] + rollup["n_intramural_cores"])
                    / rollup["n_linked_cores"].replace(0, np.nan))
            rollup["has_unpriced_funding"] = (rollup["n_contract_cores"] + rollup["n_intramural_cores"]) > 0
            assert rollup["root_accession"].is_unique, "Funding rollup must have one row per root."
            assert not (rollup["n_priceable_cores_1to1"] > rollup["n_linked_cores"]).any()
            rollup.to_parquet(PATHS["reporter_study_rollup"], index=False)
            joined = pd.read_parquet(PATHS["joined"])
            cols = [c for c in rollup.columns if c != "root_accession"]
            joined = joined.drop(columns=[c for c in cols if c in joined.columns], errors="ignore").merge(
                    rollup, on="root_accession", how="left")
            for c in ["n_linked_cores", "n_grant_cores", "n_contract_cores", "n_intramural_cores",
                          "n_priceable_cores", "current_period_priceable_award"]:
                    joined[c] = joined[c].fillna(0)
            joined["has_unpriced_funding"] = joined["has_unpriced_funding"].fillna(False).astype(bool)
            assert not joined["root_accession"].duplicated().any()
            joined.to_parquet(PATHS["joined"], index=False)
            vr = []
            def v(check, status, observed, detail=None):
                    vr.append({"check": check, "status": status, "observed": str(observed),
                               "detail": None if detail is None else str(detail),
                               "validated_utc": datetime.now(timezone.utc).isoformat()})
                    print(f"[{status:4}] {check}: {observed}" + (f" -- {detail}" if detail else ""))
            v("corpus_pmids", "INFO", n_corpus_pmids)
            v("pmids_linked_to_core", "INFO", pub_links["pmid"].nunique(),
                  f"{pub_links['pmid'].nunique()/max(n_corpus_pmids,1):.1%}")
            v("distinct_cores_linked", "INFO", pub_links["core_project_num"].nunique())
            cls = core_funding["funding_class"].value_counts().to_dict()
            v("core_funding_class_mix", "INFO", json.dumps(cls))
            contract_priced = core_funding.loc[core_funding.funding_class.eq("contract"),
                "current_period_award"].notna().mean()
            v("contract_priced_share", "PASS" if pd.isna(contract_priced) or contract_priced <= 0.8 else "WARN",
                  round(float(contract_priced), 3) if pd.notna(contract_priced) else "n/a")
            med_yrs = core_funding.loc[core_funding.funding_class.eq("grant"), "current_period_years"].median()
            v("grant_current_period_years_median", "PASS" if 1 <= (med_yrs or 0) <= 10 else "WARN", med_yrs,
                  "sanity: R-series current period typically ~3-5 FYs")
            v("no_negative_awards", "PASS" if int((rollup["current_period_priceable_award"] < 0).sum()) == 0 else "FAIL",
                  int((rollup["current_period_priceable_award"] < 0).sum()))
            v("joined_unique_root", "PASS" if int(joined["root_accession"].duplicated().sum()) == 0 else "FAIL",
                  int(joined["root_accession"].duplicated().sum()))
            pd.DataFrame(vr).to_parquet(PATHS["reporter_verification"], index=False)
            if os.path.exists(PATHS["icite_study_rollup"]):
                    # Distinct priceable cores reached via 1:1 edges, each priced once.
                    cores_11 = (rc_11[rc_11["is_priceable"]]
                                .drop_duplicates("core_project_num")[["core_project_num", "priceable_award"]])
                    total_award_m = cores_11["priceable_award"].sum() / 1e6

                    # Impact of the 1:1 papers themselves (per-paper, no study rollup), so the
                    # numerator and denominator share the same 1:1 edge set.
                    icite = pd.read_parquet(PATHS["icite_metrics"])
                    icite = icite.loc[icite["icite_fetch_note"].eq("ok")].copy()
                    icite["pmid"] = icite["pmid"].astype(str)
                    icite["rcr"] = pd.to_numeric(icite.get("rcr"), errors="coerce")
                    icite["n_clinical_citations"] = pd.to_numeric(icite.get("n_clinical_citations"),
                        errors="coerce")
                    pmids_11 = set(mn.loc[mn["pmid"].isin(clean_pmids) &
                                          mn["priceable"].fillna(False), "pmid"])
                    imp = icite[icite["pmid"].isin(pmids_11)]
                    sum_rcr_11 = imp["rcr"].sum(min_count=1)
                    clin_11 = imp["n_clinical_citations"].sum(min_count=1)

                    print("\nHEADLINE (award-centric; 1:1 PMID->award; current period; "
                          "each priceable grant counted once corpus-wide):")
                    print(f"  Distinct priceable grant cores (1:1):      {len(cores_11):,}")
                    print(f"  Current-period award (summed once each):   ${total_award_m:,.0f}M")
                    print(f"  1:1 papers on those grants (RCR-scored):   {imp['rcr'].notna().sum():,}")
                    if total_award_m > 0:
                        print(f"  Field-normalized impact (sum RCR) per $1M: {sum_rcr_11/total_award_m:.2f}")
                        print(f"  Clinical-article citations per $1M:        {clin_11/total_award_m:.2f}")
                    else:
                        print("  Per-dollar summaries undefined: no positive priceable award denominator.")
                    print("  NOTE: per-study award columns in `rollup` are for per-study display only;")
                    print("  do NOT sum them across studies (a shared award would be counted repeatedly).")
            print(f"\nM:N table     -> {PATHS['reporter_pmid_award_mn']}")
            print(f"Core funding  -> {PATHS['reporter_core_funding']}")
            print(f"Study rollup  -> {PATHS['reporter_study_rollup']}")
            display(rollup.sort_values("current_period_priceable_award", ascending=False).head(15))
        # Validate the completed staging set before publishing any final product.
        completed_joined = pd.read_parquet(PATHS["joined"])
        assert completed_joined.root_accession.is_unique
        assert set(completed_joined.root_accession) == set(root_table.root_accession)
        for key in output_keys:
            require_files(key)
        for key, final in original_paths.items():
            atomic_table(pd.read_parquet(PATHS[key]), final)
        save_json(str(staging / "_complete.json"), {"run_id":RUN_ID, "outputs":original_paths})
    finally:
        PATHS.update(original_paths)
    # Staging products are retained for interrupted-run inspection; optional cleanup is at the end.


def run_join_stage():
    pub = read_frame("reporter_pub_links_full", ["pmid", "core_project_num"])
    prj = read_frame("reporter_projects_full", ["core_project_num"])
    include = ENABLE_REPORTER and not pub.empty and not prj.empty and pub.core_project_num.notna().any()
    outputs = BASE_JOIN_OUTPUTS + (FUNDING_OUTPUTS if include else [])
    rebuilt = run_derived("joins", JOIN_INPUTS, outputs, lambda: build_join_products(include),
                          extra={"funding_included":bool(include)})
    if not include:
        print("Funding inputs are absent/empty or disabled. Funding artifacts are not recomputed; any existing standalone funding files remain historical.")
    result = read_frame("joined")
    brief_stats("Final joined table", result, ["root_accession"])
    return result

joined = run_join_stage()
# Retain familiar in-memory names for downstream notebook/deck compatibility.
study_counts_clean = read_frame("study_counts_analysis_ready")
study_page_clean = read_frame("study_page_metadata_analysis_ready")
sstr_clean = read_frame("sstr_summary_analysis_ready")
pubmed_links = read_frame("pubmed_links")
ct_links = read_frame("clinical_trials_direct")


## 16. Accession timing joins, checks and optional figures
Adds your FTP/DAR workflow using copies of input frames. It preserves the explicit proxy interpretation and readiness thresholds. Raw/DAR chronology works without dedup_access; family-normalized fields remain missing and figures 30/31 are withheld unless that independently supplied input is available. FTP timestamps must not be substituted for actual release dates in year-three storage analyses.


In [ ]:
# @title 16. Accession timing joins, checks and optional figures
def timing_input_frames():
    roots = read_frame("root_accessions")[["root_accession"]].copy()
    roots["root_accession"] = roots["root_accession"].astype("string").str.strip().str.lower()
    roots = roots.dropna(subset=["root_accession"]).loc[lambda f: f.root_accession.str.match(r"^phs\d+$",
            na=False)].drop_duplicates().sort_values("root_accession").reset_index(drop=True)
    raw = read_frame("dar_records").copy()
    if "root_accession" not in raw:
        require_columns(raw, ["queried_accession"], "DAR timing input")
        raw["root_accession"] = raw.queried_accession.astype("string").str.extract(r"^(phs\d+)", expand=False)
    require_columns(raw, ["root_accession", "wlid", "initial_approval_date"], "DAR timing input")
    raw = raw.loc[raw.root_accession.isin(roots.root_accession)].copy()
    dedup = None
    if os.path.isfile(DEDUP_ACCESS_PATH):
        dedup = pd.read_parquet(DEDUP_ACCESS_PATH)
    elif isinstance(globals().get("dedup_access"), pd.DataFrame):
        dedup = globals()["dedup_access"].copy()
    if dedup is not None:
        require_columns(dedup, ["root_accession", "wlid", "initial_approval_date"],
            "Supplied family-deduplicated timing")
        dedup = dedup.loc[dedup.root_accession.isin(roots.root_accession)].copy()
        available = True
    else:
        dedup = pd.DataFrame(columns=["root_accession", "wlid", "initial_approval_date"])
        available = False
        print("No supplied dedup_access table: raw DAR/FTP chronology will run; family counts/rates are withheld, not filled with zero.")
    return roots, raw, dedup, available


def savefig(filename, dpi=300):
    if MAKE_TIMING_FIGURES:
        plt.savefig(os.path.join(TIMING_FIGURE_DIR, filename), dpi=dpi, bbox_inches="tight", facecolor="white")


def build_accession_timing():
    root_universe, dar, dedup_access_local, has_dedup_access = timing_input_frames()
    dedup_access = dedup_access_local
    canonical_root_count = len(root_universe)
    if canonical_root_count == 0:
        raise ValueError("Cannot compute timing without the canonical root universe")
    ftp_root_results = normalize_ftp_results(read_frame("ftp_root_results", list(empty_ftp_roots().columns)))
    ftp_root_results = ftp_root_results.loc[ftp_root_results.root_accession.isin(root_universe.root_accession)].copy()
    dar_time = dar.copy().assign(root_accession=lambda frame: frame['root_accession'].astype('string').str.strip().str.lower(),
        wlid=lambda frame: frame['wlid'].astype('string').str.strip(),
        approval_datetime=lambda frame: pd.to_datetime(frame['initial_approval_date'], errors='coerce',
            utc=True)).loc[lambda frame: frame['root_accession'].str.match('^phs\\d+$', na=False)].copy()
    dedup_dar_time = dedup_access.copy().assign(root_accession=lambda frame: frame['root_accession'].astype('string').str.strip().str.lower(),
        wlid=lambda frame: frame['wlid'].astype('string').str.strip(),
        approval_datetime=lambda frame: pd.to_datetime(frame['initial_approval_date'], errors='coerce',
            utc=True)).loc[lambda frame: frame['root_accession'].str.match('^phs\\d+$', na=False)].copy()
    first_observed_dar_by_root = dar_time.dropna(subset=['root_accession',
            'approval_datetime']).groupby('root_accession',
        as_index=False).agg(first_observed_dar_datetime=('approval_datetime', 'min'), raw_dar_rows=('wlid', 'size'),
        unique_requester_wlids=('wlid', 'nunique'))
    first_observed_dar_by_root['first_observed_dar_year'] = first_observed_dar_by_root['first_observed_dar_datetime'].dt.year
    print('Roots with a dated first observed DAR:', f'{len(first_observed_dar_by_root):,}')
    ftp_collected_root_count = ftp_root_results['root_accession'].nunique()
    ftp_preferred_root_count = int(ftp_root_results['first_preferred_public_versioned_ftp_timestamp'].notna().sum())
    ftp_sensitivity_only_root_count = int(ftp_root_results['primary_missing_sensitivity_present'].sum())
    ftp_no_current_directory_count = int(ftp_root_results['collection_status'].eq('no_current_public_ftp_root_directory').sum())
    ftp_true_failure_count = int(ftp_root_results['requires_ftp_retry'].fillna(False).astype(bool).sum())
    ftp_directory_present_no_artifact_count = int((ftp_root_results['has_current_public_ftp_root_directory'].fillna(False).astype(bool) & ftp_root_results['first_preferred_public_versioned_ftp_timestamp'].isna() & ~ftp_root_results['primary_missing_sensitivity_present'].fillna(False).astype(bool)).sum())
    ftp_collection_coverage_corpus = ftp_collected_root_count / canonical_root_count
    ftp_preferred_coverage_corpus = ftp_preferred_root_count / canonical_root_count
    ftp_preferred_yield_processed = ftp_preferred_root_count / ftp_collected_root_count if ftp_collected_root_count > 0 else np.nan
    ftp_collection_status_summary = ftp_root_results.groupby('collection_status',
        dropna=False).size().rename('root_accessions').reset_index().sort_values('root_accessions', ascending=False)
    ftp_collection_status_summary['share_of_processed_roots'] = ftp_collection_status_summary['root_accessions'] / ftp_collected_root_count
    ftp_access_audit_summary = pd.DataFrame([{'category': 'Preferred study-specific public FTP artifact found',
                'roots': ftp_preferred_root_count}, {'category': 'Sensitivity-only artifact; no preferred artifact',
                'roots': ftp_sensitivity_only_root_count}, {'category': 'No current public FTP root directory',
                'roots': ftp_no_current_directory_count},
            {'category': 'Current FTP root present but no preferred artifact',
                'roots': ftp_directory_present_no_artifact_count},
            {'category': 'True FTP collector failure requiring retry', 'roots': ftp_true_failure_count}])
    ftp_access_audit_summary['share_of_processed_roots'] = ftp_access_audit_summary['roots'] / ftp_collected_root_count
    print('\nFTP collection coverage:')
    print(f'  Processed roots: {ftp_collected_root_count:,}/{canonical_root_count:,} ({ftp_collection_coverage_corpus:.2%} of corpus)')
    print(f'  Preferred artifacts: {ftp_preferred_root_count:,}/{canonical_root_count:,} ({ftp_preferred_coverage_corpus:.2%} of corpus)')
    print(f'  Preferred-artifact yield among processed roots: {ftp_preferred_yield_processed:.2%}')
    print(f'  No current public FTP root directory: {ftp_no_current_directory_count:,}')
    print(f'  True collector failures requiring retry: {ftp_true_failure_count:,}')
    display(ftp_collection_status_summary)
    display(ftp_access_audit_summary)
    ftp_dar_validation = root_universe.merge(ftp_root_results[['root_accession', 'collection_status',
                'collection_error', 'has_current_public_ftp_root_directory', 'requires_ftp_retry',
                'n_version_folders', 'selected_version_folder', 'used_later_version_fallback', 'n_shallow_objects',
                'n_files_with_dates', 'collection_attempts', 'first_preferred_public_versioned_ftp_timestamp',
                'first_preferred_public_versioned_ftp_year', 'first_preferred_public_versioned_ftp_object_name',
                'first_preferred_public_versioned_ftp_object_path', 'first_preferred_public_versioned_ftp_category',
                'first_any_substantive_public_versioned_ftp_timestamp',
                'first_any_substantive_public_versioned_ftp_object_name',
                'first_any_substantive_public_versioned_ftp_object_path',
                'first_any_substantive_public_versioned_ftp_category', 'primary_missing_sensitivity_present',
                'manual_review_low_phs_late_ftp', 'phs_numeric_id']], on='root_accession', how='left',
        validate='one_to_one').merge(first_observed_dar_by_root, on='root_accession', how='left',
        validate='one_to_one')
    ftp_dar_validation['preferred_ftp_minus_first_observed_dar_days'] = (ftp_dar_validation['first_preferred_public_versioned_ftp_timestamp'] - ftp_dar_validation['first_observed_dar_datetime']).dt.total_seconds() / 86400
    ftp_dar_validation['timing_sequence_flag'] = np.select([bool_condition(ftp_dar_validation['collection_status'].isna()),
            bool_condition(ftp_dar_validation['collection_status'].eq('no_current_public_ftp_root_directory')),
            bool_condition(ftp_dar_validation['requires_ftp_retry']),
            bool_condition(ftp_dar_validation['primary_missing_sensitivity_present']),
            bool_condition(ftp_dar_validation['first_preferred_public_versioned_ftp_timestamp'].isna()),
            bool_condition(ftp_dar_validation['first_observed_dar_datetime'].isna()),
            bool_condition(ftp_dar_validation['preferred_ftp_minus_first_observed_dar_days'].lt(-365)),
            bool_condition(ftp_dar_validation['preferred_ftp_minus_first_observed_dar_days'].between(-365, -1,
                inclusive='both')),
            bool_condition(ftp_dar_validation['preferred_ftp_minus_first_observed_dar_days'].between(0, 365,
                inclusive='both')),
            bool_condition(ftp_dar_validation['preferred_ftp_minus_first_observed_dar_days'].gt(365))],
        ['FTP not collected in current run', 'No current public FTP root directory',
            'FTP collector failure requiring retry', 'Sensitivity-only FTP artifact; no preferred artifact',
            'No preferred FTP artifact', 'No dated observed DAR',
            'Preferred FTP artifact >1 year before first observed DAR',
            'Preferred FTP artifact ≤1 year before first observed DAR',
            'Preferred FTP artifact 0–1 year after first observed DAR',
            'Preferred FTP artifact >1 year after first observed DAR'], default='Unclassified')
    ftp_validation_summary = ftp_dar_validation.loc[ftp_dar_validation['collection_status'].notna()].groupby('timing_sequence_flag',
        dropna=False).size().rename('root_accessions').reset_index().sort_values('root_accessions', ascending=False)
    ftp_validation_summary['share_of_collected_roots'] = ftp_validation_summary['root_accessions'] / ftp_collected_root_count
    display(ftp_validation_summary)
    ftp_public_directory_absences = ftp_dar_validation.loc[ftp_dar_validation['collection_status'].eq('no_current_public_ftp_root_directory')].sort_values('root_accession').copy()
    true_ftp_collector_failures = ftp_dar_validation.loc[ftp_dar_validation['requires_ftp_retry'].fillna(False).astype(bool)].sort_values('root_accession').copy()
    ftp_no_preferred_artifact_with_directory = ftp_dar_validation.loc[ftp_dar_validation['has_current_public_ftp_root_directory'].fillna(False).astype(bool) & ftp_dar_validation['first_preferred_public_versioned_ftp_timestamp'].isna() & ~ftp_dar_validation['primary_missing_sensitivity_present'].fillna(False).astype(bool)].sort_values('root_accession').copy()
    manual_review_mask = bool_condition(ftp_dar_validation['manual_review_low_phs_late_ftp']) | bool_condition(ftp_dar_validation['primary_missing_sensitivity_present']) | bool_condition(ftp_dar_validation['timing_sequence_flag'].eq('Preferred FTP artifact >1 year after first observed DAR')) | bool_condition(ftp_dar_validation['requires_ftp_retry'])
    manual_review_ftp_cases = ftp_dar_validation.loc[manual_review_mask].sort_values(['preferred_ftp_minus_first_observed_dar_days',
            'phs_numeric_id'], ascending=[False, True], na_position='last')
    print('\nNo current public FTP root directory:', f'{len(ftp_public_directory_absences):,}')
    print('True collector failures requiring retry:', f'{len(true_ftp_collector_failures):,}')
    print('Current FTP root present but no preferred artifact:',
        f'{len(ftp_no_preferred_artifact_with_directory):,}')
    print('Manual chronology-review cases:', f'{len(manual_review_ftp_cases):,}')
    if not ftp_public_directory_absences.empty:
        display(ftp_public_directory_absences[['root_accession', 'collection_status', 'collection_error',
                    'first_observed_dar_datetime']])
    if not true_ftp_collector_failures.empty:
        display(true_ftp_collector_failures[['root_accession', 'collection_status', 'collection_error',
                    'collection_attempts']])
    if not ftp_no_preferred_artifact_with_directory.empty:
        display(ftp_no_preferred_artifact_with_directory[['root_accession', 'collection_status',
                    'selected_version_folder', 'n_version_folders', 'n_shallow_objects', 'n_files_with_dates',
                    'first_any_substantive_public_versioned_ftp_timestamp',
                    'first_any_substantive_public_versioned_ftp_category']])
    display(manual_review_ftp_cases[['root_accession', 'phs_numeric_id', 'collection_status',
                'selected_version_folder', 'used_later_version_fallback',
                'first_preferred_public_versioned_ftp_timestamp', 'first_preferred_public_versioned_ftp_category',
                'first_preferred_public_versioned_ftp_object_name',
                'first_any_substantive_public_versioned_ftp_timestamp',
                'first_any_substantive_public_versioned_ftp_category',
                'first_any_substantive_public_versioned_ftp_object_name', 'first_observed_dar_datetime',
                'preferred_ftp_minus_first_observed_dar_days', 'timing_sequence_flag']])
    corrected_coverage_summary = pd.DataFrame([{'measure': 'Canonical root accessions',
                'numerator': canonical_root_count, 'denominator': canonical_root_count,
                'denominator_label': 'canonical root accessions', 'share_or_rate': 1.0, 'unit': 'root accessions'},
            {'measure': 'Roots processed by FTP collector', 'numerator': ftp_collected_root_count,
                'denominator': canonical_root_count, 'denominator_label': 'canonical root accessions',
                'share_or_rate': ftp_collection_coverage_corpus, 'unit': 'root accessions'},
            {'measure': 'Preferred FTP artifacts among all roots', 'numerator': ftp_preferred_root_count,
                'denominator': canonical_root_count, 'denominator_label': 'canonical root accessions',
                'share_or_rate': ftp_preferred_coverage_corpus, 'unit': 'root accessions'},
            {'measure': 'Preferred FTP artifact yield among processed roots', 'numerator': ftp_preferred_root_count,
                'denominator': ftp_collected_root_count, 'denominator_label': 'FTP-collected root accessions',
                'share_or_rate': ftp_preferred_yield_processed, 'unit': 'root accessions'},
            {'measure': 'No current public FTP root directory among processed roots',
                'numerator': ftp_no_current_directory_count, 'denominator': ftp_collected_root_count,
                'denominator_label': 'FTP-collected root accessions',
                'share_or_rate': ftp_no_current_directory_count / ftp_collected_root_count,
                'unit': 'root accessions'}, {'measure': 'True FTP collector failures among processed roots',
                'numerator': ftp_true_failure_count, 'denominator': ftp_collected_root_count,
                'denominator_label': 'FTP-collected root accessions',
                'share_or_rate': ftp_true_failure_count / ftp_collected_root_count, 'unit': 'root accessions'},
            {'measure': 'Roots with dated first observed DAR', 'numerator': len(first_observed_dar_by_root),
                'denominator': canonical_root_count, 'denominator_label': 'canonical root accessions',
                'share_or_rate': len(first_observed_dar_by_root) / canonical_root_count, 'unit': 'root accessions'},
            {'measure': 'Raw DAR rows with usable approval date',
                'numerator': int(dar_time['approval_datetime'].notna().sum()), 'denominator': len(dar_time),
                'denominator_label': 'raw DAR authorization rows',
                'share_or_rate': dar_time['approval_datetime'].notna().mean(), 'unit': 'DAR authorization rows'},
            {'measure': 'Deduplicated DAR authorizations with usable approval date',
                'numerator': int(dedup_dar_time['approval_datetime'].notna().sum()),
                'denominator': len(dedup_dar_time), 'denominator_label': 'family–requester authorization records',
                'share_or_rate': dedup_dar_time['approval_datetime'].notna().mean(),
                'unit': 'family–requester authorizations'}])
    corrected_coverage_summary['share_or_rate_percent'] = 100 * corrected_coverage_summary['share_or_rate']
    display(corrected_coverage_summary)
    is_corpus_ready = FTP_RUN_MODE == 'full' and has_dedup_access and (ftp_collection_coverage_corpus >= MIN_FTP_COLLECTION_COVERAGE) and (ftp_preferred_coverage_corpus >= MIN_PREFERRED_ARTIFACT_COVERAGE)
    deck_readiness = pd.DataFrame([{'output': 'First observed DAR onset (Figure 32)', 'status': 'DECK-READY',
                'reason': 'Uses full assembled DAR extract and is explicitly defined as first observed initial approval.'},
            {'output': 'Requester concentration / requester profile figures', 'status': 'NOT ASSESSED',
                'reason': 'Requester-profile readiness is not assessed by this timing block.'},
            {'output': 'Current public FTP artifact timeline (Figure 30)',
                'status': 'DECK-READY' if is_corpus_ready else 'HOLD',
                'reason': 'Full collection and artifact-coverage thresholds passed; supplied family timing available.' if is_corpus_ready else 'Current FTP data are incomplete, audit-only, or lack supplied family timing and cannot represent corpus-wide artifact chronology.'},
            {'output': 'Matched FTP-normalized DAR rate (Figure 31)',
                'status': 'DECK-READY' if is_corpus_ready else 'HOLD',
                'reason': 'Matched root universe is complete enough for descriptive normalization.' if is_corpus_ready else 'Matched numerator/denominator coverage is not sufficient.'}])
    display(deck_readiness)
    if not is_corpus_ready:
        print('\nFTP-derived corpus trends are NOT deck-ready.')
        print(f'  FTP collection coverage: {ftp_collection_coverage_corpus:.2%} (threshold: {MIN_FTP_COLLECTION_COVERAGE:.0%})')
        print(f'  Preferred-artifact coverage: {ftp_preferred_coverage_corpus:.2%} (threshold: {MIN_PREFERRED_ARTIFACT_COVERAGE:.0%})')
        print('  Use Figure 32 and requester analyses now. Hold FTP annual/cumulative trends and FTP-normalized DAR rates.')
    annual_raw_dar = dar_time.dropna(subset=['approval_datetime']).assign(year=lambda frame: frame['approval_datetime'].dt.year).loc[lambda frame: frame['year'].between(MIN_VALID_YEAR,
            MAX_VALID_YEAR)].groupby('year').size().rename('raw_accession_requester_rows')
    annual_dedup_dar = dedup_dar_time.dropna(subset=['approval_datetime']).assign(year=lambda frame: frame['approval_datetime'].dt.year).loc[lambda frame: frame['year'].between(MIN_VALID_YEAR,
            MAX_VALID_YEAR)].groupby('year').size().rename('deduplicated_family_requester_authorizations')
    _observed_years = sorted(set(annual_raw_dar.index) | set(annual_dedup_dar.index))
    dar_timeline_years = range(int(min(_observed_years)), int(max(_observed_years)) + 1) if _observed_years else []
    annual_dar_timeline = pd.DataFrame(index=pd.Index(dar_timeline_years, name='year'))
    annual_dar_timeline = annual_dar_timeline.join(annual_raw_dar, how='left').join(annual_dedup_dar,
        how='left').fillna(0)
    for column in annual_dar_timeline.columns:
        annual_dar_timeline[column] = annual_dar_timeline[column].astype(int)
    annual_dar_timeline['cumulative_raw_accession_requester_rows'] = annual_dar_timeline['raw_accession_requester_rows'].cumsum()
    annual_dar_timeline['cumulative_deduplicated_family_requester_authorizations'] = annual_dar_timeline['deduplicated_family_requester_authorizations'].cumsum()
    if not has_dedup_access:
        annual_dar_timeline['deduplicated_family_requester_authorizations'] = pd.Series(pd.NA,
            index=annual_dar_timeline.index, dtype='Int64')
        annual_dar_timeline['cumulative_deduplicated_family_requester_authorizations'] = pd.Series(pd.NA,
            index=annual_dar_timeline.index, dtype='Int64')
    annual_first_observed_dar = first_observed_dar_by_root.dropna(subset=['first_observed_dar_year']).loc[lambda frame: frame['first_observed_dar_year'].between(MIN_VALID_YEAR,
            MAX_VALID_YEAR)].groupby('first_observed_dar_year').size().rename('roots_with_first_observed_dar')
    if MAKE_TIMING_FIGURES and (not annual_first_observed_dar.empty):
        annual_first_observed_dar = annual_first_observed_dar.reindex(range(int(annual_first_observed_dar.index.min()),
                int(annual_first_observed_dar.index.max()) + 1), fill_value=0)
        annual_first_observed_dar.index.name = 'year'
        cumulative_first_observed_dar = annual_first_observed_dar.cumsum()
        fig, axes = plt.subplots(1, 2, figsize=(14, 5.1))
        axes[0].bar(annual_first_observed_dar.index, annual_first_observed_dar.values, color=COLORS['blue'],
            width=0.85)
        axes[0].set_title('Roots with first observed DAR approval by year', loc='left', fontweight='bold')
        axes[0].set_xlabel('Initial-approval year')
        axes[0].set_ylabel('Root accessions')
        axes[0].grid(axis='y', alpha=0.2)
        axes[1].plot(cumulative_first_observed_dar.index, cumulative_first_observed_dar.values,
            color=COLORS['blue'], linewidth=2.5)
        axes[1].set_title('Cumulative roots with an observed DAR approval', loc='left', fontweight='bold')
        axes[1].set_xlabel('Initial-approval year')
        axes[1].set_ylabel('Cumulative root accessions')
        axes[1].grid(axis='y', alpha=0.2)
        fig.suptitle('Observed onset of recorded dbGaP access activity', x=0.01, ha='left', fontsize=15,
            fontweight='bold')
        fig.text(0.01, -0.03,
            'Date is the earliest initial-approval timestamp observed in the assembled DAR extract for each root accession. It is not an accession-creation, public-release, or necessarily true first-access date.',
            ha='left', fontsize=8.8, color=COLORS['slate'])
        savefig('32_roots_first_observed_dar_approval.png')
        plt.show()
    matched_ftp_dar_timeline = pd.DataFrame()
    if is_corpus_ready:
        preferred_ftp_dates = ftp_root_results[['root_accession',
                'first_preferred_public_versioned_ftp_timestamp']].dropna(subset=['first_preferred_public_versioned_ftp_timestamp']).assign(ftp_year=lambda frame: frame['first_preferred_public_versioned_ftp_timestamp'].dt.year).loc[lambda frame: frame['ftp_year'].between(MIN_VALID_YEAR,
                MAX_VALID_YEAR)]
        preferred_ftp_roots = set(preferred_ftp_dates['root_accession'].astype(str).tolist())
        annual_preferred_ftp = preferred_ftp_dates.groupby('ftp_year').size().rename('roots_first_surviving_substantive_public_ftp_artifact')
        dedup_matched = dedup_dar_time.loc[dedup_dar_time['root_accession'].isin(preferred_ftp_roots)].dropna(subset=['approval_datetime']).assign(dar_year=lambda frame: frame['approval_datetime'].dt.year).loc[lambda frame: frame['dar_year'].between(MIN_VALID_YEAR,
                MAX_VALID_YEAR)]
        annual_dedup_matched = dedup_matched.groupby('dar_year').size().rename('deduplicated_family_requester_authorizations')
        _matched_years = sorted(set(annual_preferred_ftp.index) | set(annual_dedup_matched.index))
        matched_years = range(int(min(_matched_years)), int(max(_matched_years)) + 1) if _matched_years else []
        matched_ftp_dar_timeline = pd.DataFrame(index=pd.Index(matched_years, name='year'))
        matched_ftp_dar_timeline = matched_ftp_dar_timeline.join(annual_preferred_ftp,
            how='left').join(annual_dedup_matched, how='left').fillna(0)
        for column in matched_ftp_dar_timeline.columns:
            matched_ftp_dar_timeline[column] = matched_ftp_dar_timeline[column].astype(int)
        matched_ftp_dar_timeline['cumulative_preferred_ftp_roots'] = matched_ftp_dar_timeline['roots_first_surviving_substantive_public_ftp_artifact'].cumsum()
        matched_ftp_dar_timeline['cumulative_deduplicated_authorizations'] = matched_ftp_dar_timeline['deduplicated_family_requester_authorizations'].cumsum()
        matched_ftp_dar_timeline['annual_deduplicated_authorizations_per_100_cumulative_roots'] = 100 * matched_ftp_dar_timeline['deduplicated_family_requester_authorizations'] / matched_ftp_dar_timeline['cumulative_preferred_ftp_roots'].replace(0,
            np.nan)
        matched_ftp_dar_timeline['cumulative_deduplicated_authorizations_per_100_cumulative_roots'] = 100 * matched_ftp_dar_timeline['cumulative_deduplicated_authorizations'] / matched_ftp_dar_timeline['cumulative_preferred_ftp_roots'].replace(0,
            np.nan)
    if is_corpus_ready and MAKE_TIMING_FIGURES:
        fig, axes = plt.subplots(2, 2, figsize=(16, 10.5))
        axes[0, 0].bar(matched_ftp_dar_timeline.index,
            matched_ftp_dar_timeline['roots_first_surviving_substantive_public_ftp_artifact'], color=COLORS['blue'],
            width=0.85)
        axes[0, 0].set_title('Roots by earliest surviving substantive public FTP artifact', loc='left',
            fontweight='bold')
        axes[0, 0].set_xlabel('Artifact year')
        axes[0, 0].set_ylabel('Root accessions')
        axes[0, 0].grid(axis='y', alpha=0.2)
        axes[0, 1].plot(matched_ftp_dar_timeline.index,
            matched_ftp_dar_timeline['deduplicated_family_requester_authorizations'], color=COLORS['amber'],
            linewidth=2.4)
        axes[0, 1].set_title('Observed DAR activity among matched FTP roots', loc='left', fontweight='bold')
        axes[0, 1].set_xlabel('Approval year')
        axes[0, 1].set_ylabel('Family–requester authorizations')
        axes[0, 1].grid(axis='y', alpha=0.2)
        axes[1, 0].plot(matched_ftp_dar_timeline.index, matched_ftp_dar_timeline['cumulative_preferred_ftp_roots'],
            color=COLORS['blue'], linewidth=2.5)
        axes[1, 0].set_title('Cumulative roots with a surviving public FTP artifact', loc='left', fontweight='bold')
        axes[1, 0].set_xlabel('Artifact year')
        axes[1, 0].set_ylabel('Cumulative root accessions')
        axes[1, 0].grid(axis='y', alpha=0.2)
        axes[1, 1].plot(matched_ftp_dar_timeline.index,
            matched_ftp_dar_timeline['cumulative_deduplicated_authorizations'], color=COLORS['amber'],
            linewidth=2.5)
        axes[1, 1].set_title('Cumulative observed DAR activity among matched FTP roots', loc='left',
            fontweight='bold')
        axes[1, 1].set_xlabel('Approval year')
        axes[1, 1].set_ylabel('Cumulative family–requester authorizations')
        axes[1, 1].grid(axis='y', alpha=0.2)
        fig.suptitle('Surviving public dbGaP FTP artifacts and matched observed DAR activity over time', x=0.01,
            ha='left', fontsize=15.5, fontweight='bold')
        fig.text(0.01, -0.025,
            'FTP date is the earliest retained timestamp of a study-specific, non-template, non-interchange file in the current public FTP hierarchy. It is not accession assignment, submission, registration, or necessarily first release. DAR activity is restricted to roots in the same preferred-FTP artifact universe.',
            ha='left', fontsize=8.6, color=COLORS['slate'])
        savefig('30_surviving_public_ftp_artifacts_and_matched_dar_activity.png')
        plt.show()
        fig, ax = plt.subplots(figsize=(12, 5.2))
        ax.plot(matched_ftp_dar_timeline.index,
            matched_ftp_dar_timeline['annual_deduplicated_authorizations_per_100_cumulative_roots'],
            color=COLORS['amber'], linewidth=2.4, marker='o', markersize=3.2)
        ax.set_title('Observed family-deduplicated DAR authorizations per 100 cumulative matched FTP-artifact roots',
            loc='left', fontweight='bold')
        ax.set_xlabel('Year')
        ax.set_ylabel('Annual family–requester authorizations per 100 cumulative matched roots')
        ax.grid(axis='y', alpha=0.2)
        ax.text(0.02, 0.95,
            'Descriptive normalization only. FTP artifact roots differ in availability, study age, data scope, and access restrictions; this is not an incidence rate.',
            transform=ax.transAxes, ha='left', va='top', fontsize=8.4, color=COLORS['slate'])
        savefig('31_matched_dar_authorizations_per_100_ftp_artifact_roots.png')
        plt.show()
    next_steps = pd.DataFrame([{'priority': 1, 'action': 'Complete full checkpointed FTP collection',
                'why': 'Only a sufficiently complete collection can support a corpus-wide public-artifact chronology.',
                'output': 'Current FTP provenance for all canonical dbGaP roots.'}, {'priority': 2,
                'action': 'Treat FTP 550 responses as public-directory absences',
                'why': 'A missing current public FTP directory is a data-coverage state, not a transient collector failure.',
                'output': 'Explicit root-level current-public-FTP coverage table.'}, {'priority': 3,
                'action': 'Retry only genuine collector failures',
                'why': 'Timeouts, transport failures, and non-550 listing errors need technical remediation; directory absences do not.',
                'output': 'Small retry queue, distinct from absent-directory roots.'}, {'priority': 4,
                'action': 'Review artifact-after-DAR chronology cases',
                'why': 'Public FTP artifacts can postdate observed DAR approvals, indicating later packaging, migration, regenerated versions, or incomplete surviving public history.',
                'output': 'Methods-appendix taxonomy of chronology anomalies.'}, {'priority': 5,
                'action': 'Generate matched FTP/DAR figures only after readiness passes',
                'why': 'The DAR numerator and public-artifact denominator must refer to the same root-accession universe.',
                'output': 'Deck-safe Figures 30 and 31, or a documented decision to exclude FTP timing from the main deck.'},
            {'priority': 6, 'action': 'Retain accession minting as unresolved without internal data',
                'why': 'Public FTP timestamps do not identify internal accession assignment or study-submission events.',
                'output': 'Methods caveat distinguishing artifact chronology from actual accession minting.'}])
    display(next_steps)
    # Keep generated tables separate from raw source files. Raw FTP tables are never
    # rewritten by this local join/validation block.
    tables = {
        "ftp_dar_validation": ftp_dar_validation,
        "ftp_validation_summary": ftp_validation_summary,
        "ftp_collection_status_summary": ftp_collection_status_summary,
        "ftp_access_audit_summary": ftp_access_audit_summary,
        "ftp_public_directory_absences": ftp_public_directory_absences,
        "ftp_true_collector_failures": true_ftp_collector_failures,
        "ftp_no_preferred_artifact_with_directory": ftp_no_preferred_artifact_with_directory,
        "corrected_coverage_summary": corrected_coverage_summary,
        "manual_review_cases": manual_review_ftp_cases,
        "annual_dar_timeline": annual_dar_timeline.reset_index(),
        "first_observed_dar": first_observed_dar_by_root,
        "deck_readiness": deck_readiness,
        "next_steps": next_steps,
        "matched_ftp_dar_timeline": matched_ftp_dar_timeline.reset_index(),
    }
    if matched_ftp_dar_timeline.empty:
        # Preserve a stable empty schema; do not leave a previous run's populated rate table active.
        tables["matched_ftp_dar_timeline"] = pd.DataFrame(columns=["year",
            "roots_first_surviving_substantive_public_ftp_artifact", "deduplicated_family_requester_authorizations",
            "cumulative_preferred_ftp_roots", "cumulative_deduplicated_authorizations",
            "annual_deduplicated_authorizations_per_100_cumulative_roots",
            "cumulative_deduplicated_authorizations_per_100_cumulative_roots"])
    for name, frame in tables.items():
        atomic_table(frame, artifact_paths[name])
    if not is_corpus_ready:
        stale_figures = [os.path.join(TIMING_FIGURE_DIR, name) for name in TIMING_MATCHED_FIGURES]
        existing_stale = [p for p in stale_figures if os.path.isfile(p)]
        if existing_stale:
            print("HOLD: existing figures are not valid for this timing run; not deleted or replaced:",
                existing_stale)
            print("Use the optional end-of-notebook archive switch to quarantine these files.")
    print(f"Timing: {len(first_observed_dar_by_root):,} roots with dated DAR; {ftp_preferred_root_count:,} preferred FTP artifacts; matched figures ready={is_corpus_ready}.")

TIMING_MATCHED_FIGURES = ['30_surviving_public_ftp_artifacts_and_matched_dar_activity.png',
    '31_matched_dar_authorizations_per_100_ftp_artifact_roots.png']
TIMING_DERIVED_KEYS = ["timing_"+name for name in artifact_paths if name not in {"ftp_root_results",
        "ftp_shallow_objects"}]
if ENABLE_FTP_TIMING:
    inputs = ["root_accessions", "dar_records", "dar_checkpoint", "ftp_root_results", "ftp_shallow_objects",
        DEDUP_ACCESS_PATH]
    in_memory_dedup = globals().get("dedup_access")
    memory_fingerprint = None
    if isinstance(in_memory_dedup, pd.DataFrame) and not os.path.isfile(DEDUP_ACCESS_PATH):
        memory_fingerprint = hashlib.sha256(json.dumps(stable_value(in_memory_dedup.to_dict("records")),
                sort_keys=True).encode()).hexdigest()
    run_derived("timing", inputs, TIMING_DERIVED_KEYS, build_accession_timing,
        extra={"ftp_mode":FTP_RUN_MODE, "minimum_coverage":MIN_FTP_COLLECTION_COVERAGE,
               "minimum_preferred":MIN_PREFERRED_ARTIFACT_COVERAGE, "make_figures":MAKE_TIMING_FIGURES,
               "min_year":MIN_VALID_YEAR, "max_year":MAX_VALID_YEAR, "in_memory_dedup_sha256":memory_fingerprint})
    display(read_frame("timing_corrected_coverage_summary"))
    display(read_frame("timing_deck_readiness"))
else:
    print("Accession timing joins skipped because ENABLE_FTP_TIMING=False.")


## 17. iCite validation
Original verification-table fields and plausibility tests; a benchmark check is not a validation of data-sharing impact.


In [ ]:
# @title 17. iCite validation
def build_icite_validation():
    checks = []

    def safe(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        if isinstance(v, (list, tuple, set, dict)):
            return json.dumps(v, default=str)
        return str(v)

    def check(name, status, observed, expected=None, detail=None):
        row = {'check': str(name), 'status': str(status), 'observed': safe(observed), 'expected': safe(expected),
            'detail': safe(detail), 'validated_utc': datetime.now(timezone.utc).isoformat()}
        checks.append(row)
        print(f"[{status}] {name}: {row['observed']}")
    icite = pd.read_parquet(PATHS['icite_metrics'])
    icite.pmid = icite.pmid.astype(str)
    check('metrics_unique_pmid', 'PASS' if not icite.pmid.duplicated().any() else 'FAIL',
        int(icite.pmid.duplicated().sum()), 0)
    corpus = set(corpus_pmids())
    okset = set(icite.loc[icite.icite_fetch_note.eq('ok'), 'pmid'])
    nr = set(icite.loc[icite.icite_fetch_note.eq('not_returned'), 'pmid'])
    check('corpus_pmids', 'INFO', len(corpus))
    check('resolved_ok', 'INFO', len(okset), detail=f'{len(okset) / max(len(corpus),1):.1%}')
    check('not_returned_terminal', 'INFO', len(nr))
    check('unfetched', 'PASS' if not corpus - okset - nr else 'WARN', len(corpus - okset - nr), 0)
    x = icite[icite.icite_fetch_note.eq('ok')].copy()
    for col in ['rcr', 'nih_percentile', 'apt', 'citation_count']:
        x[col] = pd.to_numeric(x.get(col), errors='coerce')
    check('rcr_nonnegative', 'PASS' if not (x.rcr < 0).any() else 'FAIL', int((x.rcr < 0).sum()), 0)
    check('nih_percentile_0_100',
        'PASS' if not ((x.nih_percentile < 0) | (x.nih_percentile > 100)).any() else 'FAIL',
        int(((x.nih_percentile < 0) | (x.nih_percentile > 100)).sum()), 0)
    check('apt_0_1', 'PASS' if not ((x.apt < 0) | (x.apt > 1)).any() else 'FAIL',
        int(((x.apt < 0) | (x.apt > 1)).sum()), 0)
    share = x.rcr.notna().mean()
    check('rcr_defined_share', 'PASS' if 0.3 <= share <= 0.995 else 'WARN', f'{share:.1%}', '30%–99.5%',
        'RCR may be absent for recent or non-research publications. High coverage is plausible in a mature PubMed-indexed corpus; investigate only near-zero or near-universal coverage.')
    median = x.rcr.median()
    check('rcr_median_vs_NIH_benchmark', 'PASS' if pd.notna(median) and 0.5 <= median <= 4 else 'WARN',
        f'{median:.3f}' if pd.notna(median) else 'undefined', 'plausible 0.5–4.0; NIH reference ≈1.0')
    if os.path.exists(PATHS['pubmed_metadata']):
        pm = pd.read_parquet(PATHS['pubmed_metadata'])
        pm.pmid = pm.pmid.astype(str)
        pm['pub_year'] = pd.to_numeric(pm.pubmed_pub_date.astype(str).str.extract('(\\d{4})')[0], errors='coerce')
        z = x[['pmid', 'icite_year']].merge(pm[['pmid', 'pub_year']], on='pmid').dropna()
        z.icite_year = pd.to_numeric(z.icite_year, errors='coerce')
        rate = (z.icite_year - z.pub_year).abs().gt(1).mean() if len(z) else 0
        check('pmid_year_agreement', 'PASS' if rate <= 0.01 else 'WARN', f'{rate:.2%}', '≤1% >1-year discrepancy',
            f'n={len(z):,}')
    rp = pd.read_parquet(PATHS['icite_study_rollup'])
    check('rollup_rcr_subset_ok',
        'PASS' if not (rp.n_pmids_with_rcr.fillna(0) > rp.n_pmids_icite_ok.fillna(0)).any() else 'FAIL',
        int((rp.n_pmids_with_rcr.fillna(0) > rp.n_pmids_icite_ok.fillna(0)).sum()), 0)
    check('rollup_unique_root', 'PASS' if not rp.root_accession.duplicated().any() else 'FAIL',
        int(rp.root_accession.duplicated().sum()), 0)
    report = pd.DataFrame(checks).astype({'check': 'string', 'status': 'string', 'observed': 'string',
            'expected': 'string', 'detail': 'string', 'validated_utc': 'string'})
    report.to_parquet(PATHS['icite_verification'], index=False)
    display(report)
    print(f"Saved verification: {PATHS['icite_verification']}")

run_derived("icite_validation", ["icite_metrics", "pubmed_links", "pubmed_metadata", "icite_study_rollup"],
            ["icite_verification"], build_icite_validation)
display(read_frame("icite_verification"))


## 18. Overall sanity and run report
Read-only table checks plus source-check chronology. Missing optional products are reported; failures are not automatically repaired.


In [ ]:
# @title 18. Overall sanity and run report
def quick_sanity_checks():
    """Print shape/key/value checks. Never write files or modify loaded tables."""
    passed = failed = skipped = 0
    def check(label, condition, detail=""):
        nonlocal passed, failed
        if bool(condition):
            passed += 1
            print(f"PASS  {label}" + (f" - {detail}" if detail else ""))
        else:
            failed += 1
            print(f"FAIL  {label}" + (f" - {detail}" if detail else ""))
    def read(key, required, unique=None):
        nonlocal skipped
        if not os.path.exists(PATHS[key]):
            skipped += 1
            print(f"SKIP  {key}: file absent")
            return None
        frame = pd.read_parquet(PATHS[key])
        missing = sorted(set(required) - set(frame.columns))
        check(f"{key}: required columns", not missing, f"rows={len(frame):,}; missing={missing}")
        if missing:
            return None
        check(f"{key}: distinct column labels", frame.columns.is_unique)
        if unique:
            check(f"{key}: unique {unique}", not frame.duplicated(subset=unique).any())
            check(f"{key}: non-null keys", not frame[unique].isna().any().any())
        return frame

    root = read("root_accessions", ["root_accession", "latest_accession"], ["root_accession"])
    joined = read("joined", ["root_accession"], ["root_accession"])
    if root is not None and joined is not None:
        check("joined: root universe matches", set(root.root_accession) == set(joined.root_accession))
        check("joined: row count matches", len(root) == len(joined))
    links = read("pubmed_links", ["root_accession", "pmid"])
    icite = read("icite_metrics", ["pmid", "rcr", "icite_fetch_note"], ["pmid"])
    if icite is not None:
        scores = pd.to_numeric(icite["rcr"], errors="coerce")
        check("iCite: no negative defined RCR", not scores.lt(0).any())
        print(f"INFO  iCite RCR defined for {scores.notna().sum():,}/{len(scores):,} stored PMIDs; missing retained.")
    rp = read("reporter_pub_links_full", ["pmid", "core_project_num"])
    cf = read("reporter_core_funding", ["core_project_num", "current_period_award", "funding_class", "priceable"],
        ["core_project_num"])
    rf = read("reporter_study_rollup", ["root_accession", "n_linked_cores", "n_priceable_cores_1to1",
            "current_period_priceable_award_1to1", "n_priceable_cores", "current_period_priceable_award",
            "has_unpriced_funding"], ["root_accession"])
    if cf is not None:
        check("RePORTER: expected funding classes", cf.funding_class.isin(["grant", "contract", "intramural",
                    "unknown"]).all())
        check("RePORTER: no negative awards", not pd.to_numeric(cf.current_period_award,
                errors="coerce").lt(0).any())
    if rf is not None:
        check("funding: selected cores <= linked cores", not (rf.n_priceable_cores_1to1 > rf.n_linked_cores).any())
        check("funding: count aliases agree", rf.n_priceable_cores.equals(rf.n_priceable_cores_1to1))
        check("funding: dollar aliases agree",
            rf.current_period_priceable_award.equals(rf.current_period_priceable_award_1to1))
    if links is not None and rp is not None:
        corpus = set(links.pmid.dropna().astype(str))
        matched = set(rp.pmid.dropna().astype(str)) & corpus
        done = set(map(str, load_json(PATHS["reporter_pub_checkpoint"], {"done_pmids": []}).get("done_pmids", [])))
        print(f"INFO  RePORTER: {len(done & corpus):,}/{len(corpus):,} corpus PMIDs checkpointed; {len(matched):,} linked; {len(done - corpus):,} out-of-corpus checkpoint IDs retained.")
        print("INFO  A probed PMID with no returned award is not proof of no NIH support.")
    print(f"\nSanity checks: {passed} passed; {failed} failed; {skipped} missing artifacts skipped.")
    if failed:
        raise AssertionError("Sanity checks failed. Inspect the messages; no data were automatically changed.")

quick_sanity_checks()

# Additional timing and freshness checks; read-only against saved source files.
for key in ["authorized_access_metadata", "timing_first_observed_dar", "ftp_root_results"]:
    frame = read_frame(key)
    if len(frame) and "root_accession" in frame:
        assert frame.root_accession.is_unique, f"Duplicate roots in {key}"
        print(f"PASS  {key}: {len(frame):,} unique root rows")
for block, state in PIPELINE_STATE["sources"].items():
    for item, record in state.get("items", {}).items():
        checked = pd.to_datetime(record.get("last_checked_utc"), utc=True, errors="coerce")
        attempted = pd.to_datetime(record.get("last_attempt_utc"), utc=True, errors="coerce")
        assert pd.isna(checked) or pd.isna(attempted) or checked <= attempted, f"Future successful-check clock: {block}/{item}"
failed_requests = sum(int(r.get("failed",0)) for r in RUN_REPORT)
print(f"INFO  This run: {failed_requests:,} failed source checks. Prior successful rows, where present, were retained.")
print("INFO  Loading a cache and rebuilding a join do not advance successful source-check timestamps.")
run_audit = {"run_id":RUN_ID, "completed_utc":utc_now().isoformat(), "run_mode":RUN_MODE,
             "stale_months":STALE_MONTHS, "blocks":RUN_REPORT, "sources":source_inventory,
             "ftp_date_definition":"earliest preferred retained file in the first numeric version yielding one; not release/minting",
             "maintenance_executed":False}
save_json(os.path.join(DRIVE_ROOT, "notebook_a_last_run_report.json"), run_audit)
print("Run report:", os.path.join(DRIVE_ROOT, "notebook_a_last_run_report.json"))


## 19. Optional maintenance and targeted resets - OFF
No maintenance is required for normal Run all. Reset requests archive registered files and affect the next run. Overlap screening does not constitute human adjudication. No guessed misattribution is corrected.


In [ ]:
# @title 19. Optional maintenance and targeted resets - OFF
def detect_dar_duplicate_accessions(min_shared_wlids=50, min_jaccard=0.98):
    """Flag accession pairs with near-identical requester-WLID sets.

    This is an integrity screen for potential dbGaP alias/redirect behavior.
    It does not establish which accession, if either, is misattributed.
    """
    columns = [
        "accession_a", "accession_b", "n_wlids_a", "n_wlids_b",
        "shared_wlids", "union_wlids", "jaccard_similarity", "set_relation",
    ]
    path = PATHS["dar_records"]
    if not os.path.exists(path):
        print("No DAR record file found. Run Section 2 before this integrity check.")
        return pd.DataFrame(columns=columns)

    dar = pd.read_parquet(path)
    required = {"queried_accession", "wlid"}
    missing = required - set(dar.columns)
    if missing:
        raise ValueError(f"DAR records are missing required columns: {sorted(missing)}")

    clean = dar.loc[:, ["queried_accession", "wlid"]].dropna().copy()
    clean["queried_accession"] = clean["queried_accession"].astype(str).str.strip()
    clean["wlid"] = clean["wlid"].astype(str).str.strip()
    clean = clean[(clean["queried_accession"] != "") & (clean["wlid"] != "")].drop_duplicates()

    wlid_sets = clean.groupby("queried_accession")["wlid"].agg(frozenset).to_dict()
    eligible = {acc: values for acc, values in wlid_sets.items() if len(values) >= min_shared_wlids}
    if len(eligible) < 2:
        print("Fewer than two accessions meet the WLID threshold.")
        return pd.DataFrame(columns=columns)

    inverted = {}
    for accession, wlids in eligible.items():
        for wlid in wlids:
            inverted.setdefault(wlid, []).append(accession)

    shared_counts = {}
    for accessions in inverted.values():
        if len(accessions) > 1:
            for a, b in combinations(sorted(accessions), 2):
                shared_counts[(a, b)] = shared_counts.get((a, b), 0) + 1

    rows = []
    for (a, b), shared in shared_counts.items():
        if shared < min_shared_wlids:
            continue
        set_a, set_b = eligible[a], eligible[b]
        union = len(set_a | set_b)
        jaccard = shared / union if union else 0.0
        if jaccard < min_jaccard:
            continue
        relation = (
            "exact_set_match" if set_a == set_b else
            "a_subset_of_b" if set_a < set_b else
            "b_subset_of_a" if set_b < set_a else
            "near_duplicate_overlap"
        )
        rows.append({
            "accession_a": a,
            "accession_b": b,
            "n_wlids_a": len(set_a),
            "n_wlids_b": len(set_b),
            "shared_wlids": shared,
            "union_wlids": union,
            "jaccard_similarity": round(jaccard, 6),
            "set_relation": relation,
        })

    result = pd.DataFrame(rows, columns=columns)
    if result.empty:
        print(
            "No near-duplicate requester sets found at "
            f"min_shared_wlids={min_shared_wlids}, min_jaccard={min_jaccard}."
        )
        return result

    result = result.sort_values(
        ["jaccard_similarity", "shared_wlids"], ascending=[False, False]
    ).reset_index(drop=True)
    print(f"Found {len(result)} suspicious DAR accession pair(s); manually verify before repair.")
    return result

RUN_DAR_OVERLAP_AUDIT = False
FINALIZE_EXISTING_HUMAN_REVIEW = False
RUN_DETAILED_AUDITS = False
RUN_LIVE_SPOT_CHECKS = False
RERUN_SECTION_5B_CHARACTERIZATION = False
# Reset requests below take effect on the NEXT Run all, not midway through this run.
RESET_BLOCKS = []                  # Example: ["pubmed"]
MARK_STALE_BLOCKS = []             # Example: ["icite"]
ARCHIVE_COMPLETED_WORKSPACES = False
ARCHIVE_WITHHELD_TIMING_FIGURES = False
MAINTENANCE_CONFIRMATION = ""      # Set exactly "APPLY MAINTENANCE" for reset/archive actions.

RESET_SCOPES = {
 "fhir":["studies_raw", "fhir_checkpoint", "root_accessions"],
 "dar":["dar_records", "dar_checkpoint"],
 "counts":["study_counts", "study_counts_checkpoint"],
 "metadata":["study_page_meta", "study_page_checkpoint"],
 "sstr":["sstr_summary", "sstr_checkpoint"],
 "clinical_trials":["clinical_trials_direct", "clinical_trials_direct_checkpoint"],
 "pubmed":["pubmed_metadata", "pubmed_checkpoint"],
 "authorized_access":["authorized_access_metadata", "authorized_access_checkpoint"],
 "icite":["icite_metrics", "icite_checkpoint"],
 "reporter_publications":["reporter_pub_links_full", "reporter_pub_checkpoint"],
 "reporter_projects":["reporter_projects_full", "reporter_proj_checkpoint"],
 "ftp":["ftp_root_results", "ftp_shallow_objects", "ftp_check_state", "ftp_checkpoint_metadata"],
}
if RESET_BLOCKS or MARK_STALE_BLOCKS or ARCHIVE_COMPLETED_WORKSPACES or ARCHIVE_WITHHELD_TIMING_FIGURES:
    if MAINTENANCE_CONFIRMATION != "APPLY MAINTENANCE":
        raise ValueError("Maintenance requested but not confirmed. No files changed.")
    invalid = (set(RESET_BLOCKS) | set(MARK_STALE_BLOCKS)) - set(RESET_SCOPES)
    if invalid:
        raise ValueError(f"Unrecognized maintenance blocks: {sorted(invalid)}; no files changed")
    for block in RESET_BLOCKS:
        archive_registered([PATHS[k] for k in RESET_SCOPES[block]], "reset_"+block)
        PIPELINE_STATE["sources"].pop(block, None)
    for block in MARK_STALE_BLOCKS:
        if block not in RESET_SCOPES:
            raise ValueError(f"Unrecognized stale block: {block}")
        state = PIPELINE_STATE["sources"].setdefault(block, {"items":{}})
        # Explicit maintenance expiration, separately identified; not a fabricated check.
        state["force_check_next_run"] = True
        state["manual_invalidation_utc"] = utc_now().isoformat()
    if ARCHIVE_COMPLETED_WORKSPACES:
        work = Path(DRIVE_ROOT) / "notebook_a_work"
        completed = [p for folder in work.glob("joins_*") if (folder / "_complete.json").is_file()
                     for p in folder.iterdir() if p.is_file()]
        archive_registered(completed, "completed_join_workspaces")
    if ARCHIVE_WITHHELD_TIMING_FIGURES:
        readiness = read_frame("timing_deck_readiness")
        matched = readiness.loc[readiness["output"].str.contains("Figure 30|Figure 31",
                regex=True)] if not readiness.empty else pd.DataFrame()
        if len(matched) and matched["status"].eq("HOLD").all():
            archive_registered([os.path.join(TIMING_FIGURE_DIR,n) for n in TIMING_MATCHED_FIGURES],
                "withheld_timing_figures")
    persist_state()
    print("Maintenance completed. Restart and Run all to apply resets/invalidations to downstream products.")
else:
    print("Maintenance OFF: no reset, cleanup, or cache invalidation performed.")

ALIAS_AUDIT_PATH = os.path.join(DRIVE_ROOT, "dar_exact_requester_set_pairs.parquet")
if RUN_DAR_OVERLAP_AUDIT:
    duplicate_accession_pairs = detect_dar_duplicate_accessions(min_shared_wlids=50, min_jaccard=1.0)
    atomic_table(duplicate_accession_pairs, ALIAS_AUDIT_PATH)
    display(duplicate_accession_pairs.head(10))
else:
    duplicate_accession_pairs = pd.read_parquet(ALIAS_AUDIT_PATH) if os.path.isfile(ALIAS_AUDIT_PATH) else pd.DataFrame()

if FINALIZE_EXISTING_HUMAN_REVIEW:
    REVIEW_CSV = "dar_alias_stage2_review_completed.csv"
    FINAL_ADJUDICATION_PATH = os.path.join(DRIVE_ROOT, "dar_alias_final_adjudications.parquet")
    SENSITIVITY_ACCESSION_PATH = os.path.join(DRIVE_ROOT, "dar_confirmed_misattribution_accessions.parquet")

    review_candidates = [
        REVIEW_CSV,
        os.path.join(DRIVE_ROOT, REVIEW_CSV),
    ]
    review_path = next((path for path in review_candidates if os.path.exists(path)), None)
    if review_path is None:
        print("Completed DAR alias review CSV not found; finalization skipped.")
    else:
        reviewed = pd.read_csv(review_path).copy()
        allowed = {
            "valid_related_studies",
            "valid_umbrella_substudy_relationship",
            "valid_legacy_current_accession_relationship",
            "valid_shared_consent_or_access_structure",
            "confirmed_endpoint_misattribution",
            "insufficient_evidence",
        }
        for column in ["review_status", "final_disposition", "include_primary_analysis",
            "include_sensitivity_analysis"]:
            reviewed[column] = reviewed[column].astype("string").str.strip()
        invalid = reviewed.loc[reviewed["final_disposition"].notna() & ~reviewed["final_disposition"].isin(allowed)]
        incomplete = reviewed.loc[~reviewed["review_status"].eq("complete")]
        if len(invalid):
            raise ValueError(f"Invalid final-disposition values: {invalid['final_disposition'].dropna().unique().tolist()}")
        if len(incomplete):
            raise ValueError(f"Incomplete alias reviews: {len(incomplete)}")
        confirmed = reviewed.loc[reviewed["final_disposition"].eq("confirmed_endpoint_misattribution")].copy()
        sensitivity_accessions = pd.DataFrame({
            "accession": pd.concat([confirmed["accession_a"], confirmed["accession_b"]],
                ignore_index=True).dropna().drop_duplicates()
        })
        reviewed.to_parquet(FINAL_ADJUDICATION_PATH, index=False)
        sensitivity_accessions.to_parquet(SENSITIVITY_ACCESSION_PATH, index=False)
        print("DAR alias adjudication finalized.")
        print(reviewed["final_disposition"].value_counts().to_string())
        print(f"Confirmed endpoint-misattribution pairs: {len(confirmed):,}")
        print(f"Sensitivity-exclusion accessions: {len(sensitivity_accessions):,}")
        print("Primary accession-level analysis retains all DAR records.")


## 20. Optional detailed saved-data audits - OFF
Retained audit/report schemas. Enable RUN_DETAILED_AUDITS only in the preceding maintenance cell; live checks remain separately disabled.


In [ ]:
# @title 20. Optional detailed saved-data audits - OFF
if RUN_DETAILED_AUDITS:

    # Read-only current-state validation and deterministic spot-check selection.
    VALIDATION_RANDOM_SEED = 20260930
    N_RANDOM_SPOT_CHECKS = 12

    rng = np.random.default_rng(VALIDATION_RANDOM_SEED)
    report_rows = []


    def validation_row(domain, check, status, observed, expected=None, detail=None):
        processed_detail = None
        if detail is not None:
            if isinstance(detail, (list, dict)):
                processed_detail = json.dumps(detail)
            else:
                processed_detail = str(detail)

        report_rows.append({
            "domain": domain,
            "check": check,
            "status": status,
            "observed": observed,
            "expected": expected,
            "detail": processed_detail,
            "validated_utc": datetime.now(timezone.utc).isoformat(),
        })


    def stage_done_set(checkpoint_key, default):
        state = load_json(PATHS[checkpoint_key], default)
        done = state.get("done_accessions", {})
        return set(map(str, done.keys())) if isinstance(done, dict) else set(map(str, done))


    root = pd.read_parquet(PATHS["root_accessions"])
    joined = pd.read_parquet(PATHS["joined"])
    latest = set(root["latest_accession"].dropna().astype(str))

    print("=" * 84)
    print("CURRENT-STATE VALIDATION")
    print("=" * 84)
    print(f"Current latest-accession universe: {len(latest):,}")
    print(f"Analysis-ready joined rows: {len(joined):,}")

    # 1. Canonical join checks.
    join_unique = not joined["root_accession"].duplicated().any()
    join_match = len(joined) == len(root)
    validation_row("joined", "one_row_per_root", "PASS" if join_unique else "FAIL",
        int(joined["root_accession"].duplicated().sum()), 0)
    validation_row("joined", "cardinality_matches_root_table", "PASS" if join_match else "FAIL", len(joined),
        len(root))

    # 2. Current coverage from checkpoints.
    stages = [
        ("DAR", "dar_checkpoint", {"done_accessions": {}}),
        ("study_counts", "study_counts_checkpoint", {"done_accessions": []}),
        ("study_page_metadata", "study_page_checkpoint", {"done_accessions": []}),
        ("SSTR", "sstr_checkpoint", {"done_accessions": {}}),
    ]
    for label, key, default in stages:
        done = stage_done_set(key, default)
        covered = len(done & latest)
        stale = len(done - latest)
        validation_row(label, "current_universe_checkpoint_coverage", "PASS" if covered == len(latest) else "FAIL",
            covered, len(latest), f"stale_checkpoint_entries={stale}")

    # 3. Raw and analysis-ready source checks.
    source_specs = [
        ("study_counts", PATHS["study_counts"], "accession"),
        ("study_page_metadata", PATHS["study_page_meta"], "accession"),
        ("SSTR", PATHS["sstr_summary"], "accession"),
    ]
    for label, path, col in source_specs:
        df = pd.read_parquet(path)
        observed = set(df[col].dropna().astype(str)) if col in df else set()
        duplicates = int(df[col].duplicated().sum()) if col in df else len(df)
        validation_row(label, "raw_rows", "INFO", len(df))
        validation_row(label, "raw_accessions_linked_to_current",
            "PASS" if len(observed & latest) == len(latest) else "WARN", len(observed & latest), len(latest),
            f"duplicate_accession_rows={duplicates}; stale_or_foreign={len(observed-latest)}")

    # 4. DAR persistence and duplicate checks.
    dar = pd.read_parquet(PATHS["dar_records"])
    dar_accessions = set(dar["queried_accession"].dropna().astype(str))
    dar_checkpoint_state = load_json(PATHS["dar_checkpoint"], {"done_accessions": {}}).get("done_accessions", {})
    dar_ok = {str(a) for a, note in dar_checkpoint_state.items() if note == "ok"}
    missing_ok_rows = dar_ok - dar_accessions
    identity_cols = [c for c in ["queried_accession", "wlid", "status",
            "initial_approval_date"] if c in dar.columns]
    validation_row("DAR", "checkpoint_ok_without_persisted_rows", "PASS" if not missing_ok_rows else "FAIL",
        len(missing_ok_rows), 0, sorted(missing_ok_rows)[:10])
    validation_row("DAR", "literal_duplicate_rows",
        "PASS" if not dar.duplicated(subset=identity_cols).any() else "WARN",
        int(dar.duplicated(subset=identity_cols).sum()), 0)

    # 5. SSTR outcome coverage.
    sstr = pd.read_parquet(PATHS["sstr_summary"])
    if "sstr_fetch_note" in sstr.columns:
        status_counts = sstr["sstr_fetch_note"].fillna("<missing>").value_counts().to_dict()
        for note, count in status_counts.items():
            validation_row("SSTR", f"fetch_note:{note}", "INFO", int(count))

    # 6. PubMed full enrichment completeness and linkage checks.
    pubmed = pd.read_parquet(PATHS["pubmed_metadata"])
    pubmed_links = pd.read_parquet(PATHS["pubmed_links"])
    pubmed["pmid"] = pubmed["pmid"].astype(str)
    pubmed_links["pmid"] = pubmed_links["pmid"].astype(str)
    unique_pmids = pubmed_links["pmid"].nunique()
    ok_pmids = pubmed.loc[pubmed["pubmed_fetch_note"].eq("ok"), "pmid"].nunique()
    not_returned = pubmed.loc[pubmed["pubmed_fetch_note"].eq("not_returned"), "pmid"].nunique()
    missing_link_metadata = int(pubmed_links["pubmed_fetch_note"].isna().sum())
    validation_row("PubMed", "unique_pmids_in_linkage", "INFO", unique_pmids)
    validation_row("PubMed", "successful_metadata_pmids",
        "PASS" if ok_pmids + not_returned == unique_pmids else "WARN", ok_pmids, unique_pmids,
        f"not_returned={not_returned}")
    validation_row("PubMed", "link_rows_missing_metadata", "PASS" if missing_link_metadata == 0 else "WARN",
        missing_link_metadata, 0)

    # 7. Direct ClinicalTrials.gov linkage completeness.
    ct = pd.read_parquet(PATHS["clinical_trials_direct"])
    ct_ok = int(ct["ctgov_fetch_note"].eq("ok").sum()) if "ctgov_fetch_note" in ct else 0
    validation_row("ClinicalTrials.gov", "direct_accession_nct_pairs", "INFO", len(ct))
    validation_row("ClinicalTrials.gov", "successful_direct_pair_resolution",
        "PASS" if ct_ok == len(ct) else "WARN", ct_ok, len(ct))

    # Spot-check queue: selected deterministically from high-value strata.
    spot_rows = []
    # High DAR counts.
    if "n_dar_total" in joined.columns:
        high_dar = joined.nlargest(4, "n_dar_total")[["root_accession", "latest_accession", "n_dar_total"]].copy()
        high_dar["reason"] = "highest_DAR_counts"
        spot_rows.append(high_dar)
    # Direct clinical links.
    if "direct_ctgov_link_present" in joined.columns:
        trial = joined.loc[joined["direct_ctgov_link_present"], ["root_accession",
                "latest_accession"]].head(4).copy()
        trial["reason"] = "direct_ClinicalTrials_link"
        spot_rows.append(trial)
    # Non-ok SSTR examples.
    non_ok_sstr = sstr.loc[sstr.get("sstr_fetch_note", pd.Series(index=sstr.index, dtype="string")).ne("ok"),
        ["accession", "sstr_fetch_note"]].drop_duplicates().head(4).copy()
    if not non_ok_sstr.empty:
        non_ok_sstr = non_ok_sstr.rename(columns={"accession": "latest_accession"})
        non_ok_sstr["root_accession"] = non_ok_sstr["latest_accession"].str.split(".").str[0]
        non_ok_sstr["reason"] = "non_ok_SSTR_outcome"
        spot_rows.append(non_ok_sstr)
    # Deterministic random roots.
    random_roots = root.sample(min(N_RANDOM_SPOT_CHECKS, len(root)),
        random_state=VALIDATION_RANDOM_SEED)[["root_accession", "latest_accession"]].copy()
    random_roots["reason"] = "deterministic_random_sample"
    spot_rows.append(random_roots)
    spot_checks = pd.concat(spot_rows, ignore_index=True, sort=False).drop_duplicates(subset=["latest_accession",
            "reason"])
    SPOT_PATH = os.path.join(DRIVE_ROOT, "pipeline_spot_check_queue.parquet")
    spot_checks.to_parquet(SPOT_PATH, index=False)

    validation = pd.DataFrame(report_rows)
    VALIDATION_PATH = os.path.join(DRIVE_ROOT, "pipeline_current_state_validation.parquet")
    validation.to_parquet(VALIDATION_PATH, index=False)
    print()
    print("VALIDATION RESULTS")
    print(validation.to_string(index=False))
    print()
    print(f"Saved validation report: {VALIDATION_PATH}")
    print(f"Saved spot-check queue: {SPOT_PATH}")

    # Optional, small live API verification. No writes or checkpoint changes.
    if RUN_LIVE_SPOT_CHECKS:
        print("\nLIVE SPOT CHECKS")
        for accession in spot_checks["latest_accession"].dropna().astype(str).head(5):
            dar_df, dar_note = fetch_dar_tsv(accession)
            print(f"{accession}: DAR live={dar_note}; rows={None if dar_df is None else len(dar_df)}")
            time.sleep(0.5)
    else:
        print("Live spot checks skipped. Set RUN_LIVE_SPOT_CHECKS = True to make up to five network requests.")

else:
    print("Detailed validation skipped; existing reports retained.")


if RUN_DETAILED_AUDITS:
    # Read-only integrity audit and source-linkage summary.
    # Does not modify raw data, checkpoints, or joins.


    AUDIT_PATH = os.path.join(DRIVE_ROOT, "pipeline_integrity_findings.parquet")
    SUMMARY_PATH = os.path.join(DRIVE_ROOT, "pipeline_source_linkage_summary.parquet")

    findings = []
    summary_rows = []


    def add_finding(severity, domain, issue, affected_n=None, sample=None, action=None):
        # Ensure 'sample' is always stored as a string for consistent Parquet conversion
        processed_sample = None
        if sample is not None:
            if isinstance(sample, (list, dict)):
                processed_sample = json.dumps(sample)
            else:
                processed_sample = str(sample)

        findings.append({
            "severity": severity,
            "domain": domain,
            "issue": issue,
            "affected_n": affected_n,
            "sample": processed_sample,
            "recommended_action": action,
            "audited_utc": datetime.now(timezone.utc).isoformat(),
        })


    def add_summary(source, metric, value, denominator=None, notes=None):
        summary_rows.append({
            "source": source,
            "metric": metric,
            "value": value,
            "denominator": denominator,
            "notes": notes,
        })


    def read_parquet_if_present(path, label):
        if not os.path.exists(path):
            add_finding("ERROR", label, "Expected parquet artifact is missing.",
                action="Run the corresponding acquisition section.")
            return pd.DataFrame()
        try:
            return pd.read_parquet(path)
        except Exception as exc:
            add_finding("ERROR", label, f"Parquet cannot be read: {type(exc).__name__}: {exc}",
                action="Restore or rebuild this artifact.")
            return pd.DataFrame()


    def checkpoint_accessions(path_key, default):
        state = load_json(PATHS[path_key], default)
        done = state.get("done_accessions", {})
        if isinstance(done, dict):
            return set(map(str, done.keys())), done
        if isinstance(done, list):
            return set(map(str, done)), {str(x): "done" for x in done}
        add_finding("WARNING", path_key, "Checkpoint has an unexpected done_accessions structure.",
            action="Inspect checkpoint JSON before relying on resumability.")
        return set(), {}


    print("=" * 88)
    print("PIPELINE INTEGRITY AUDIT AND SOURCE-LINKAGE SUMMARY")
    print("=" * 88)

    root = read_parquet_if_present(PATHS["root_accessions"], "root_accessions")
    dar = read_parquet_if_present(PATHS["dar_records"], "DAR")
    counts = read_parquet_if_present(PATHS["study_counts"], "study_counts")
    page = read_parquet_if_present(PATHS["study_page_meta"], "study_page_metadata")
    sstr = read_parquet_if_present(PATHS["sstr_summary"], "SSTR")
    joined = read_parquet_if_present(PATHS["joined"], "joined")

    root_set = set(root.get("root_accession", pd.Series(dtype="string")).dropna().astype(str))
    latest_set = set(root.get("latest_accession", pd.Series(dtype="string")).dropna().astype(str))
    add_summary("FHIR", "root_accessions", len(root_set), notes="Unique root dbGaP accessions")
    add_summary("FHIR", "latest_accessions", len(latest_set), notes="Latest version/participant-set accessions")

    if root.empty:
        add_finding("ERROR", "FHIR", "Root accession table is empty.", action="Rebuild Section 1.")
    else:
        n_duplicate_roots = int(root["root_accession"].duplicated().sum())
        if n_duplicate_roots:
            add_finding("ERROR", "FHIR", "Duplicate root accession rows.", n_duplicate_roots,
                action="Rebuild Section 1 root table.")
        n_missing_latest = int(root["latest_accession"].isna().sum())
        if n_missing_latest:
            add_finding("WARNING", "FHIR", "Root rows lacking latest accession.", n_missing_latest,
                action="Inspect FHIR parsing for malformed identifiers.")

    # Checkpoint completeness and agreement with root universe.
    stage_specs = [
        ("dar_checkpoint", {"done_accessions": {}}, "DAR"),
        ("study_counts_checkpoint", {"done_accessions": []}, "study_counts"),
        ("study_page_checkpoint", {"done_accessions": []}, "study_page_metadata"),
        ("sstr_checkpoint", {"done_accessions": {}}, "SSTR"),
    ]
    checkpoint_sets = {}
    for key, default, label in stage_specs:
        done_set, done_map = checkpoint_accessions(key, default)
        checkpoint_sets[label] = done_set
        covered = len(latest_set & done_set)
        foreign = sorted(done_set - latest_set)
        missing = sorted(latest_set - done_set)
        add_summary(label, "checkpoint_covered_latest_accessions", covered, len(latest_set))
        add_summary(label, "checkpoint_missing_latest_accessions", len(missing), len(latest_set))
        if foreign:
            add_finding("WARNING", label,
                "Checkpoint contains accessions outside current latest-accession universe.", len(foreign),
                foreign[:10], "These may be stale-version entries; inspect before deleting.")
        if missing:
            add_finding("WARNING", label, "Current latest accessions absent from checkpoint.", len(missing),
                missing[:10], "Re-run the relevant resumable crawler.")

    # Generic accession-output coverage and duplicate checks.
    def check_accession_table(label, frame, accession_col, expected_set):
        if frame.empty:
            return set()
        if accession_col not in frame.columns:
            add_finding("ERROR", label, f"Missing required accession column: {accession_col}.",
                action="Rebuild the artifact.")
            return set()
        observed = set(frame[accession_col].dropna().astype(str))
        duplicates = int(frame[accession_col].duplicated().sum())
        coverage = len(observed & expected_set)
        add_summary(label, "rows", len(frame))
        add_summary(label, "unique_accessions", len(observed))
        add_summary(label, "linked_to_latest_accessions", coverage, len(expected_set))
        add_summary(label, "unlinked_accessions", len(observed - expected_set))
        if duplicates:
            add_finding("WARNING", label, "Duplicate accession rows in a nominally study-level table.", duplicates,
                action="Compact to one latest record per accession before joining.")
        unexpected = sorted(observed - expected_set)
        if unexpected:
            add_finding("WARNING", label,
                "Output contains accessions not present in current latest-accession universe.", len(unexpected),
                unexpected[:10], "Inspect historical versions or stale output rows.")
        return observed

    count_set = check_accession_table("study_counts", counts, "accession", latest_set)
    page_set = check_accession_table("study_page_metadata", page, "accession", latest_set)
    sstr_set = check_accession_table("SSTR", sstr, "accession", latest_set)

    # DAR integrity and linkage.
    if not dar.empty:
        required_dar = {"queried_accession", "wlid"}
        missing_columns = required_dar - set(dar.columns)
        if missing_columns:
            add_finding("ERROR", "DAR", f"Missing columns: {sorted(missing_columns)}.",
                action="Restore/rebuild DAR records.")
        else:
            dar_accessions = set(dar["queried_accession"].dropna().astype(str))
            add_summary("DAR", "rows", len(dar))
            add_summary("DAR", "studies_with_requester_rows", len(dar_accessions), len(latest_set))
            add_summary("DAR", "requester_wlids", int(dar["wlid"].nunique()))
            identity_columns = [c for c in ["queried_accession", "wlid", "status",
                    "initial_approval_date"] if c in dar.columns]
            literal_duplicates = int(dar.duplicated(subset=identity_columns).sum())
            add_summary("DAR", "literal_duplicate_rows", literal_duplicates)
            if literal_duplicates:
                add_finding("WARNING", "DAR", "Literal duplicate requester rows.", literal_duplicates,
                    action="Deduplicate only after preserving provenance and reconciling checkpoints.")
            if "source" in dar.columns:
                mixed = dar.groupby("queried_accession")["source"].nunique()
                mixed = mixed[mixed > 1]
                add_summary("DAR", "accessions_with_multiple_sources", len(mixed))
                if len(mixed):
                    add_finding("WARNING", "DAR", "Accessions contain rows from multiple endpoint sources.",
                        len(mixed), mixed.index.tolist()[:10],
                        "Review for partial recrawls or duplicate provenance.")
            dar_done, dar_notes = checkpoint_accessions("dar_checkpoint", {"done_accessions": {}})
            ok_set = {a for a, note in dar_notes.items() if note == "ok"}
            ok_missing_rows = sorted(ok_set - dar_accessions)
            add_summary("DAR", "checkpoint_ok_without_rows", len(ok_missing_rows))
            if ok_missing_rows:
                add_finding("WARNING", "DAR", "Checkpoint ok records lack persisted DAR rows.",
                    len(ok_missing_rows), ok_missing_rows[:10],
                    "Live-fetch and restore; distinguish actual empty responses from persistence loss.")
            no_requester_set = {a for a, note in dar_notes.items() if note == "no_requesters"}
            overlap = no_requester_set & dar_accessions
            if overlap:
                add_finding("WARNING", "DAR",
                    "no_requesters checkpoint entries nevertheless have persisted requester rows.", len(overlap),
                    sorted(overlap)[:10], "Inspect historical recrawls and endpoint changes.")

            # Exact requester-set duplicates are a review queue, not automatic errors.
            try:
                alias_pairs = pd.read_parquet(ALIAS_AUDIT_PATH) if os.path.exists(ALIAS_AUDIT_PATH) else pd.DataFrame()
                if alias_pairs.empty:
                    add_finding("INFO", "DAR alias screening",
                        "No saved nonempty alias screen available; not recomputed by this audit.",
                        action="Enable RUN_SECTION_2B_ALIAS_SCREEN to refresh the saved screen.")
                if not alias_pairs.empty:
                    add_summary("DAR", "exact_requester_set_duplicate_pairs", len(alias_pairs))
                    add_finding("INFO", "DAR alias screening",
                        "Saved exact requester-WLID set duplicate pairs; screen not recomputed in this audit.",
                        len(alias_pairs), alias_pairs[["accession_a", "accession_b"]].head(10).to_dict("records"),
                        "Treat as an adjudication queue; do not auto-delete related-study or alias records.")
            except Exception as exc:
                add_finding("WARNING", "DAR alias screening",
                    f"Could not run requester-set duplicate screen: {type(exc).__name__}: {exc}",
                    action="Check DAR schema.")

    # SSTR status and discontinuity checks.
    if not sstr.empty:
        if "sstr_fetch_note" in sstr.columns:
            notes = sstr["sstr_fetch_note"].fillna("<missing>").value_counts().to_dict()
            for note, n in notes.items():
                add_summary("SSTR", f"fetch_note:{note}", int(n))
        if "accession" in sstr.columns:
            non_ok_rows = sstr.loc[sstr.get("sstr_fetch_note", pd.Series(index=sstr.index,
                        dtype="string")).ne("ok")]
            if len(non_ok_rows):
                add_finding("INFO", "SSTR", "Persisted SSTR rows with non-ok fetch notes.", len(non_ok_rows),
                    non_ok_rows["accession"].head(10).tolist(),
                    "Interpret as source-coverage limitations unless live rechecks now succeed.")

    # Study-level publication metadata and joined table checks.
    if not page.empty and "n_publications" in page.columns:
        pubs = pd.to_numeric(page["n_publications"], errors="coerce")
        add_summary("dbGaP study-page", "studies_with_publications", int((pubs.fillna(0) > 0).sum()), len(page))
        add_summary("dbGaP study-page", "publications_total", int(pubs.fillna(0).sum()))

    if not joined.empty:
        add_summary("joined", "rows", len(joined))
        if "root_accession" in joined.columns:
            joined_roots = set(joined["root_accession"].dropna().astype(str))
            missing_joined = sorted(root_set - joined_roots)
            extra_joined = sorted(joined_roots - root_set)
            add_summary("joined", "linked_root_accessions", len(joined_roots & root_set), len(root_set))
            if missing_joined or extra_joined:
                add_finding("ERROR", "joined", "Joined root universe differs from root table.",
                    len(missing_joined) + len(extra_joined), {"missing": missing_joined[:5],
                        "extra": extra_joined[:5]}, "Rebuild Section 7 and inspect merge keys.")
            duplicate_joined = int(joined["root_accession"].duplicated().sum())
            if duplicate_joined:
                add_finding("ERROR", "joined", "Duplicate root rows in joined table.", duplicate_joined,
                    action="Rebuild Section 7 after resolving duplicate study-level source tables.")
        for col in ["n_dar_total", "n_publications", "n_publications_corrected", "n_subjects_total",
            "n_samples_total"]:
            if col in joined.columns:
                vals = pd.to_numeric(joined[col], errors="coerce")
                negative = int((vals < 0).sum())
                if negative:
                    add_finding("ERROR", "joined", f"Negative values detected in {col}.", negative,
                        action="Inspect source parser and join logic.")
                add_summary("joined", f"nonmissing:{col}", int(vals.notna().sum()), len(joined))
        if {"n_publications", "n_publications_corrected"}.issubset(joined.columns):
            bad_rollup = int((joined["n_publications_corrected"].fillna(0) < joined["n_publications"].fillna(0)).sum())
            if bad_rollup:
                add_finding("ERROR", "joined", "Corrected publication count below raw count.", bad_rollup,
                    action="Rebuild publication-rollup calculation.")

    # Optional PubMed enrichment summary.
    # PubMed metadata is PMID-anchored to dbGaP study-page output and is the
    # only external publication source retained in the production workflow.

    pubmed_path = PATHS.get(
        "pubmed_metadata"
    )

    if pubmed_path and os.path.exists(pubmed_path):
        pubmed = read_parquet_if_present(
            pubmed_path,
            "PubMed enrichment",
        )

        if not pubmed.empty:
            if "pmid" in pubmed.columns:
                n_pubmed_pmids = (
                    pubmed["pmid"]
                    .dropna()
                    .astype(str)
                    .nunique()
                )

                add_summary(
                    "PubMed enrichment",
                    "unique_pmids_with_metadata",
                    n_pubmed_pmids,
                    notes=(
                        "PMID metadata retrieved from "
                        "dbGaP-provided publication links"
                    ),
                )

            if "pubmed_fetch_note" in pubmed.columns:
                pubmed_status = (
                    pubmed["pubmed_fetch_note"]
                    .fillna("<missing>")
                    .value_counts()
                )

                for note, count in pubmed_status.items():
                    add_summary(
                        "PubMed enrichment",
                        f"fetch_note:{note}",
                        int(count),
                        len(pubmed),
                        notes=(
                            "Optional PMID metadata "
                            "enrichment"
                        ),
                    )

            if "pubmed_fetch_note" in pubmed.columns:
                non_ok_pubmed = int(
                    pubmed["pubmed_fetch_note"]
                    .fillna("<missing>")
                    .ne("ok")
                    .sum()
                )

                if non_ok_pubmed:
                    add_finding(
                        "WARNING",
                        "PubMed enrichment",
                        (
                            "Persisted PubMed metadata rows "
                            "with non-ok fetch status."
                        ),
                        non_ok_pubmed,
                        action=(
                            "Re-run only missing or failed PMIDs "
                            "before expanding the enrichment corpus."
                        ),
                    )

    else:
        add_finding(
            "INFO",
            "PubMed enrichment",
            (
                "No PubMed metadata output is present. "
                "Core dbGaP data remain complete."
            ),
            action=(
                "Run the full-corpus PubMed enrichment stage."
            ),
        )

    findings_df = pd.DataFrame(findings)
    if findings_df.empty:
        findings_df = pd.DataFrame(columns=["severity", "domain", "issue", "affected_n", "sample",
                "recommended_action", "audited_utc"])
    else:
        order = {"ERROR": 0, "WARNING": 1, "INFO": 2}
        findings_df["_order"] = findings_df["severity"].map(order)
        findings_df = findings_df.sort_values(["_order", "domain",
                "issue"]).drop(columns="_order").reset_index(drop=True)

    summary_df = pd.DataFrame(summary_rows)
    findings_df.to_parquet(AUDIT_PATH, index=False)
    summary_df.to_parquet(SUMMARY_PATH, index=False)

    print("\nSOURCE/LINKAGE SUMMARY")
    print(summary_df.to_string(index=False))
    print("\nAUDIT FINDINGS")
    if findings_df.empty:
        print("No automated integrity findings.")
    else:
        print(findings_df.to_string(index=False))
        print("\nFinding counts:", findings_df["severity"].value_counts().to_dict())
    print(f"\nSaved audit findings: {AUDIT_PATH}")
    print(f"\nSaved source/linkage summary: {SUMMARY_PATH}")

else:
    print("Detailed integrity audit skipped; existing reports retained.")


## 21. Optional SSTR diagnostics - OFF
Historical source diagnostics are kept here rather than mixed into collection. The obsolete PubMed pilot repair/re-query path is removed; current PubMed work goes through one collector.


In [ ]:
# @title 21. Optional SSTR diagnostics - OFF
root_table = pd.read_parquet(PATHS["root_accessions"])

if not RERUN_SECTION_5B_CHARACTERIZATION:
    print(
        "Section 5b characterization skipped. "
        "Set RERUN_SECTION_5B_CHARACTERIZATION = True "
        "to summarize persisted non-ok SSTR accessions."
    )

else:
    sstr_state = load_json(
        PATHS["sstr_checkpoint"],
        {"done_accessions": {}},
    )

    sstr_done = sstr_state.get(
        "done_accessions",
        {},
    )

    non_ok = {
        accession: note
        for accession, note in sstr_done.items()
        if note != "ok"
    }

    failed_root_accessions = {
        str(accession).split(".")[0]
        for accession in non_ok
    }

    failed_in_root_table = root_table.loc[
        root_table["root_accession"].isin(
            failed_root_accessions
        )
    ].copy()

    print(
        f"Non-ok SSTR accessions: {len(non_ok):,}"
    )

    print()

    print(
        "Persisted SSTR status distribution:"
    )

    print(
        pd.Series(non_ok)
        .value_counts()
        .to_string()
    )

    print()

    print(
        f"Failed latest accessions represented "
        f"in current root table: "
        f"{len(failed_in_root_table):,}"
    )

    if "latest_v" in failed_in_root_table.columns:
        print()

        print(
            "Version distribution among failed "
            "accessions:"
        )

        print(
            failed_in_root_table[
                "latest_v"
            ]
            .value_counts()
            .sort_index()
            .to_string()
        )

    display(
        failed_in_root_table[
            [
                column
                for column in [
                    "root_accession",
                    "latest_accession",
                    "latest_v",
                    "titles_seen",
                    "statuses_seen",
                ]
                if column in failed_in_root_table.columns
            ]
        ].head(50)
    )


## Notes for interpreting the cache and timing outputs

- `notebook_a_refresh_state.json` records successful checks, attempts, versions and local input signatures; `notebook_a_last_run_report.json` records this run. Do not treat its completion time as every source's acquisition time.
- A transient failed refresh keeps an earlier successful row active and reports the failed attempt. Inspect the run report before describing the whole corpus as freshly checked.
- An interrupted FHIR snapshot is in `notebook_a_work/fhir`; a clean build archives previous candidate files as well. Other source collectors checkpoint successful batches and resume under `reuse`.
- `notebook_a_backups` is never automatically purged. Storage management here refers only to notebook artifacts, not dbGaP/SRA scientific data.
- Local join products use temporary files and final atomic per-file replacement. This is not a filesystem-wide atomic transaction; an interruption during multi-file publication is detected through signatures and missing/output-hash checks on the next run.
- Neither this refactor nor a passing sanity check validates publication provenance, the meaning of a requester identity, award-cost attribution, or a causal effect of data sharing.
